In [ ]:
# ============================================================
# CELL 1 — Setup and verified download
# ============================================================
import os, io, json, zipfile, gc, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import precision_score

warnings.filterwarnings('ignore')

SEED     = 42
DT       = 1.0      # nominal message interval, seconds
GAP_MAX  = 2.0      # residuals undefined beyond this interval
SCENARIO = "InTAS_highway_2"
BASE     = "https://zenodo.org/records/19665762/files"

ATTACKS = '''accelerationMultiplication constantPositionOffset constantSpeedOffset
dataReplay dosAttack feignedBraking positionMirroring randomPositionOffset
randomSpeedOffset reversedHeading suddenConstantSpeed suddenStop
timeDelayAttack trafficCongestionSybil zeroSpeedReport'''.split()

FEATS = ["s_spd","s_acl","s_hed","disp","dt","implied_spd",
         "spd_resid","hed_resid","acl_resid","latency"]
RESID = ["spd_resid","hed_resid","acl_resid"]

os.makedirs("data", exist_ok=True)
os.makedirs("out",  exist_ok=True)

# BUGFIX: the old loop wrote HTML error pages into the .zip, and
# `if not os.path.exists(p)` then guaranteed it was never retried.
for a in ATTACKS:
    p = f"data/{SCENARIO}_{a}.zip"
    if not (os.path.exists(p) and zipfile.is_zipfile(p)):
        if os.path.exists(p):
            os.remove(p)
        !wget -q --tries=3 "{BASE}/{SCENARIO}_{a}.zip?download=1" -O "{p}"
    assert zipfile.is_zipfile(p), f"DOWNLOAD FAILED: {a} — rerun this cell"

print(f"{len(ATTACKS)} attack subsets verified")

In [ ]:
# ============================================================
# CELL 2 — Loading and the ONE feature function
#   NaN residuals are PRESERVED. Nothing is filled here.
# ============================================================

def num(v, idx=None):
    if v is None: return None
    if idx is not None:
        try: return float(str(v).split(",")[idx])
        except (ValueError, IndexError): return None
    try: return float(v)
    except (ValueError, TypeError): return None

def load(zip_path, split, label):
    outer = zipfile.ZipFile(zip_path)
    inner_name = [n for n in outer.namelist()
                  if f"/{split}/" in n and n.endswith(".zip")][0]
    inner = zipfile.ZipFile(io.BytesIO(outer.read(inner_name)))
    rows = []
    for name in inner.namelist():
        if not name.endswith(".json"): continue
        for m in json.loads(inner.read(name)):
            s, r = m.get("sender", {}), m.get("receiver", {})
            rows.append({
                "rcvTime_s":   (num(m.get("rcvTime"))  or 0) / 1e9,
                "sendTime_s":  (num(m.get("sendTime")) or 0) / 1e9,
                "sender_id":   m.get("sender_id"),
                "attacker":    int(m.get("attacker", 0)),
                "attack_type": label,
                "s_x": num(s.get("pos"), 0), "s_y": num(s.get("pos"), 1),
                "s_spd": num(s.get("spd")),  "s_acl": num(s.get("acl")),
                "s_hed": num(s.get("hed")),
                "r_x": num(r.get("pos"), 0), "r_y": num(r.get("pos"), 1),
            })
    return pd.DataFrame(rows)

def load_split(split):
    return pd.concat(
        [load(f"data/{SCENARIO}_{a}.zip", split, a) for a in ATTACKS],
        ignore_index=True)


def build_features(df, group_cols=("attack_type", "sender_id")):
    '''The single feature path. Benign, attack, rotated and generated
    messages ALL go through this. Residuals stay NaN where undefined.'''
    g_cols = list(group_cols)

    # VeReMi logs are per-receiver: one transmission appears once per
    # receiving vehicle. Deduplicate before any inter-message difference.
    d = (df.sort_values(g_cols + ["sendTime_s"])
           .drop_duplicates(g_cols + ["sendTime_s"])
           .copy())

    g  = d.groupby(g_cols)
    dt = g.sendTime_s.diff()
    dx, dy = g.s_x.diff(), g.s_y.diff()

    d["dt"]          = dt
    d["disp"]        = np.hypot(dx, dy)
    d["implied_spd"] = d.disp / dt.replace(0, np.nan)
    d["spd_resid"]   = d.implied_spd - d.s_spd

    # CAM heading: clockwise from north
    move_hed = (90 - np.degrees(np.arctan2(dy, dx))) % 360
    hd = (move_hed - d.s_hed).abs() % 360
    d["hed_resid"] = np.minimum(hd, 360 - hd)

    d["acl_resid"] = g.s_spd.diff() / dt.replace(0, np.nan) - d.s_acl
    d["latency"]   = d.rcvTime_s - d.sendTime_s

    # Undefined where the gap is too large for straight-line displacement.
    # KEEP AS NaN — this is the bug that produced a partial label before.
    d.loc[d.dt > GAP_MAX, RESID] = np.nan
    return d


def X(d, fill=0.0):
    '''Model input. Filling happens HERE and nowhere else.'''
    return d[FEATS].replace([np.inf, -np.inf], np.nan).fillna(fill)


tr = build_features(load_split("Train"))
te = build_features(load_split("Test"))
gc.collect()

tr.to_parquet("out/trF.parquet")   # NaNs preserved
te.to_parquet("out/teF.parquet")
print("transmissions:", tr.shape[0], te.shape[0])

In [ ]:
# ============================================================
# CELL 3 — SANITY GATE.  If any assert fails, STOP and fix.
# ============================================================
b = tr[tr.attacker == 0]      # benign TRAIN  (bounds source from here on)
bt = te[te.attacker == 0]     # benign TEST   (reporting only)

print(f"benign train  dt {b.dt.median():.3f}s   "
      f"spd_resid {b.spd_resid.abs().median():.4f} m/s   "
      f"hed_resid {b.hed_resid.abs().median():.3f} deg")

assert abs(b.dt.median() - 1.0) < 0.02, \
    "dt is not ~1.0s -> deduplication did not happen"
assert abs(b.spd_resid.abs().median() - 0.098) < 0.02, \
    "speed residual is not ~0.098 -> residuals computed over log rows"
assert abs(b.hed_resid.abs().median() - 2.9) < 0.4, \
    "heading residual is not ~2.9deg -> wrong angle convention"

# ---- C1 DIAGNOSTIC: is the old fillna(0) a label? ----
undef = tr.spd_resid.isna()
print(f"\nrows with undefined residuals : {undef.sum():,} ({undef.mean():.2%})")
assert undef.sum() > 0, "NaNs were lost -> residuals were filled somewhere upstream"
print(f"attacker rate on those rows   : {tr.loc[undef,'attacker'].mean():.3f}")
print(f"attacker rate overall         : {tr.attacker.mean():.3f}")
print("\n>>> If the first number is far above the second, fillna(0) was")
print(">>> acting as a partial label. Cell 4 measures what that cost.")

# receiver-centric duplication ratio (for the paper)
raw = load(f"data/{SCENARIO}_constantPositionOffset.zip", "Test", 'x')
uniq = raw.groupby(['sender_id','sendTime_s']).ngroups
print(f"\nlog rows {len(raw):,} -> unique transmissions {uniq:,} "
      f"(ratio {len(raw)/uniq:.2f})")
del raw; gc.collect()

In [ ]:
# ============================================================
# CELL 4 — E1: SENTINEL ABLATION.  The blocking experiment.
# ============================================================
def per_attack(pred, data):
    out = {}
    for a in ATTACKS:
        m = ((data.attack_type == a) & (data.attacker == 1)).values
        if m.sum(): out[a] = float(pred[m].mean())
    return out

FOUR = ['constantPositionOffset','timeDelayAttack','dataReplay','positionMirroring']

# Model A — old behaviour: undefined residuals filled with 0
rf_fill = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)
rf_fill.fit(X(tr), tr.attacker)
pred_fill = rf_fill.predict(X(te))

# Model B — clean: rows with undefined residuals dropped from TRAINING
ok = tr[RESID].notna().all(axis=1)
tr_clean = tr[ok]
rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)
rf.fit(X(tr_clean), tr_clean.attacker)
pred = rf.predict(X(te))

rec_fill, rec = per_attack(pred_fill, te), per_attack(pred, te)

print(f"dropped {(~ok).sum():,} training rows ({(~ok).mean():.2%})\n")
print(f"precision   fill-0 {precision_score(te.attacker, pred_fill):.3f}"
      f"   clean {precision_score(te.attacker, pred):.3f}")
print(f"pred rate   fill-0 {pred_fill.mean():.3f}"
      f"   clean {pred.mean():.3f}   actual {te.attacker.mean():.3f}\n")
print(f"{'attack':28s} {'fill-0':>8s} {'clean':>8s} {'delta':>8s}")
for a in sorted(rec, key=rec.get):
    print(f"{a:28s} {rec_fill[a]:8.3f} {rec[a]:8.3f} {rec[a]-rec_fill[a]:+8.3f}")

print("\n>>> The four evasive attacks must STAY below 0.30 in the clean column.")
print(">>> If they do, your headline result is unaffected by the bug.")
print(">>> Cell 9 then re-tests the 0.973 generation result against rf (clean).")

# `rf` (clean) is the detector used for everything downstream.
FP_RATE = pred[(te.attacker == 0).values].mean()
print(f"\nfalse-positive rate (clean model): {FP_RATE:.4f}   <- theta=0 control in Cell 7")

In [ ]:
# ============================================================
# CELL 5 — Detection results, three models, Figures 1-3, Table 1
# ============================================================
models = {
    'RandomForest': rf,
    'DecisionTree': DecisionTreeClassifier(max_depth=12, random_state=SEED)
                        .fit(X(tr_clean), tr_clean.attacker),
    'LogisticReg' : make_pipeline(StandardScaler(),
                        LogisticRegression(max_iter=2000))
                        .fit(X(tr_clean), tr_clean.attacker),
}
res, prec = {}, {}
for nm, mdl in models.items():
    p = mdl.predict(X(te))
    prec[nm] = precision_score(te.attacker, p)
    res[nm]  = per_attack(p, te)
tbl2 = pd.DataFrame(res).sort_values('RandomForest')
print(tbl2.round(3).to_string())
print("\nprecision:", {k: round(v,3) for k,v in prec.items()})

# ---- training presence (kills the "unfamiliarity" explanation) ----
print("\nattack-labelled TRAINING messages:")
for a in FOUR:
    print(f"  {a:28s} {((tr.attack_type==a)&(tr.attacker==1)).sum():7,}")

# ---- sender-level aggregation ----
te = te.copy(); te['pred'] = pred
bs = te[te.attacker==0].groupby('sender_id').pred.mean()
print(f"\n{'':28s} {'msg':>8s} {'sender':>8s} {'>50%':>8s}")
print(f"{'BENIGN':28s} {te[te.attacker==0].pred.mean():8.3f} "
      f"{bs.mean():8.3f} {(bs>0.5).mean():8.3f}")
for a in FOUR + ['randomPositionOffset','zeroSpeedReport']:
    d = te[(te.attack_type==a) & (te.attacker==1)]
    s = d.groupby('sender_id').pred.mean()
    print(f"{a:28s} {d.pred.mean():8.3f} {s.mean():8.3f} {(s>0.5).mean():8.3f}"
          f"   n_veh={len(s)}")

# ---- Table 1: residual ratios ----
base = {c: bt[c].abs().median() for c in RESID}
rows = []
for a in ATTACKS:
    d = te[(te.attack_type==a) & (te.attacker==1)]
    if not len(d): continue
    ratios = [d[c].abs().median()/base[c] if base[c] else np.nan for c in RESID]
    rows.append((a, np.nanmax(ratios), res['RandomForest'][a]))

# ---- FIGURE 1: per-attack recall ----
s = pd.Series(res['RandomForest']).sort_values()
fig, ax = plt.subplots(figsize=(5.4, 3.6))
ax.barh(s.index, s.values,
        color=['#AA2222' if v < 0.3 else '#227722' for v in s.values])
ax.axvline(0.5, ls='--', c='k', lw=0.8)
ax.set_xlabel('Recall', fontsize=9); ax.set_xlim(0,1); ax.tick_params(labelsize=7.5)
plt.tight_layout(); plt.savefig('out/fig1_recall.pdf', bbox_inches='tight'); plt.show()

# ---- FIGURE 2: residual disturbance vs recall ----
pts = [(a,r,v) for a,r,v in rows if not np.isnan(r)]
fig, ax = plt.subplots(figsize=(5.4, 3.4))
for a,r,v in pts:
    ax.scatter(r, v, s=55, color='#AA2222' if v<0.3 else '#227722',
               edgecolor='white', linewidth=0.8, zorder=3)
ax.axvline(1.0, ls='--', c='k', lw=0.9); ax.set_xscale('log')
ax.set_xlabel('Largest residual disturbance (ratio to benign median, log)', fontsize=8.5)
ax.set_ylabel('Recall', fontsize=9); ax.set_ylim(-0.06, 1.10)
ax.grid(alpha=0.22, zorder=0); ax.tick_params(labelsize=8)
plt.tight_layout(); plt.savefig('out/fig2_resid_recall.pdf', bbox_inches='tight'); plt.show()

# ---- FIGURE 3: offset schematic, from a REAL trajectory ----
ex = (te[(te.attacker==0)&(te.attack_type=='constantPositionOffset')]
        .sort_values('sendTime_s').groupby('sender_id').head(6))
ex = ex[ex.sender_id == ex.sender_id.iloc[0]]
t  = np.arange(len(ex)); true_y = ex.s_y.values
rng0 = np.random.default_rng(SEED)
rand_y  = true_y + rng0.uniform(-120, 120, len(ex))
const_y = true_y + 100.0
fig, ax = plt.subplots(figsize=(5.2, 2.4))
ax.plot(t, true_y,  'o-',  color='#333333', lw=1.6, ms=4, label='true')
ax.plot(t, const_y, 's--', color='#227722', lw=1.6, ms=4, label='constant offset')
ax.plot(t, rand_y,  '^:',  color='#AA2222', lw=1.4, ms=4, label='random offset')
ax.set_xlabel('message index', fontsize=9)
ax.set_ylabel('reported position, y (m)', fontsize=9)
ax.legend(fontsize=8, frameon=False); ax.tick_params(labelsize=8)
plt.tight_layout(); plt.savefig('out/fig3_offset.pdf', bbox_inches='tight'); plt.show()

# ---- TRUE offset magnitude (the paper currently reports the wrong quantity) ----
d = te[(te.attack_type=='constantPositionOffset') & (te.attacker==1)]
dx = np.maximum(0, np.maximum(bt.s_x.min()-d.s_x, d.s_x-bt.s_x.max()))
dy = np.maximum(0, np.maximum(bt.s_y.min()-d.s_y, d.s_y-bt.s_y.max()))
outd = np.hypot(dx, dy)
print(f"\nENVELOPE distance (what the draft calls 61.8 m): max {outd.max():.1f} m, "
      f"fraction outside {(outd>0).mean():.3f}")
print(">>> NOTE: this is distance outside a bounding box, NOT offset magnitude.")
print(">>> For true magnitude you need the ground-truth trace. Report it as")
print(">>> 'envelope-anomalous fraction', not as displacement.")

In [ ]:
# ============================================================
# CELL 5b — per-attack AUC (threshold-independent)
# ============================================================
from sklearn.metrics import roc_auc_score

proba = rf.predict_proba(X(te))[:, 1]
ben_p = proba[(te.attacker == 0).values]

print(f"{'attack':28s} {'AUC':>7s} {'recall':>8s}")
auc = {}
for a in ATTACKS:
    m = ((te.attack_type == a) & (te.attacker == 1)).values
    if not m.sum(): continue
    y = np.r_[np.zeros(len(ben_p)), np.ones(m.sum())]
    s = np.r_[ben_p, proba[m]]
    auc[a] = roc_auc_score(y, s)

for a in sorted(auc, key=auc.get):
    print(f"{a:28s} {auc[a]:7.3f} {res['RandomForest'][a]:8.3f}")

print(f"\nbenign message flag rate: {FP_RATE:.4f}")
print(">>> AUC near 0.500 means the detector has NO information about")
print(">>> that attack — not merely a bad threshold. Strongest form of")
print(">>> your claim. If constantPositionOffset lands near 0.50, that")
print(">>> number goes in the abstract.")

pd.Series(auc).to_csv('out/auc_per_attack.csv')

In [ ]:
# ============================================================
# CELL 6 (v2) — E2 part 1: rotation transform + preservation proof
# ============================================================
def rotate(traj, theta_deg):
    d = traj.copy().reset_index(drop=True)
    t = np.radians(theta_deg)
    x0, y0 = d.s_x.iloc[0], d.s_y.iloc[0]
    dx, dy = d.s_x - x0, d.s_y - y0
    d['s_x']   = x0 + dx*np.cos(t) - dy*np.sin(t)
    d['s_y']   = y0 + dx*np.sin(t) + dy*np.cos(t)
    d['s_hed'] = (d.s_hed - theta_deg) % 360      # NOT +theta
    return d

SEQ = 12
ben_pool = te[(te.attacker==0) & (te.attack_type=='constantPositionOffset')]
trajs = [g.sort_values('sendTime_s').head(SEQ)
         for _, g in ben_pool.groupby('sender_id') if len(g) >= SEQ][:40]
print(f"{len(trajs)} benign trajectories of length {SEQ}")

def featurise(traj_list, tag):
    out = []
    for i, t0 in enumerate(traj_list):
        d = t0.copy()
        d['attack_type'] = tag
        d['sender_id']   = f"{tag}_{i:03d}"
        out.append(d)
    return build_features(pd.concat(out, ignore_index=True))

# Tolerances scaled per residual. Coordinates are ~4.5e5, so rotation
# carries ~5e-11 m of rounding; arctan2 amplifies this where consecutive
# displacement is small. acl_resid is EXACT: rotation touches neither
# speed nor acceleration.
TOL = {'spd_resid': 1e-6, 'hed_resid': 1e-3, 'acl_resid': 1e-9}

print(f"\n{'residual':12s} {'max|delta|':>12s} {'benign med':>12s} {'ratio':>10s}  ")
ok_all = True
for th in (90.0, 180.0, 237.0):
    orig = featurise(trajs[:5], 'chk_o')
    rot  = featurise([rotate(t0, th) for t0 in trajs[:5]], 'chk_r')
    print(f"-- theta = {th}")
    for c in RESID:
        delta = np.nanmax(np.abs(orig[c].values - rot[c].values))
        med   = b[c].abs().median()
        flag  = "OK" if delta < TOL[c] else "FAIL"
        if delta >= TOL[c]: ok_all = False
        print(f"  {c:12s} {delta:12.3e} {med:12.5f} {delta/med:10.2e}  {flag}")

# non-residual features must also survive
orig = featurise(trajs[:5], 'chk_o2'); rot = featurise([rotate(t,90.) for t in trajs[:5]], 'chk_r2')
for c in ['disp','implied_spd','dt','s_spd','s_acl','latency']:
    d_ = np.nanmax(np.abs(orig[c].values - rot[c].values))
    print(f"  {c:12s} {d_:12.3e}")

assert ok_all, "a residual moved beyond float precision — transform is wrong"
print("\nAll residuals preserved to float precision at every angle tested.")

# Diagnostic: heading error should concentrate at low displacement
orig = featurise(trajs, 'diag_o')
rot  = featurise([rotate(t0, 90.0) for t0 in trajs], 'diag_r')
err  = np.abs(orig.hed_resid.values - rot.hed_resid.values)
m    = ~np.isnan(err) & ~np.isnan(orig.disp.values)
i    = np.nanargmax(np.where(m, err, np.nan))
print(f"\nlargest heading deviation {err[i]:.3e} deg "
      f"at displacement {orig.disp.values[i]:.3f} m "
      f"(speed {orig.s_spd.values[i]:.3f} m/s)")
print(f"benign median heading residual {b.hed_resid.abs().median():.3f} deg "
      f"-> deviation is {b.hed_resid.abs().median()/err[i]:.0e}x smaller")

In [ ]:
# ============================================================
# CELL 7 (v2) — E2: angle sweep, corrected severity metric
# ============================================================
from sklearn.neighbors import NearestNeighbors

# FIX 1: contiguous trajectories only
def contiguous(g, n):
    g = g.sort_values('sendTime_s')
    d = g.sendTime_s.diff().iloc[1:n]
    return g.head(n) if (d.between(0.8, 1.2)).all() else None

trajs = [t for t in (contiguous(g, SEQ) for _, g in ben_pool.groupby('sender_id'))
         if t is not None and len(t) == SEQ][:40]
print(f"{len(trajs)} contiguous benign trajectories")

# FIX 2: envelope AND point cloud from benign TEST (same split as trajs)
XMIN, XMAX = bt.s_x.min(), bt.s_x.max()
YMIN, YMAX = bt.s_y.min(), bt.s_y.max()
cloud = bt[['s_x','s_y']].dropna().sample(60000, random_state=SEED).values
nn = NearestNeighbors(n_neighbors=1).fit(cloud)

# baseline: how far is benign traffic from other benign traffic?
probe = bt[['s_x','s_y']].dropna().sample(5000, random_state=7).values
base_d = np.median(nn.kneighbors(probe)[0])
print(f"benign-to-benign median NN distance: {base_d:.2f} m\n")

hist, _ = np.histogram(bt.s_hed.dropna(), bins=72, range=(0,360), density=True)
hed_density = lambda v: hist[np.clip(((v % 360)/5).astype(int), 0, 71)]

thetas = np.arange(0, 360, 15)
sweep = []
for th in thetas:
    rot_list = [rotate(t0, float(th)) for t0 in trajs]
    G = featurise(rot_list, f'rot{th:03d}')
    core = (G.groupby('sender_id').cumcount() > 0).values
    row = {'theta': th}
    for nm, mdl in models.items():
        row[nm] = float(mdl.predict(X(G))[core].mean())
    row['hed_density'] = float(np.median(hed_density(G.s_hed.values[core])))
    dx = np.maximum(0, np.maximum(XMIN - G.s_x, G.s_x - XMAX))
    dy = np.maximum(0, np.maximum(YMIN - G.s_y, G.s_y - YMAX))
    row['frac_outside_box'] = float((np.hypot(dx, dy) > 0).mean())
    # FIX 3: real "on the drivable region" measure
    row['nn_median'] = float(np.median(nn.kneighbors(G[['s_x','s_y']].values)[0]))
    row['nn_p90']    = float(np.percentile(nn.kneighbors(G[['s_x','s_y']].values)[0], 90))
    # FIX 4: displacement computed per trajectory, no misalignment
    row['max_disp'] = float(max(
        np.hypot(r.s_x.values - t.s_x.values, r.s_y.values - t.s_y.values).max()
        for r, t in zip(rot_list, trajs)))
    sweep.append(row)

S = pd.DataFrame(sweep)
print(S.round(3).to_string(index=False))
S.to_csv('out/rotation_sweep.csv', index=False)

r0 = S.loc[S.theta == 0, 'RandomForest'].iloc[0]
assert abs(r0 - FP_RATE) < 0.05, "theta=0 must reproduce FP rate"
assert S.loc[S.theta == 0, 'nn_median'].iloc[0] < 1.0, "theta=0 must sit on the road"
print(f"\ncontrols passed: recall {r0:.4f} vs FP {FP_RATE:.4f}, "
      f"NN {S.loc[S.theta==0,'nn_median'].iloc[0]:.3f} m")

fig, ax = plt.subplots(figsize=(5.6, 3.4))
ax.plot(S.theta, S.RandomForest, 'o-', color='#AA2222', lw=1.8, ms=4, label='recall (RF)')
ax.axhline(FP_RATE, ls=':', c='#AA2222', lw=1.2)
ax.text(5, FP_RATE+0.02, 'false-positive rate', fontsize=7.5, color='#AA2222')
ax.set_xlabel(r'rotation angle $\theta$ (degrees)', fontsize=9)
ax.set_ylabel('recall', fontsize=9)
ax.set_xticks(np.arange(0,361,45)); ax.set_ylim(-0.02, 0.35)
ax2 = ax.twinx()
ax2.plot(S.theta, S.nn_median, 's--', color='#2255AA', lw=1.4, ms=3.5,
         label='median distance to benign road positions')
ax2.set_ylabel('distance off benign positions (m)', fontsize=9, color='#2255AA')
ax2.tick_params(labelsize=8, colors='#2255AA')
ax.legend(fontsize=8, frameon=False, loc='upper left')
ax2.legend(fontsize=8, frameon=False, loc='upper right')
ax.grid(alpha=0.2); ax.tick_params(labelsize=8)
plt.tight_layout(); plt.savefig('out/fig4_rotation.pdf', bbox_inches='tight'); plt.show()

i180 = int(np.argmin(np.abs(S.theta.values - 180)))
print(f"\ntheta=180: recall {S.RandomForest.iloc[i180]:.3f}, "
      f"NN median {S.nn_median.iloc[i180]:.2f} m (benign baseline {base_d:.2f} m)")
print(">>> If NN median at 180 is close to the benign baseline, the wrong-way")
print(">>> ghost is ON the road: content-based detection AND road-geometry")
print(">>> validation both fail. That is the severity argument.")

In [ ]:
# ============================================================
# CELL 7d — NN baseline, split by VEHICLE
# ============================================================
from sklearn.neighbors import NearestNeighbors

ben_te = te[te.attacker == 0]
traj_senders = {t.sender_id.iloc[0] for t in trajs}

# unique physical positions, deduplicated across attack subsets
pos = (ben_te[['sender_id','s_x','s_y']].dropna()
       .assign(kx=lambda d: d.s_x.round(2), ky=lambda d: d.s_y.round(2))
       .drop_duplicates(['kx','ky']))
print(f"benign rows {len(ben_te):,} -> unique positions {len(pos):,}")

senders = pos.sender_id.unique()
rs = np.random.default_rng(SEED)
half = set(rs.permutation(senders)[:len(senders)//2])

cloud = pos[pos.sender_id.isin(half)][['s_x','s_y']].values
probe = pos[(~pos.sender_id.isin(half)) &
            (~pos.sender_id.isin(traj_senders))][['s_x','s_y']].values
print(f"cloud {len(cloud):,} positions from {len(half)} vehicles")
print(f"probe {len(probe):,} positions from other vehicles\n")

nn = NearestNeighbors(n_neighbors=1).fit(cloud)
d_ben = nn.kneighbors(probe)[0].ravel()
BEN_MED, BEN_P90 = np.median(d_ben), np.percentile(d_ben, 90)
print(f"BENIGN baseline   median {BEN_MED:6.2f} m   p90 {BEN_P90:6.2f} m   "
      f"p99 {np.percentile(d_ben,99):6.2f} m")

print(f"\n{'theta':>6s} {'recall':>8s} {'NN med':>9s} {'NN p90':>9s} {'x benign':>9s}")
for th in (0, 15, 45, 90, 135, 180, 225, 270, 315):
    G = featurise([rotate(t0, float(th)) for t0 in trajs], f'nn{th:03d}')
    d = nn.kneighbors(G[['s_x','s_y']].values)[0].ravel()
    core = (G.groupby('sender_id').cumcount() > 0).values
    print(f"{th:6d} {rf.predict(X(G))[core].mean():8.3f} "
          f"{np.median(d):9.2f} {np.percentile(d,90):9.2f} "
          f"{np.median(d)/BEN_MED:8.1f}x")

In [ ]:
# ============================================================
# CELL 7e — theta=0 control on THESE trajectories
# ============================================================
G0 = featurise(trajs, 'ctl0')
core0 = (G0.groupby('sender_id').cumcount() > 0).values
ctl = rf.predict(X(G0))[core0].mean()
print(f"theta=0 flag rate on these trajectories: {ctl:.4f}")
print(f"global benign flag rate (all test):      {FP_RATE:.4f}")
print(f"rotation sweep range:                    {S.RandomForest.min():.4f}"
      f" – {S.RandomForest.max():.4f}")
assert abs(S.loc[S.theta==0,'RandomForest'].iloc[0] - ctl) < 0.005
print("\nControl passed. Every rotated angle sits inside the unmodified-traffic"
      "\nflag rate, so rotation produces no detection signal at any angle.")

In [ ]:
# ============================================================
# CELL 7f — FIGURE 4 (final)
# ============================================================
sw = []
for th in np.arange(0, 360, 15):
    G = featurise([rotate(t0, float(th)) for t0 in trajs], f'f4_{th:03d}')
    core = (G.groupby('sender_id').cumcount() > 0).values
    d = nn.kneighbors(G[['s_x','s_y']].values)[0].ravel()
    row = {'theta': th, 'nn_med': np.median(d), 'nn_p90': np.percentile(d, 90)}
    for k, m in models.items():
        row[k] = m.predict(X(G))[core].mean()
    sw.append(row)
S4 = pd.DataFrame(sw)
S4.to_csv('out/rotation_sweep_final.csv', index=False)
print(S4.round(3).to_string(index=False))

fig, (a1, a2) = plt.subplots(2, 1, figsize=(5.4, 4.4), sharex=True,
                             gridspec_kw={'height_ratios': [1, 1]})

a1.plot(S4.theta, S4.RandomForest, 'o-',  color='#AA2222', lw=1.7, ms=4, label='Random Forest')
a1.plot(S4.theta, S4.DecisionTree, 's--', color='#CC7722', lw=1.3, ms=3.5, label='Decision Tree')
a1.plot(S4.theta, S4.LogisticReg,  '^:',  color='#884488', lw=1.3, ms=3.5, label='Logistic Reg.')
a1.axhline(ctl, ls=':', c='#555555', lw=1.1)
a1.text(4, ctl + 0.004, r'unmodified traffic ($\theta=0$)', fontsize=7, color='#555555')
a1.set_ylabel('recall', fontsize=9); a1.set_ylim(-0.005, 0.075)
a1.legend(fontsize=7.5, frameon=False, ncol=3, loc='upper right')
a1.grid(alpha=0.2); a1.tick_params(labelsize=8)

a2.plot(S4.theta, S4.nn_med, 'o-', color='#2255AA', lw=1.7, ms=4, label='median')
a2.plot(S4.theta, S4.nn_p90, 's--', color='#77AADD', lw=1.3, ms=3.5, label='90th pct')
a2.axhline(BEN_MED, ls=':', c='#555555', lw=1.1)
a2.text(4, BEN_MED * 1.25, f'benign baseline {BEN_MED:.2f} m', fontsize=7, color='#555555')
a2.set_yscale('log')
a2.set_ylabel('distance to nearest\nbenign position (m)', fontsize=9)
a2.set_xlabel(r'rotation angle $\theta$ (degrees)', fontsize=9)
a2.set_xticks(np.arange(0, 361, 45))
a2.legend(fontsize=7.5, frameon=False, loc='lower right')
a2.grid(alpha=0.2, which='both'); a2.tick_params(labelsize=8)

plt.tight_layout(); plt.savefig('out/fig4_rotation.pdf', bbox_inches='tight'); plt.show()

print(f"\nrecall range {S4[['RandomForest','DecisionTree','LogisticReg']].values.min():.4f}"
      f" – {S4[['RandomForest','DecisionTree','LogisticReg']].values.max():.4f}"
      f"   control {ctl:.4f}")
print(f"NN median range {S4.nn_med.min():.2f} – {S4.nn_med.max():.2f} m"
      f"   benign {BEN_MED:.2f} m   -> {S4.nn_med.max()/BEN_MED:.0f}x")

In [ ]:
# ============================================================
# CELL 8 — E3: invariance-group coverage across benchmarks
#   NextGen is measured. The other two rows are STRUCTURAL
#   classification you must fill from the published papers.
# ============================================================
# inside  = leaves every first-difference feature unchanged
# partial = reuses genuine trajectories (consistent because it occurred)
# outside = disturbs a residual or a reported-value marginal

nextgen = {
    'constantPositionOffset':     'inside',
    'timeDelayAttack':            'inside',
    'dataReplay':                 'partial',
    'positionMirroring':          'partial',
    'randomPositionOffset':       'outside',
    'constantSpeedOffset':        'outside',
    'randomSpeedOffset':          'outside',
    'zeroSpeedReport':            'outside',
    'suddenStop':                 'outside',
    'suddenConstantSpeed':        'outside',
    'reversedHeading':            'outside',
    'feignedBraking':             'outside',
    'accelerationMultiplication': 'outside',
    'dosAttack':                  'outside',
    'trafficCongestionSybil':     'outside',
}
assert set(nextgen) == set(ATTACKS), "classification must cover all 15"

cnt = pd.Series(nextgen).value_counts()
print("VeReMi NextGen (measured):")
for k in ['inside','partial','outside']:
    print(f"  {k:8s} {cnt.get(k,0):2d} / {len(nextgen)}")

pd.Series(nextgen).to_csv('out/coverage_nextgen.csv')

In [ ]:
# ============================================================
# CELL 9 — Section 7: generation as a NEGATIVE RESULT
#   All bounds from TRAIN benign. Fair baseline over x,y only.
# ============================================================
ACL_MIN, ACL_MAX = b.s_acl.min(), b.s_acl.max()     # TRAIN, not test
SPD_MAX          = b.s_spd.max()
EPS_V, EPS_PSI   = 0.30, 6.0
mad = lambda x: 1.4826 * (x.dropna() - x.dropna().median()).abs().median()
SIG_SPD, SIG_HED = mad(b.spd_resid), mad(b.hed_resid)
COL = {'x':'s_x','y':'s_y','spd':'s_spd','acl':'s_acl','hed':'s_hed'}
print(f"bounds acl [{ACL_MIN:.2f},{ACL_MAX:.2f}]  spd_max {SPD_MAX:.2f}")
print(f"sigma_v {SIG_SPD:.4f}  sigma_psi {SIG_HED:.4f}   (all from TRAIN)\n")

def apply_spec(traj, spec):
    d = traj.copy().reset_index(drop=True); n = len(d); i = np.arange(n); err = []
    for op in spec['ops']:
        f, sh = op.get('field'), op.get('shape')
        if f not in COL: err.append(f"bad field {f}"); continue
        c = COL[f]
        if   sh == 'constant_offset': d[c] = d[c] + op['amount']
        elif sh == 'linear_drift':    d[c] = d[c] + op['rate'] * i
        elif sh == 'sinusoid':        d[c] = d[c] + op['amp']*np.sin(2*np.pi*i/op['period'])
        elif sh == 'step':            d.loc[n//2:, c] = d.loc[n//2:, c] + op['amount']
        elif sh == 'match_drift':
            if f != 'hed': err.append(f"match_drift on {f}")
        elif sh == 'compensate':
            if f != 'spd': err.append(f"compensate on {f}")
        else: err.append(f"bad shape {sh}")
    if any(o.get('shape')=='match_drift' and o.get('field')=='hed' for o in spec['ops']):
        dx, dy = d.s_x.diff(), d.s_y.diff()
        d['s_hed'] = ((90 - np.degrees(np.arctan2(dy, dx))) % 360).bfill()
    if any(o.get('shape')=='compensate' and o.get('field')=='spd' for o in spec['ops']):
        d['s_spd'] = (np.hypot(d.s_x.diff(), d.s_y.diff()) / DT).bfill()
    d['s_acl'] = d.s_spd.diff().div(DT).bfill()
    return d, err

def check(d, orig):
    bad = {}
    if d.s_acl.min() < ACL_MIN-1e-6 or d.s_acl.max() > ACL_MAX+1e-6: bad['accel']=1
    if d.s_spd.min() < -1e-6 or d.s_spd.max() > SPD_MAX: bad['speed']=1
    if (np.hypot(d.s_x.diff(), d.s_y.diff())/DT - d.s_spd).abs().max() > EPS_V: bad['disp']=1
    mh = (90 - np.degrees(np.arctan2(d.s_y.diff(), d.s_x.diff()))) % 360
    hd = (mh - d.s_hed).abs() % 360
    if np.minimum(hd, 360-hd).max() > EPS_PSI: bad['head']=1
    shift = np.hypot(d.s_x - orig.s_x.values, d.s_y - orig.s_y.values).max()
    return bad, shift

SPECS = [
 {"name":"lateral_pivot_drift","ops":[{"field":"y","shape":"linear_drift","rate":0.35},
   {"field":"hed","shape":"match_drift"},{"field":"spd","shape":"compensate"}]},
 {"name":"self_cancelling_bow","ops":[{"field":"x","shape":"sinusoid","amp":15.0,"period":10},
   {"field":"y","shape":"sinusoid","amp":10.0,"period":10},
   {"field":"hed","shape":"match_drift"},{"field":"spd","shape":"compensate"}]},
 {"name":"longitudinal_speed_multiplier","ops":[{"field":"x","shape":"linear_drift","rate":3.5},
   {"field":"y","shape":"linear_drift","rate":-2.0},
   {"field":"hed","shape":"match_drift"},{"field":"spd","shape":"compensate"}]},
 {"name":"lateral_lane_invasion","ops":[{"field":"x","shape":"linear_drift","rate":2.0},
   {"field":"y","shape":"linear_drift","rate":3.5},
   {"field":"hed","shape":"match_drift"},{"field":"spd","shape":"compensate"}]},
 {"name":"high_amplitude_weaving","ops":[{"field":"x","shape":"sinusoid","amp":15.0,"period":20.0},
   {"field":"y","shape":"sinusoid","amp":-15.0,"period":20.0},
   {"field":"hed","shape":"match_drift"},{"field":"spd","shape":"compensate"}]},
 {"name":"sybil_parallel_universe","ops":[{"field":"y","shape":"constant_offset","amount":50.0},
   {"field":"x","shape":"linear_drift","rate":-2.0},
   {"field":"hed","shape":"match_drift"},{"field":"spd","shape":"compensate"}]},
 {"name":"vector_override_turn","ops":[{"field":"x","shape":"linear_drift","rate":-15.0},
   {"field":"y","shape":"linear_drift","rate":-15.0},
   {"field":"hed","shape":"match_drift"},{"field":"spd","shape":"compensate"}]},
 {"name":"offset_with_evasive_lurch","ops":[{"field":"x","shape":"constant_offset","amount":40.0},
   {"field":"y","shape":"step","amount":2.5},
   {"field":"hed","shape":"match_drift"},{"field":"spd","shape":"compensate"}]},
]

llm_shifts = {}
for sp in SPECS:
    out, errs = apply_spec(trajs[0], sp); bad, shift = check(out, trajs[0])
    if not errs and not bad: llm_shifts[sp['name']] = shift
print(f"LLM specs valid: {len(llm_shifts)}/{len(SPECS)}")

# ---- FAIR BASELINE: x,y only. spd/acl are overwritten by the closure ops,
#      so sampling them was giving the baseline inert draws.
rng = np.random.default_rng(SEED)
SHAPES = ['constant_offset','linear_drift','sinusoid','step']
def random_spec(i):
    k = int(rng.integers(1,3)); ops, used = [], set()
    for _ in range(k):
        f = str(rng.choice(['x','y']))          # <-- x,y ONLY
        if f in used: continue
        used.add(f); sh = str(rng.choice(SHAPES)); op = {'field':f,'shape':sh}
        if   sh=='constant_offset': op['amount']=float(rng.uniform(-60,60))
        elif sh=='linear_drift':    op['rate']  =float(rng.uniform(-4,4))
        elif sh=='sinusoid':        op.update(amp=float(rng.uniform(-16,16)),
                                              period=float(rng.choice([4,8,10,20])))
        elif sh=='step':            op['amount']=float(rng.uniform(-30,30))
        ops.append(op)
    ops += [{'field':'hed','shape':'match_drift'},{'field':'spd','shape':'compensate'}]
    return {'name': f'rand_{i:04d}', 'ops': ops}

N_RAND = 1000
rand_shifts = []
for i in range(N_RAND):
    sp = random_spec(i)
    out, errs = apply_spec(trajs[0], sp); bad, shift = check(out, trajs[0])
    if not errs and not bad: rand_shifts.append(shift)
rand_shifts = np.array(rand_shifts)

print(f"random specs valid: {len(rand_shifts)}/{N_RAND}\n")
print(f"{'threshold':>12s} {'LLM (of 8)':>12s} {'random (rate)':>14s}")
for thr in (1.0, 10.0, 20.0):
    l = sum(1 for v in llm_shifts.values() if v > thr)
    r = (rand_shifts > thr).mean()
    print(f"{thr:12.1f} {l:12d} {r:13.1%}")
med_llm = np.median(list(llm_shifts.values()))
print(f"\nmedian displacement  LLM {med_llm:.1f} m   "
      f"random {np.median(rand_shifts):.1f} m")
print(f"LLM median percentile within random distribution: "
      f"{(rand_shifts < med_llm).mean():.1%}")
print("\n>>> If that percentile is not extreme, the LLM has no measured")
print(">>> advantage. Report it. That is the negative result.")

# ---- assemble + detect, through build_features ----
def assemble(seq_len, noise=None, rng_=None, tag='gen'):
    pool = [g.sort_values('sendTime_s').head(seq_len)
            for _, g in ben_pool.groupby('sender_id') if len(g) >= seq_len][:10]
    rows, kept, tried = [], 0, 0
    for sp in SPECS:
        if sp['name'] not in llm_shifts: continue
        for j, t0 in enumerate(pool):
            tried += 1
            out, errs = apply_spec(t0, sp)
            if errs: continue
            if noise is not None:
                n = len(out)
                out['s_spd'] = out.s_spd + rng_.normal(0, noise[0], n)
                out['s_hed'] = (out.s_hed + rng_.normal(0, noise[1], n)) % 360
                out['s_acl'] = out.s_spd.diff().div(DT).bfill()
            bad, shift = check(out, t0)
            if bad: continue
            out = out.copy()
            out['attack_type'] = tag
            out['sender_id']   = f"{tag}_{sp['name']}_{j}"
            rows.append(out); kept += 1
    if not rows: return None, 0, tried
    return build_features(pd.concat(rows, ignore_index=True)), kept, tried

print("\n--- generation against the CLEAN detector ---")
G0, k0, t0n = assemble(10, tag='gen_nonoise')
core0 = (G0.groupby('sender_id').cumcount() > 0).values
print(f"no noise:  {k0}/{t0n} kept   recall {rf.predict(X(G0))[core0].mean():.3f}"
      f"   (fill-0 model: {rf_fill.predict(X(G0))[core0].mean():.3f})")
print(f"  spd_resid {G0.loc[core0,'spd_resid'].abs().median():.4f}"
      f"  vs benign {bt.spd_resid.abs().median():.4f}")
print(">>> If the clean model is FAR below the fill-0 model, the 0.973 was")
print(">>> the sentinel artefact and the claim must come OUT of the paper.")

rngN = np.random.default_rng(SEED)
for L in (10, 30):
    G, k, tn = assemble(L, noise=(SIG_SPD, SIG_HED), rng_=rngN, tag=f'gen{L}')
    if G is None: print(f"len {L}: 0 kept"); continue
    core = (G.groupby('sender_id').cumcount() > 0).values
    print(f"\nlen {L}: {k}/{tn} kept, {core.sum()} core msgs")
    for nm, mdl in models.items():
        print(f"   {nm:14s} recall {mdl.predict(X(G))[core].mean():.3f}")
    # tail check: are generated residuals Gaussian where benign are heavy-tailed?
    from scipy import stats
    ks = stats.ks_2samp(G.loc[core,'spd_resid'].dropna(),
                        bt.spd_resid.dropna().sample(5000, random_state=SEED))
    print(f"   KS vs benign spd_resid: D={ks.statistic:.3f} p={ks.pvalue:.2e}")

In [ ]:
# ============================================================
# CELL 10 — every number the paper needs, in one place
# ============================================================
print("="*62)
print("PAPER NUMBERS")
print("="*62)
print(f"transmissions        train {len(tr):,}   test {len(te):,}")
print(f"benign median        dt {b.dt.median():.3f}s  "
      f"spd_resid {b.spd_resid.abs().median():.4f}  "
      f"hed_resid {b.hed_resid.abs().median():.3f}deg")
print(f"RF precision         {precision_score(te.attacker, pred):.3f}")
print(f"predicted rate       {pred.mean():.3f}  actual {te.attacker.mean():.3f}")
print(f"false-positive rate  {FP_RATE:.4f}")
print("\nfour evasive attacks (clean model):")
for a in FOUR:
    print(f"   {a:28s} {res['RandomForest'][a]:.3f}  "
          f"(train msgs {((tr.attack_type==a)&(tr.attacker==1)).sum():,})")
print(f"\nrotation sweep       min recall {S.RandomForest.min():.3f} "
      f"at theta={int(S.loc[S.RandomForest.idxmin(),'theta'])}")
print(f"                     max recall {S.RandomForest.max():.3f} "
      f"at theta={int(S.loc[S.RandomForest.idxmax(),'theta'])}")
print(f"theta=0 control      {r0:.4f}  (must match FP rate)")
print(f"coverage NextGen     inside {cnt.get('inside',0)}  "
      f"partial {cnt.get('partial',0)}  outside {cnt.get('outside',0)}  of 15")
print("\nfiles written to out/:")
for f in sorted(os.listdir('out')): print("   ", f)
print('''
CHECKLIST BEFORE WRITING
  [ ] Cell 3 asserts passed
  [ ] Cell 4: four attacks still < 0.30 on the CLEAN model
  [ ] Cell 6 assert passed (residuals preserved to 1e-9)
  [ ] Cell 7 theta=0 control matched the FP rate
  [ ] Cell 8 VeReMi / Extension rows filled in by hand
  [ ] Cell 9: decided whether the 0.973 claim survives
''')

In [ ]:
print(pd.Series(rf.feature_importances_, index=FEATS).sort_values().round(4))

In [ ]:
# ============================================================
# CELL F — regenerate ALL FOUR figures from the CLEAN run,
#          sized for ACM sigconf \columnwidth, then download.
#
# Run AFTER cells 1-5, 6v2, 7d, 7e, 7f.
# Needs in memory: res, models, te, bt, b, rf, ATTACKS, RESID,
#                  X, S4, ctl, BEN_MED, trajs, rotate, featurise, nn
# ============================================================
import os, zipfile
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# ACM requires embedded fonts. Type 42 = TrueType, not Type 3.
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype']  = 42
matplotlib.rcParams['font.family']  = 'sans-serif'

COLW = 3.33          # ACM sigconf \columnwidth in inches
RED, GRN, BLU = '#AA2222', '#227722', '#2255AA'
os.makedirs('out', exist_ok=True)

# ---------- shared: per-attack recall + AUC + residual ratios ----------
from sklearn.metrics import roc_auc_score
proba = rf.predict_proba(X(te))[:, 1]
ben_p = proba[(te.attacker == 0).values]
base  = {c: bt[c].abs().median() for c in RESID}

rec, auc, ratio = {}, {}, {}
for a in ATTACKS:
    m = ((te.attack_type == a) & (te.attacker == 1)).values
    if not m.sum():
        continue
    rec[a] = res['RandomForest'][a]
    y = np.r_[np.zeros(len(ben_p)), np.ones(m.sum())]
    auc[a] = roc_auc_score(y, np.r_[ben_p, proba[m]])
    d = te[m]
    r = [d[c].abs().median() / base[c] if base[c] else np.nan for c in RESID]
    ratio[a] = np.nanmax(r)

pred_te = rf.predict(X(te))
FP   = float(pred_te[(te.attacker == 0).values].mean())
PREC = float(precision_score(te.attacker, pred_te))
print(f"benign flag rate {FP:.4f}   RF precision {PREC:.3f}")

# =====================================================================
# FIGURE 1 — per-attack recall, with the benign flag rate marked
# =====================================================================
s = pd.Series(rec).sort_values()
fig, ax = plt.subplots(figsize=(COLW, 3.1))
ax.barh(s.index, s.values,
        color=[RED if v < 0.30 else GRN for v in s.values], height=0.72)
ax.axvline(FP, ls=':', c='#444444', lw=1.0)
ax.text(FP + 0.02, -0.55, f'benign flag rate {FP:.3f}', fontsize=5.8, color='#444444')
ax.set_xlabel('Recall', fontsize=8)
ax.set_xlim(0, 1.02)
ax.tick_params(axis='y', labelsize=5.8)
ax.tick_params(axis='x', labelsize=7)
ax.grid(axis='x', alpha=0.18, zorder=0)
ax.set_axisbelow(True)
plt.tight_layout(pad=0.3)
plt.savefig('out/fig1_recall.pdf', bbox_inches='tight'); plt.show()
print(f"Fig 1  precision {PREC:.3f} ... benign flag rate {FP:.3f}")

# =====================================================================
# FIGURE 2 — residual disturbance vs AUC (AUC, not recall: threshold-free)
# =====================================================================
pts = [(a, ratio[a], auc[a]) for a in ratio if not np.isnan(ratio[a])]
fig, ax = plt.subplots(figsize=(COLW, 2.5))
for a, r, v in pts:
    ax.scatter(r, v, s=42, color=RED if v < 0.70 else GRN,
               edgecolor='white', linewidth=0.7, zorder=3)
ax.axhline(0.5, ls=':', c='#444444', lw=1.0)
ax.text(1.05, 0.52, 'chance', fontsize=6, color='#444444')
ax.axvline(1.0, ls='--', c='#888888', lw=0.8)
ax.set_xscale('log')
ax.set_xlabel('Largest residual disturbance (ratio to benign median)', fontsize=7.5)
ax.set_ylabel('AUC vs benign', fontsize=8)
ax.set_ylim(0.42, 1.04)
ax.grid(alpha=0.18, zorder=0); ax.set_axisbelow(True)
ax.tick_params(labelsize=7)
plt.tight_layout(pad=0.3)
plt.savefig('out/fig2_resid_recall.pdf', bbox_inches='tight'); plt.show()

# =====================================================================
# FIGURE 3 — why uniformity decides detection (real trajectory)
# =====================================================================
ex = trajs[0]
t  = np.arange(len(ex))
ty = ex.s_y.values
rg = np.random.default_rng(42)
fig, ax = plt.subplots(figsize=(COLW, 2.0))
ax.plot(t, ty,        'o-',  color='#333333', lw=1.4, ms=3.2, label='true')
ax.plot(t, ty + 100,  's--', color=GRN,       lw=1.4, ms=3.2, label='constant offset')
ax.plot(t, ty + rg.uniform(-120, 120, len(ex)),
                      '^:',  color=RED,       lw=1.2, ms=3.2, label='random offset')
ax.set_xlabel('message index', fontsize=8)
ax.set_ylabel('reported $y$ (m)', fontsize=8)
ax.legend(fontsize=6.2, frameon=False, loc='best')
ax.tick_params(labelsize=7)
ax.grid(alpha=0.18); ax.set_axisbelow(True)
plt.tight_layout(pad=0.3)
plt.savefig('out/fig3_offset.pdf', bbox_inches='tight'); plt.show()

# =====================================================================
# FIGURE 4 — rotation sweep, two panels (the money figure)
# =====================================================================
fig, (a1, a2) = plt.subplots(2, 1, figsize=(COLW, 3.6), sharex=True)

a1.plot(S4.theta, S4.RandomForest, 'o-',  color=RED,       lw=1.5, ms=3.2, label='RF')
a1.plot(S4.theta, S4.DecisionTree, 's--', color='#CC7722', lw=1.1, ms=2.8, label='DT')
a1.plot(S4.theta, S4.LogisticReg,  '^:',  color='#884488', lw=1.1, ms=2.8, label='LogReg')
a1.axhline(ctl, ls=':', c='#444444', lw=1.0)
a1.text(3, ctl + 0.004,
            r'unmodified traffic ($\theta=0$): ' + f'{ctl:.3f}',
            fontsize=6, color='#444444')
a1.set_ylabel('recall', fontsize=8)
a1.set_ylim(-0.004, 0.062)
a1.legend(fontsize=6.2, frameon=False, ncol=3, loc='upper right')
a1.grid(alpha=0.18); a1.set_axisbelow(True); a1.tick_params(labelsize=7)

a2.plot(S4.theta, S4.nn_med, 'o-',  color=BLU,       lw=1.5, ms=3.2, label='median')
a2.plot(S4.theta, S4.nn_p90, 's--', color='#77AADD', lw=1.1, ms=2.8, label='90th pct')
a2.axhline(BEN_MED, ls=':', c='#444444', lw=1.0)
a2.text(3, BEN_MED * 1.35, f'benign baseline {BEN_MED:.2f} m', fontsize=6, color='#444444')
a2.set_yscale('log')
a2.set_ylabel('dist. to nearest\nbenign position (m)', fontsize=8)
a2.set_xlabel(r'rotation angle $\theta$ (degrees)', fontsize=8)
a2.set_xticks(np.arange(0, 361, 45))
a2.legend(fontsize=6.2, frameon=False, loc='lower right')
a2.grid(alpha=0.18, which='both'); a2.set_axisbelow(True); a2.tick_params(labelsize=7)

plt.tight_layout(pad=0.3, h_pad=0.6)
plt.savefig('out/fig4_rotation.pdf', bbox_inches='tight'); plt.show()

# =====================================================================
# numbers the captions must match
# =====================================================================
print("\n--- CAPTION CHECK ---")
print(f"Fig 1  precision 0.828 ... benign flag rate {FP:.3f}")
print(f"Fig 3  constant {rec['constantPositionOffset']:.3f}  "
      f"random {rec['randomPositionOffset']:.3f}")
print(f"Fig 4  control {ctl:.4f}   recall range "
      f"{S4[['RandomForest','DecisionTree','LogisticReg']].values.min():.4f}"
      f"-{S4[['RandomForest','DecisionTree','LogisticReg']].values.max():.4f}")
print(f"Fig 4  NN {S4.nn_med.min():.2f}-{S4.nn_med.max():.2f} m   "
      f"benign {BEN_MED:.2f} m   factor {S4.nn_med.max()/BEN_MED:.0f}x")

# =====================================================================
# also dump the tables the paper needs, then zip everything
# =====================================================================
pd.DataFrame({'auc': auc, 'recall': rec, 'ratio': ratio}) \
  .sort_values('auc').to_csv('out/table_auc.csv')
pd.DataFrame(res).to_csv('out/table_models.csv')
S4.to_csv('out/rotation_sweep_final.csv', index=False)

FIGS = ['fig1_recall.pdf', 'fig2_resid_recall.pdf',
        'fig3_offset.pdf', 'fig4_rotation.pdf']
TABS = ['table_auc.csv', 'table_models.csv', 'rotation_sweep_final.csv']

with zipfile.ZipFile('aintec_figures.zip', 'w') as z:
    for f in FIGS + TABS:
        p = f'out/{f}'
        if os.path.exists(p):
            z.write(p, f)
            print(f"  packed {f:28s} {os.path.getsize(p)/1024:6.1f} KB")
        else:
            print(f"  MISSING {f}")

from google.colab import files
files.download('aintec_figures.zip')

In [ ]:
# =====================================================================
#  SUPPLEMENTARY EXPERIMENTS — run in order.
#
#  S1  Sequence model (GRU).
#  S2  Permutation importance.
#  S3  Figure corrections.
#  S4  Second density (OPTIONAL).
#
#  Needs in memory from the main notebook: tr, te, b, bt, rf, models,
#  res, ATTACKS, FEATS, RESID, X, GAP_MAX, SEED, trajs, rotate,
#  featurise, S4 (sweep), ctl, BEN_MED, nn
# =====================================================================


# =====================================================================
# CELL S1 — GRU over windows of the SAME features, and a second GRU
#           with absolute position added.
#
#  Two questions, two answers:
#   (a) Does a sequence model over difference-derived features inherit
#       the blind spot?  Proposition predicts YES.
#   (b) Does adding absolute position break it, and does that break
#       transfer across the geographic split?  Predicts: catches the
#       attack, fails to transfer.  This closes the loop on Section 4.1.
# =====================================================================
import numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.metrics import roc_auc_score, precision_score

torch.manual_seed(SEED); np.random.seed(SEED)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", DEV)

W       = 10     # window length, messages
STRIDE  = 5      # training stride (test uses stride 1 for coverage)
EPOCHS  = 6
BATCH   = 512


def make_windows(df, feats, w=W, stride=1):
    """Contiguous windows per (attack_type, sender_id). Label = last message."""
    Xs, ys, ats = [], [], []
    cols = feats + ['dt', 'attacker']
    for (at, sid), g in df.groupby(['attack_type', 'sender_id'], sort=False):
        g = g.sort_values('sendTime_s')
        if len(g) < w:
            continue
        M   = g[feats].replace([np.inf, -np.inf], np.nan).fillna(0.0).values.astype('float32')
        dtv = g['dt'].values
        lab = g['attacker'].values
        for s in range(0, len(g) - w + 1, stride):
            # reject windows spanning a gap
            if np.nanmax(dtv[s + 1:s + w]) > GAP_MAX:
                continue
            Xs.append(M[s:s + w]); ys.append(lab[s + w - 1]); ats.append(at)
    return (np.stack(Xs), np.array(ys, dtype='float32'), np.array(ats))


class GRUDet(nn.Module):
    def __init__(self, d, h=64):
        super().__init__()
        self.gru = nn.GRU(d, h, num_layers=1, batch_first=True)
        self.fc  = nn.Sequential(nn.Linear(h, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        o, _ = self.gru(x)
        return self.fc(o[:, -1]).squeeze(-1)


def run_gru(feats, tag):
    Xtr, ytr, _    = make_windows(tr, feats, stride=STRIDE)
    Xte, yte, atte = make_windows(te, feats, stride=1)
    print(f"\n[{tag}]  train windows {len(Xtr):,}   test windows {len(Xte):,}")

    mu, sd = Xtr.reshape(-1, Xtr.shape[-1]).mean(0), Xtr.reshape(-1, Xtr.shape[-1]).std(0) + 1e-8
    Xtr = (Xtr - mu) / sd
    Xte = (Xte - mu) / sd

    m   = GRUDet(Xtr.shape[-1]).to(DEV)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    pw  = torch.tensor([(1 - ytr.mean()) / ytr.mean()], device=DEV)
    lf  = nn.BCEWithLogitsLoss(pos_weight=pw)

    Xt = torch.tensor(Xtr); yt = torch.tensor(ytr)
    for ep in range(EPOCHS):
        m.train(); perm = torch.randperm(len(Xt)); tot = 0.0
        for i in range(0, len(Xt), BATCH):
            j  = perm[i:i + BATCH]
            xb, yb = Xt[j].to(DEV), yt[j].to(DEV)
            opt.zero_grad(); l = lf(m(xb), yb); l.backward(); opt.step()
            tot += l.item() * len(j)
        print(f"   epoch {ep+1}/{EPOCHS}  loss {tot/len(Xt):.4f}")

    m.eval(); scores = []
    with torch.no_grad():
        Xv = torch.tensor(Xte)
        for i in range(0, len(Xv), 4096):
            scores.append(torch.sigmoid(m(Xv[i:i+4096].to(DEV))).cpu().numpy())
    p = np.concatenate(scores)
    pred = (p > 0.5).astype(int)

    ben = p[yte == 0]
    print(f"   precision {precision_score(yte, pred):.3f}   "
          f"benign flag rate {pred[yte == 0].mean():.4f}")
    out = {}
    for a in ATTACKS:
        mk = (atte == a) & (yte == 1)
        if not mk.sum(): continue
        yy = np.r_[np.zeros(len(ben)), np.ones(mk.sum())]
        out[a] = (roc_auc_score(yy, np.r_[ben, p[mk]]), float(pred[mk].mean()))
    return m, out, (mu, sd), float(pred[yte == 0].mean())


# ---- (a) same features as the tree models -------------------------
gru_a, res_a, norm_a, fp_a = run_gru(FEATS, 'GRU / difference features')
print(f"\n{'attack':28s} {'AUC':>7s} {'recall':>8s}   (RF AUC for comparison)")
for a in sorted(res_a, key=lambda k: res_a[k][0]):
    print(f"{a:28s} {res_a[a][0]:7.3f} {res_a[a][1]:8.3f}")

print("\n>>> PREDICTION: constantPositionOffset and timeDelayAttack stay")
print(">>> near AUC 0.50. If so, the sequence model inherits the blind")
print(">>> spot and Proposition 1 is confirmed empirically at W=10.")

# ---- (b) same features PLUS absolute position ---------------------
FEATS_POS = FEATS + ['s_x', 's_y']
gru_b, res_b, norm_b, fp_b = run_gru(FEATS_POS, 'GRU / + absolute position')
print(f"\n{'attack':28s} {'AUC':>7s} {'recall':>8s}")
for a in sorted(res_b, key=lambda k: res_b[k][0]):
    print(f"{a:28s} {res_b[a][0]:7.3f} {res_b[a][1]:8.3f}")

print("\n>>> If constantPositionOffset rises sharply here, absolute position")
print(">>> DOES break the invariance -- but note the benign flag rate:")
print(f">>>   diff-features benign flag rate  {fp_a:.4f}")
print(f">>>   +position     benign flag rate  {fp_b:.4f}")
print(">>> A large rise means the model is flagging legitimate test-region")
print(">>> traffic as anomalous because the test region is geographically")
print(">>> disjoint from training. That is the Section 4.1 argument,")
print(">>> demonstrated rather than asserted.")

pd.DataFrame({
    'gru_diff_auc':  {a: v[0] for a, v in res_a.items()},
    'gru_diff_rec':  {a: v[1] for a, v in res_a.items()},
    'gru_pos_auc':   {a: v[0] for a, v in res_b.items()},
    'gru_pos_rec':   {a: v[1] for a, v in res_b.items()},
}).to_csv('out/table_sequence.csv')

# ---- rotation against the GRU (does the prediction still hold?) ----
def gru_rotation(model, feats, norm, tag):
    mu, sd = norm
    rows = []
    for th in (0, 45, 90, 135, 180, 225, 270, 315):
        G = featurise([rotate(t0, float(th)) for t0 in trajs], f'gr{th:03d}')
        Xw, yw, _ = make_windows(G, feats, w=min(W, 10), stride=1)
        if len(Xw) == 0:
            rows.append((th, np.nan)); continue
        Xw = (Xw - mu) / sd
        model.eval()
        with torch.no_grad():
            pr = torch.sigmoid(model(torch.tensor(Xw).to(DEV))).cpu().numpy()
        rows.append((th, float((pr > 0.5).mean())))
    print(f"\n[{tag}] rotation sweep:")
    for th, r in rows:
        print(f"   theta {th:3d}   flag rate {r:.4f}")
    return rows

rot_gru = gru_rotation(gru_a, FEATS, norm_a, 'GRU / difference features')
pd.DataFrame(rot_gru, columns=['theta', 'flag_rate']).to_csv('out/rotation_gru.csv', index=False)


# =====================================================================
# CELL S2 — permutation importance (unbiased) vs Gini
#   R3: "Gini importance is biased toward high-cardinality continuous
#        features." s_hed IS continuous and high-cardinality, so any
#        Gini bias INFLATES it -- and it still ranks last. Permutation
#        importance settles it.
# =====================================================================
from sklearn.inspection import permutation_importance

sub = te.sample(30000, random_state=SEED)
pi  = permutation_importance(rf, X(sub), sub.attacker,
                             n_repeats=5, random_state=SEED, n_jobs=-1,
                             scoring='roc_auc')

imp = pd.DataFrame({
    'gini':        pd.Series(rf.feature_importances_, index=FEATS),
    'perm_mean':   pd.Series(pi.importances_mean, index=FEATS),
    'perm_std':    pd.Series(pi.importances_std,  index=FEATS),
}).sort_values('perm_mean')
print(imp.round(4).to_string())

RESID_F = ['spd_resid', 'hed_resid', 'acl_resid']
DIFF_F  = ['disp', 'implied_spd', 'dt']
REPORT_F= ['s_spd', 's_acl', 's_hed']
for nm, grp in [('residuals', RESID_F), ('other difference-derived', DIFF_F),
                ('reported values', REPORT_F), ('latency', ['latency'])]:
    g = imp.loc[grp, 'gini'].sum()
    p = imp.loc[grp, 'perm_mean'].sum() / imp.perm_mean.clip(lower=0).sum()
    print(f"{nm:26s}  gini {g:6.1%}   perm {p:6.1%}")

print(f"\ns_hed rank (gini): {list(imp.sort_values('gini').index).index('s_hed')+1} of 10")
print(f"s_hed rank (perm): {list(imp.index).index('s_hed')+1} of 10")
imp.to_csv('out/feature_importance.csv')


# =====================================================================
# CELL S3 — figure corrections
#   Fig 1: y-tick labels overlapped the bars -> more left margin.
#   Fig 3: caption said "against recall"; the axis is AUC. Axis label
#          and file are now unambiguous.
# =====================================================================
import matplotlib, matplotlib.pyplot as plt
matplotlib.rcParams['pdf.fonttype'] = 42
COLW, RED, GRN = 3.33, '#AA2222', '#227722'

pred_te = rf.predict(X(te))
FP = float(pred_te[(te.attacker == 0).values].mean())
rec = {a: res['RandomForest'][a] for a in res['RandomForest']}

# --- FIG 1 ---
s = pd.Series(rec).sort_values()
fig, ax = plt.subplots(figsize=(COLW, 3.2))
ax.barh(range(len(s)), s.values,
        color=[RED if v < 0.30 else GRN for v in s.values], height=0.70)
ax.set_yticks(range(len(s)))
ax.set_yticklabels(s.index, fontsize=5.6)
ax.set_ylim(-0.7, len(s) - 0.3)
ax.axvline(FP, ls=':', c='#444444', lw=1.0)
ax.text(FP + 0.03, len(s) - 1.3, f'benign flag rate {FP:.3f}',
        fontsize=5.6, color='#444444')
ax.set_xlabel('Recall', fontsize=8); ax.set_xlim(0, 1.02)
ax.tick_params(axis='x', labelsize=7)
ax.grid(axis='x', alpha=0.18); ax.set_axisbelow(True)
plt.subplots_adjust(left=0.42)
plt.savefig('out/fig1_recall.pdf', bbox_inches='tight'); plt.show()

# --- FIG 3 (AUC axis, unambiguous) ---
from sklearn.metrics import roc_auc_score
proba = rf.predict_proba(X(te))[:, 1]
ben_p = proba[(te.attacker == 0).values]
base  = {c: bt[c].abs().median() for c in RESID}
pts = []
for a in ATTACKS:
    m = ((te.attack_type == a) & (te.attacker == 1)).values
    if not m.sum(): continue
    r = np.nanmax([te[m][c].abs().median() / base[c] if base[c] else np.nan for c in RESID])
    if np.isnan(r): continue
    yy = np.r_[np.zeros(len(ben_p)), np.ones(m.sum())]
    pts.append((r, roc_auc_score(yy, np.r_[ben_p, proba[m]])))

fig, ax = plt.subplots(figsize=(COLW, 2.5))
for r, v in pts:
    ax.scatter(r, v, s=42, color=RED if v < 0.70 else GRN,
               edgecolor='white', linewidth=0.7, zorder=3)
ax.axhline(0.5, ls=':', c='#444444', lw=1.0)
ax.text(1.15, 0.515, 'chance', fontsize=6, color='#444444')
ax.axvline(1.0, ls='--', c='#888888', lw=0.8)
ax.set_xscale('log')
ax.set_xlabel('Largest residual disturbance (ratio to benign median)', fontsize=7.5)
ax.set_ylabel('AUC against benign traffic', fontsize=8)
ax.set_ylim(0.42, 1.04)
ax.grid(alpha=0.18); ax.set_axisbelow(True); ax.tick_params(labelsize=7)
plt.tight_layout(pad=0.3)
plt.savefig('out/fig3_auc_resid.pdf', bbox_inches='tight'); plt.show()
print("\nNOTE: fig2_resid_recall.pdf is now fig3_auc_resid.pdf. "
      "Update \\includegraphics in main.tex.")


# =====================================================================
# CELL S4 — second density check. Run if
#           still have days left. Answers "one scenario only".
# =====================================================================
"""
Change SCENARIO in Cell 1 to 'InTAS_highway_1' (or another density that
exists on the Zenodo record), then re-run cells 1-5 into fresh names:

    SCENARIO = "InTAS_highway_1"
    tr1, te1 = build_features(load_split("Train")), build_features(load_split("Test"))
    ok1 = tr1[RESID].notna().all(axis=1)
    rf1 = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)
    rf1.fit(X(tr1[ok1]), tr1[ok1].attacker)
    p1  = rf1.predict_proba(X(te1))[:,1]
    ben1 = p1[(te1.attacker==0).values]
    for a in ['constantPositionOffset','timeDelayAttack','randomPositionOffset']:
        m = ((te1.attack_type==a) & (te1.attacker==1)).values
        yy = np.r_[np.zeros(len(ben1)), np.ones(m.sum())]
        print(a, round(roc_auc_score(yy, np.r_[ben1, p1[m]]), 3))

One sentence in the paper is enough: "At density 1 the same two attacks
are detected at AUC X and Y."  Do NOT rebuild the whole analysis.
"""

In [ ]:
# =====================================================================
#  SUPPLEMENTARY EXPERIMENTS — run in order.
#
#  S1  Sequence model (GRU).
#  S2  Permutation importance.
#  S3  Figure corrections.
#  S4  Second density (OPTIONAL).
#
#  Needs in memory from the main notebook: tr, te, b, bt, rf, models,
#  res, ATTACKS, FEATS, RESID, X, GAP_MAX, SEED, trajs, rotate,
#  featurise, S4 (sweep), ctl, BEN_MED, nn
# =====================================================================


# =====================================================================
# CELL S1 — GRU over windows of the SAME features, and a second GRU
#           with absolute position added.
#
#  Two questions, two answers:
#   (a) Does a sequence model over difference-derived features inherit
#       the blind spot?  Proposition predicts YES.
#   (b) Does adding absolute position break it, and does that break
#       transfer across the geographic split?  Predicts: catches the
#       attack, fails to transfer.  This closes the loop on Section 4.1.
# =====================================================================
import numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.metrics import roc_auc_score, precision_score

torch.manual_seed(SEED); np.random.seed(SEED)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", DEV)

W       = 10     # window length, messages
STRIDE  = 5      # training stride (test uses stride 1 for coverage)
EPOCHS  = 6
BATCH   = 512


def make_windows(df, feats, w=W, stride=1):
    """Contiguous windows per (attack_type, sender_id). Label = last message."""
    Xs, ys, ats = [], [], []
    cols = feats + ['dt', 'attacker']
    for (at, sid), g in df.groupby(['attack_type', 'sender_id'], sort=False):
        g = g.sort_values('sendTime_s')
        if len(g) < w:
            continue
        M   = g[feats].replace([np.inf, -np.inf], np.nan).fillna(0.0).values.astype('float32')
        dtv = g['dt'].values
        lab = g['attacker'].values
        for s in range(0, len(g) - w + 1, stride):
            # reject windows spanning a gap
            if np.nanmax(dtv[s + 1:s + w]) > GAP_MAX:
                continue
            Xs.append(M[s:s + w]); ys.append(lab[s + w - 1]); ats.append(at)
    return (np.stack(Xs), np.array(ys, dtype='float32'), np.array(ats))


class GRUDet(nn.Module):
    def __init__(self, d, h=64):
        super().__init__()
        self.gru = nn.GRU(d, h, num_layers=1, batch_first=True)
        self.fc  = nn.Sequential(nn.Linear(h, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        o, _ = self.gru(x)
        return self.fc(o[:, -1]).squeeze(-1)


def run_gru(feats, tag):
    Xtr, ytr, _    = make_windows(tr, feats, stride=STRIDE)
    Xte, yte, atte = make_windows(te, feats, stride=1)
    print(f"\n[{tag}]  train windows {len(Xtr):,}   test windows {len(Xte):,}")

    mu, sd = Xtr.reshape(-1, Xtr.shape[-1]).mean(0), Xtr.reshape(-1, Xtr.shape[-1]).std(0) + 1e-8
    Xtr = (Xtr - mu) / sd
    Xte = (Xte - mu) / sd

    m   = GRUDet(Xtr.shape[-1]).to(DEV)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    pw  = torch.tensor([(1 - ytr.mean()) / ytr.mean()], device=DEV)
    lf  = nn.BCEWithLogitsLoss(pos_weight=pw)

    Xt = torch.tensor(Xtr); yt = torch.tensor(ytr)
    for ep in range(EPOCHS):
        m.train(); perm = torch.randperm(len(Xt)); tot = 0.0
        for i in range(0, len(Xt), BATCH):
            j  = perm[i:i + BATCH]
            xb, yb = Xt[j].to(DEV), yt[j].to(DEV)
            opt.zero_grad(); l = lf(m(xb), yb); l.backward(); opt.step()
            tot += l.item() * len(j)
        print(f"   epoch {ep+1}/{EPOCHS}  loss {tot/len(Xt):.4f}")

    m.eval(); scores = []
    with torch.no_grad():
        Xv = torch.tensor(Xte)
        for i in range(0, len(Xv), 4096):
            scores.append(torch.sigmoid(m(Xv[i:i+4096].to(DEV))).cpu().numpy())
    p = np.concatenate(scores)
    pred = (p > 0.5).astype(int)

    ben = p[yte == 0]
    print(f"   precision {precision_score(yte, pred):.3f}   "
          f"benign flag rate {pred[yte == 0].mean():.4f}")
    out = {}
    for a in ATTACKS:
        mk = (atte == a) & (yte == 1)
        if not mk.sum(): continue
        yy = np.r_[np.zeros(len(ben)), np.ones(mk.sum())]
        out[a] = (roc_auc_score(yy, np.r_[ben, p[mk]]), float(pred[mk].mean()))
    return m, out, (mu, sd), float(pred[yte == 0].mean())


# ---- (a) same features as the tree models -------------------------
gru_a, res_a, norm_a, fp_a = run_gru(FEATS, 'GRU / difference features')
print(f"\n{'attack':28s} {'AUC':>7s} {'recall':>8s}   (RF AUC for comparison)")
for a in sorted(res_a, key=lambda k: res_a[k][0]):
    print(f"{a:28s} {res_a[a][0]:7.3f} {res_a[a][1]:8.3f}")

print("\n>>> PREDICTION: constantPositionOffset and timeDelayAttack stay")
print(">>> near AUC 0.50. If so, the sequence model inherits the blind")
print(">>> spot and Proposition 1 is confirmed empirically at W=10.")

# ---- (b) same features PLUS absolute position ---------------------
FEATS_POS = FEATS + ['s_x', 's_y']
gru_b, res_b, norm_b, fp_b = run_gru(FEATS_POS, 'GRU / + absolute position')
print(f"\n{'attack':28s} {'AUC':>7s} {'recall':>8s}")
for a in sorted(res_b, key=lambda k: res_b[k][0]):
    print(f"{a:28s} {res_b[a][0]:7.3f} {res_b[a][1]:8.3f}")

print("\n>>> If constantPositionOffset rises sharply here, absolute position")
print(">>> DOES break the invariance -- but note the benign flag rate:")
print(f">>>   diff-features benign flag rate  {fp_a:.4f}")
print(f">>>   +position     benign flag rate  {fp_b:.4f}")
print(">>> A large rise means the model is flagging legitimate test-region")
print(">>> traffic as anomalous because the test region is geographically")
print(">>> disjoint from training. That is the Section 4.1 argument,")
print(">>> demonstrated rather than asserted.")

pd.DataFrame({
    'gru_diff_auc':  {a: v[0] for a, v in res_a.items()},
    'gru_diff_rec':  {a: v[1] for a, v in res_a.items()},
    'gru_pos_auc':   {a: v[0] for a, v in res_b.items()},
    'gru_pos_rec':   {a: v[1] for a, v in res_b.items()},
}).to_csv('out/table_sequence.csv')

# ---- rotation against the GRU (does the prediction still hold?) ----
def gru_rotation(model, feats, norm, tag):
    mu, sd = norm
    rows = []
    for th in (0, 45, 90, 135, 180, 225, 270, 315):
        G = featurise([rotate(t0, float(th)) for t0 in trajs], f'gr{th:03d}')
        Xw, yw, _ = make_windows(G, feats, w=min(W, 10), stride=1)
        if len(Xw) == 0:
            rows.append((th, np.nan)); continue
        Xw = (Xw - mu) / sd
        model.eval()
        with torch.no_grad():
            pr = torch.sigmoid(model(torch.tensor(Xw).to(DEV))).cpu().numpy()
        rows.append((th, float((pr > 0.5).mean())))
    print(f"\n[{tag}] rotation sweep:")
    for th, r in rows:
        print(f"   theta {th:3d}   flag rate {r:.4f}")
    return rows

rot_gru = gru_rotation(gru_a, FEATS, norm_a, 'GRU / difference features')
pd.DataFrame(rot_gru, columns=['theta', 'flag_rate']).to_csv('out/rotation_gru.csv', index=False)


# =====================================================================
# CELL S2 — permutation importance (unbiased) vs Gini
#   R3: "Gini importance is biased toward high-cardinality continuous
#        features." s_hed IS continuous and high-cardinality, so any
#        Gini bias INFLATES it -- and it still ranks last. Permutation
#        importance settles it.
# =====================================================================
from sklearn.inspection import permutation_importance

sub = te.sample(30000, random_state=SEED)
pi  = permutation_importance(rf, X(sub), sub.attacker,
                             n_repeats=5, random_state=SEED, n_jobs=-1,
                             scoring='roc_auc')

imp = pd.DataFrame({
    'gini':        pd.Series(rf.feature_importances_, index=FEATS),
    'perm_mean':   pd.Series(pi.importances_mean, index=FEATS),
    'perm_std':    pd.Series(pi.importances_std,  index=FEATS),
}).sort_values('perm_mean')
print(imp.round(4).to_string())

RESID_F = ['spd_resid', 'hed_resid', 'acl_resid']
DIFF_F  = ['disp', 'implied_spd', 'dt']
REPORT_F= ['s_spd', 's_acl', 's_hed']
for nm, grp in [('residuals', RESID_F), ('other difference-derived', DIFF_F),
                ('reported values', REPORT_F), ('latency', ['latency'])]:
    g = imp.loc[grp, 'gini'].sum()
    p = imp.loc[grp, 'perm_mean'].sum() / imp.perm_mean.clip(lower=0).sum()
    print(f"{nm:26s}  gini {g:6.1%}   perm {p:6.1%}")

print(f"\ns_hed rank (gini): {list(imp.sort_values('gini').index).index('s_hed')+1} of 10")
print(f"s_hed rank (perm): {list(imp.index).index('s_hed')+1} of 10")
imp.to_csv('out/feature_importance.csv')


# =====================================================================
# CELL S3 — figure corrections
#   Fig 1: y-tick labels overlapped the bars -> more left margin.
#   Fig 3: caption said "against recall"; the axis is AUC. Axis label
#          and file are now unambiguous.
# =====================================================================
import matplotlib, matplotlib.pyplot as plt
matplotlib.rcParams['pdf.fonttype'] = 42
COLW, RED, GRN = 3.33, '#AA2222', '#227722'

pred_te = rf.predict(X(te))
FP = float(pred_te[(te.attacker == 0).values].mean())
rec = {a: res['RandomForest'][a] for a in res['RandomForest']}

# --- FIG 1 ---
s = pd.Series(rec).sort_values()
fig, ax = plt.subplots(figsize=(COLW, 3.2))
ax.barh(range(len(s)), s.values,
        color=[RED if v < 0.30 else GRN for v in s.values], height=0.70)
ax.set_yticks(range(len(s)))
ax.set_yticklabels(s.index, fontsize=5.6)
ax.set_ylim(-0.7, len(s) - 0.3)
ax.axvline(FP, ls=':', c='#444444', lw=1.0)
ax.text(FP + 0.03, len(s) - 1.3, f'benign flag rate {FP:.3f}',
        fontsize=5.6, color='#444444')
ax.set_xlabel('Recall', fontsize=8); ax.set_xlim(0, 1.02)
ax.tick_params(axis='x', labelsize=7)
ax.grid(axis='x', alpha=0.18); ax.set_axisbelow(True)
plt.subplots_adjust(left=0.42)
plt.savefig('out/fig1_recall.pdf', bbox_inches='tight'); plt.show()

# --- FIG 3 (AUC axis, unambiguous) ---
from sklearn.metrics import roc_auc_score
proba = rf.predict_proba(X(te))[:, 1]
ben_p = proba[(te.attacker == 0).values]
base  = {c: bt[c].abs().median() for c in RESID}
pts = []
for a in ATTACKS:
    m = ((te.attack_type == a) & (te.attacker == 1)).values
    if not m.sum(): continue
    r = np.nanmax([te[m][c].abs().median() / base[c] if base[c] else np.nan for c in RESID])
    if np.isnan(r): continue
    yy = np.r_[np.zeros(len(ben_p)), np.ones(m.sum())]
    pts.append((r, roc_auc_score(yy, np.r_[ben_p, proba[m]])))

fig, ax = plt.subplots(figsize=(COLW, 2.5))
for r, v in pts:
    ax.scatter(r, v, s=42, color=RED if v < 0.70 else GRN,
               edgecolor='white', linewidth=0.7, zorder=3)
ax.axhline(0.5, ls=':', c='#444444', lw=1.0)
ax.text(1.15, 0.515, 'chance', fontsize=6, color='#444444')
ax.axvline(1.0, ls='--', c='#888888', lw=0.8)
ax.set_xscale('log')
ax.set_xlabel('Largest residual disturbance (ratio to benign median)', fontsize=7.5)
ax.set_ylabel('AUC against benign traffic', fontsize=8)
ax.set_ylim(0.42, 1.04)
ax.grid(alpha=0.18); ax.set_axisbelow(True); ax.tick_params(labelsize=7)
plt.tight_layout(pad=0.3)
plt.savefig('out/fig3_auc_resid.pdf', bbox_inches='tight'); plt.show()
print("\nNOTE: fig2_resid_recall.pdf is now fig3_auc_resid.pdf. "
      "Update \\includegraphics in main.tex.")


# =====================================================================
# CELL S4 — second density check. Run if
#           still have days left. Answers "one scenario only".
# =====================================================================
"""
Change SCENARIO in Cell 1 to 'InTAS_highway_1' (or another density that
exists on the Zenodo record), then re-run cells 1-5 into fresh names:

    SCENARIO = "InTAS_highway_1"
    tr1, te1 = build_features(load_split("Train")), build_features(load_split("Test"))
    ok1 = tr1[RESID].notna().all(axis=1)
    rf1 = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)
    rf1.fit(X(tr1[ok1]), tr1[ok1].attacker)
    p1  = rf1.predict_proba(X(te1))[:,1]
    ben1 = p1[(te1.attacker==0).values]
    for a in ['constantPositionOffset','timeDelayAttack','randomPositionOffset']:
        m = ((te1.attack_type==a) & (te1.attacker==1)).values
        yy = np.r_[np.zeros(len(ben1)), np.ones(m.sum())]
        print(a, round(roc_auc_score(yy, np.r_[ben1, p1[m]]), 3))

One sentence in the paper is enough: "At density 1 the same two attacks
are detected at AUC X and Y."  Do NOT rebuild the whole analysis.
"""

In [ ]:
# --- FIG 1 ---
s = pd.Series(rec).sort_values()
fig, ax = plt.subplots(figsize=(COLW, 3.2))
ax.barh(range(len(s)), s.values,
        color=[RED if v < 0.30 else GRN for v in s.values], height=0.70)
ax.set_yticks(range(len(s)))
ax.set_yticklabels(s.index, fontsize=5.6)
ax.set_ylim(-0.7, len(s) - 0.3)
ax.axvline(FP, ls=':', c='#444444', lw=1.0)
ax.text(FP + 0.03, len(s) - 1.3, f'benign flag rate {FP:.3f}',
        fontsize=5.6, color='#444444')
ax.set_xlabel('Recall', fontsize=8); ax.set_xlim(0, 1.02)
ax.tick_params(axis='x', labelsize=7)
ax.grid(axis='x', alpha=0.18); ax.set_axisbelow(True)
plt.subplots_adjust(left=0.42)
plt.savefig('out/fig1_recall.pdf', bbox_inches='tight'); plt.show()

In [ ]:
import os, zipfile
from google.colab import files

FIGS = ['fig1_recall.pdf', 'fig2_offset.pdf',
        'fig3_auc_resid.pdf', 'fig4_rotation.pdf']

# older runs named these differently — rename if present
RENAMES = {'fig3_offset.pdf': 'fig2_offset.pdf',
           'fig2_resid_recall.pdf': 'fig3_auc_resid.pdf'}
for old, new in RENAMES.items():
    if os.path.exists(f'out/{old}') and not os.path.exists(f'out/{new}'):
        os.rename(f'out/{old}', f'out/{new}')
        print(f"renamed {old} -> {new}")

print("\nfiles in out/:")
for f in sorted(os.listdir('out')):
    print(f"   {f:28s} {os.path.getsize(f'out/{f}')/1024:7.1f} KB")

missing = [f for f in FIGS if not os.path.exists(f'out/{f}')]
if missing:
    print(f"\nMISSING: {missing}")
    print("Re-run cell S3 to regenerate.")
else:
    with zipfile.ZipFile('aintec_figs.zip', 'w') as z:
        for f in FIGS:
            z.write(f'out/{f}', f)
        for t in ['table_auc.csv', 'table_models.csv',
                  'table_sequence.csv', 'feature_importance.csv',
                  'rotation_sweep_final.csv', 'rotation_gru.csv']:
            if os.path.exists(f'out/{t}'):
                z.write(f'out/{t}', t)
    print("\nall four figures packed")
    files.download('aintec_figs.zip')

In [ ]:
# =====================================================================
#  SECOND ROUND — four experiments, in priority order.
#
#   E-A  Why is AUC systematically BELOW 0.5?
#   E-B  Does populating the heading marginal
#        restore rotation detection?
#   E-C  Scale the rotation sweep + bootstrap CIs
#   E-D  Second scenario
#
#  Needs from the main notebook: tr, te, b, bt, rf, models, res, X,
#  ATTACKS, FEATS, RESID, GAP_MAX, SEED, rotate, featurise, build_features,
#  load_split, ben_pool
# =====================================================================
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score
from scipy import stats


# =====================================================================
# E-A — WHY BELOW CHANCE?
#
#  Hypothesis: the attack applies its offset to the GROUND-TRUTH
#  position rather than to the noisy reported one, so attack
#  trajectories carry LESS sensor noise than benign traffic. The
#  detector then ranks them as *more* benign than benign, which is
#  exactly AUC < 0.5.
#
#  If true: attack residuals are stochastically SMALLER than benign.
# =====================================================================
print("=" * 66)
print("E-A  BELOW-CHANCE DIAGNOSTIC")
print("=" * 66)

KERNEL = ['constantPositionOffset', 'timeDelayAttack']
proba  = rf.predict_proba(X(te))[:, 1]
ben_m  = (te.attacker == 0).values

for a in KERNEL + ['dataReplay', 'positionMirroring', 'randomPositionOffset']:
    m = ((te.attack_type == a) & (te.attacker == 1)).values
    if not m.sum():
        continue
    y   = np.r_[np.zeros(ben_m.sum()), np.ones(m.sum())]
    auc = roc_auc_score(y, np.r_[proba[ben_m], proba[m]])
    print(f"\n--- {a}   AUC {auc:.4f} ---")

    # 1. detector score: are attackers scored LOWER than benign?
    ps, pb = proba[m], proba[ben_m]
    u = stats.mannwhitneyu(ps, pb, alternative='less')
    print(f"  score      attack median {np.median(ps):.4f}  "
          f"benign {np.median(pb):.4f}   "
          f"P(attack<benign) {u.statistic/(len(ps)*len(pb)):.4f}  p={u.pvalue:.2e}")

    # 2. residuals: are attack residuals SMALLER (less noisy)?
    for c in RESID:
        av = te.loc[m, c].abs().dropna()
        bv = bt[c].abs().dropna()
        if len(av) < 50:
            continue
        uu = stats.mannwhitneyu(av, bv, alternative='less')
        pr = uu.statistic / (len(av) * len(bv))
        flag = "  <-- SMALLER" if pr > 0.55 else ""
        print(f"  {c:11s} attack med {av.median():9.5f}  "
              f"benign {bv.median():9.5f}   P(a<b) {pr:.3f}{flag}")

print("""
READ:
  If P(attack<benign) on the score is > 0.5, attackers are ranked as
  MORE benign than benign traffic -- that IS the sub-0.5 AUC, restated.
  If residuals are also stochastically smaller, the mechanism is that
  the attack inherits ground-truth smoothness instead of sensor noise.
  Two sentences in Section 5.1 turn W6 from a hole into a finding.
  If residuals are NOT smaller, report the sub-0.5 AUC as open and say
  so plainly -- do not leave it unmentioned.
""")


# =====================================================================
# E-B — DOES POPULATING THE HEADING MARGINAL RESTORE DETECTION?
#
#  Train angles and test angles are DISJOINT, so this is not memorisation.
# =====================================================================
print("=" * 66)
print("E-B  HEADING-MARGINAL TRAINING")
print("=" * 66)

TRAIN_ANG = [30, 60, 120, 150, 210, 240, 300, 330]
TEST_ANG  = [45, 90, 135, 180, 225, 270, 315]

def contiguous(g, n):
    g = g.sort_values('sendTime_s')
    if len(g) < n:
        return None
    d = g.sendTime_s.diff().iloc[1:n]
    return g.head(n) if d.between(0.8, 1.2).all() else None

# benign trajectories from the TRAINING split -> rotated -> labelled attack
btr_pool = tr[(tr.attacker == 0) & (tr.attack_type == 'constantPositionOffset')]
tr_trajs = [t for t in (contiguous(g, 12) for _, g in btr_pool.groupby('sender_id'))
            if t is not None][:300]
print(f"training trajectories for rotation injection: {len(tr_trajs)}")

inj = []
for th in TRAIN_ANG:
    G = featurise([rotate(t0, float(th)) for t0 in tr_trajs], f'rotinj{th:03d}')
    G = G.copy(); G['attacker'] = 1
    inj.append(G)
inj = pd.concat(inj, ignore_index=True)
inj = inj[inj.groupby('sender_id').cumcount() > 0]     # drop first message
print(f"injected attack messages: {len(inj):,}")

ok      = tr[RESID].notna().all(axis=1)
tr_aug  = pd.concat([tr[ok], inj[X(inj).notna().all(axis=1)]], ignore_index=True)

rf_aug = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)
rf_aug.fit(X(tr_aug), tr_aug.attacker)

pred_aug = rf_aug.predict(X(te))
print(f"\naugmented model: precision {precision_score(te.attacker, pred_aug):.3f}   "
      f"benign flag rate {pred_aug[ben_m].mean():.4f}")

print("\nunchanged benchmark attacks (must not degrade):")
for a in KERNEL + ['randomPositionOffset', 'dataReplay']:
    m = ((te.attack_type == a) & (te.attacker == 1)).values
    y = np.r_[np.zeros(ben_m.sum()), np.ones(m.sum())]
    pa = rf_aug.predict_proba(X(te))[:, 1]
    print(f"  {a:26s} AUC {roc_auc_score(y, np.r_[pa[ben_m], pa[m]]):.3f}  "
          f"(was {res['RandomForest'][a]:.3f} recall -> now "
          f"{pred_aug[m].mean():.3f})")

# held-out angles: does rotation detection return?
te_trajs = [t for t in (contiguous(g, 12) for _, g in ben_pool.groupby('sender_id'))
            if t is not None][:200]
print(f"\nHELD-OUT angles ({len(te_trajs)} test trajectories):")
print(f"{'theta':>6s} {'baseline RF':>12s} {'augmented RF':>13s}")
base_ctl = aug_ctl = None
for th in [0] + TEST_ANG:
    G    = featurise([rotate(t0, float(th)) for t0 in te_trajs], f'ho{th:03d}')
    core = (G.groupby('sender_id').cumcount() > 0).values
    r0   = rf.predict(X(G))[core].mean()
    r1   = rf_aug.predict(X(G))[core].mean()
    if th == 0:
        base_ctl, aug_ctl = r0, r1
    tag = "  <- control" if th == 0 else ""
    print(f"{th:6d} {r0:12.4f} {r1:13.4f}{tag}")

print(f"""
READ:
  Baseline control (theta=0) {base_ctl:.4f}, augmented control {aug_ctl:.4f}.
  If the augmented column rises WELL ABOVE its own control at held-out
  angles while the benchmark attacks are unchanged, the heading marginal
  was learnable and merely unpopulated. That is a demonstrated mechanism,
  not a post-hoc story, and it is what W3 asks for.
  If it does NOT rise, the repair is wrong and Section 5.2 must say the
  two-part criterion was falsified and is unexplained.
""")


# =====================================================================
# E-C — SCALE THE ROTATION SWEEP + BOOTSTRAP CIs
# =====================================================================
print("=" * 66)
print("E-C  SCALED SWEEP")
print("=" * 66)

big = [t for t in (contiguous(g, 12) for _, g in ben_pool.groupby('sender_id'))
       if t is not None]
print(f"trajectories: {len(big)} (was 40)")

def boot_ci(x, n=2000, seed=SEED):
    r = np.random.default_rng(seed)
    bs = [r.choice(x, len(x), replace=True).mean() for _ in range(n)]
    return np.percentile(bs, [2.5, 97.5])

rows = []
for th in np.arange(0, 360, 15):
    G    = featurise([rotate(t0, float(th)) for t0 in big], f'big{th:03d}')
    core = (G.groupby('sender_id').cumcount() > 0).values
    p    = rf.predict(X(G))[core]
    lo, hi = boot_ci(p)
    rows.append({'theta': th, 'n': int(core.sum()), 'recall': p.mean(),
                 'ci_lo': lo, 'ci_hi': hi})
S5 = pd.DataFrame(rows)

G0    = featurise(big, 'bigctl')
core0 = (G0.groupby('sender_id').cumcount() > 0).values
p0    = rf.predict(X(G0))[core0]
c_lo, c_hi = boot_ci(p0)

print(S5.round(4).to_string(index=False))
print(f"\ncontrol (theta=0): {p0.mean():.4f}  95% CI [{c_lo:.4f}, {c_hi:.4f}]  "
      f"n={core0.sum():,}")
print(f"sweep recall range: {S5.recall.min():.4f} - {S5.recall.max():.4f}")
over = S5[S5.ci_lo > c_hi]
print(f"angles whose CI lies entirely ABOVE the control CI: {len(over)} of 24")
S5.to_csv('out/rotation_sweep_scaled.csv', index=False)
print("""
READ:
  If zero angles clear the control CI, you can write "recall is
  statistically indistinguishable from the unmodified-traffic flag rate
  at every angle (n per angle in the thousands)" and W5 is closed.
""")


# =====================================================================
# E-D — SECOND SCENARIO
#   Set SCEN2 to another subset that exists on the Zenodo record.
#   Check the record listing first; density 1 and 3 are common.
# =====================================================================
print("=" * 66)
print("E-D  SECOND SCENARIO  (edit SCEN2, then run)")
print("=" * 66)

SCEN2 = "InTAS_highway_1"      # <-- EDIT ME

RUN_E_D = False                # <-- set True once SCEN2 is confirmed
if RUN_E_D:
    import os, zipfile
    for a in ATTACKS:
        p = f"data/{SCEN2}_{a}.zip"
        if not (os.path.exists(p) and zipfile.is_zipfile(p)):
            if os.path.exists(p): os.remove(p)
            get_ipython().system(
                f'wget -q --tries=3 "{BASE}/{SCEN2}_{a}.zip?download=1" -O "{p}"')
        assert zipfile.is_zipfile(p), f"download failed: {a}"

    _S = SCENARIO
    globals()['SCENARIO'] = SCEN2
    tr2 = build_features(load_split("Train"))
    te2 = build_features(load_split("Test"))
    globals()['SCENARIO'] = _S

    ok2  = tr2[RESID].notna().all(axis=1)
    rf2  = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)
    rf2.fit(X(tr2[ok2]), tr2[ok2].attacker)
    p2   = rf2.predict_proba(X(te2))[:, 1]
    bm2  = (te2.attacker == 0).values
    print(f"{SCEN2}: precision "
          f"{precision_score(te2.attacker, rf2.predict(X(te2))):.3f}   "
          f"benign flag {rf2.predict(X(te2))[bm2].mean():.4f}")
    for a in KERNEL + ['randomPositionOffset', 'dataReplay']:
        m = ((te2.attack_type == a) & (te2.attacker == 1)).values
        if not m.sum(): continue
        y = np.r_[np.zeros(bm2.sum()), np.ones(m.sum())]
        print(f"  {a:26s} AUC {roc_auc_score(y, np.r_[p2[bm2], p2[m]]):.3f}")
    print("\nOne sentence in the paper is enough. Do NOT rebuild the analysis.")
else:
    print("RUN_E_D is False. Confirm the scenario name on the Zenodo record,")
    print("set SCEN2, set RUN_E_D=True, rerun. ~15 min.")

In [ ]:
# =====================================================================
#  E-B (v2) — CORRECTED heading-marginal experiment
#
#  Why v1 failed: it injected 24,464 rotated-benign messages labelled as
#  attacks. Rotated benign traffic is identical to benign traffic on
#  every feature except heading, so the training set gained ~24k
#  near-duplicate vectors with opposite labels, against ~4k for each
#  real attack. The model degraded globally: benign flag rate 0.046 ->
#  0.632, precision 0.828 -> 0.286. Nothing about the world was learned.
#
#  v2 does two things in order:
#    B1  Ask whether rotation even MOVES the heading off the benign
#        marginal. If it does not, the two-part criterion's second
#        condition was never triggered and "never learned" is the wrong
#        explanation.
#    B2  Only if B1 says yes: inject a SIZE-MATCHED set (~4k, matching a
#        real attack) drawn from angles that actually produce
#        out-of-support headings, and require the benign flag rate to
#        stay near 0.046 or the run is void.
# =====================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score

ben_m = (te.attacker == 0).values
KERNEL = ['constantPositionOffset', 'timeDelayAttack']

def contiguous(g, n):
    g = g.sort_values('sendTime_s')
    if len(g) < n:
        return None
    d = g.sendTime_s.diff().iloc[1:n]
    return g.head(n) if d.between(0.8, 1.2).all() else None


# =====================================================================
# B1 — does rotation move the heading off the benign marginal?
# =====================================================================
print("=" * 66)
print("B1  HEADING SUPPORT DIAGNOSTIC")
print("=" * 66)

NB = 72                                    # 5-degree bins
hb, edges = np.histogram(b.s_hed.dropna(), bins=NB, range=(0, 360))
dens = hb / hb.sum()
occupied = dens > 0                        # any benign mass at all
RARE = np.percentile(dens[occupied], 10)   # bottom decile of occupied bins

print(f"benign heading bins with ANY mass : {occupied.sum()} of {NB}")
print(f"bins holding 90% of benign mass    : "
      f"{(np.sort(dens)[::-1].cumsum() < 0.90).sum() + 1} of {NB}")
print(f"rare-bin threshold (10th pct)      : {RARE:.5f}\n")

def bin_of(v):
    return np.clip(((v % 360) / (360 / NB)).astype(int), 0, NB - 1)

te_trajs = [t for t in (contiguous(g, 12) for _, g in ben_pool.groupby('sender_id'))
            if t is not None]
print(f"test trajectories: {len(te_trajs)}\n")

print(f"{'theta':>6s} {'frac unoccupied':>16s} {'frac rare':>11s} {'median dens':>12s}")
supp = []
for th in np.arange(0, 360, 15):
    G = featurise([rotate(t0, float(th)) for t0 in te_trajs], f'b1_{th:03d}')
    core = (G.groupby('sender_id').cumcount() > 0).values
    ix = bin_of(G.s_hed.values[core])
    d  = dens[ix]
    supp.append({'theta': th,
                 'frac_unoccupied': float((d == 0).mean()),
                 'frac_rare':       float((d <= RARE).mean()),
                 'median_density':  float(np.median(d))})
    print(f"{th:6d} {supp[-1]['frac_unoccupied']:16.3f} "
          f"{supp[-1]['frac_rare']:11.3f} {supp[-1]['median_density']:12.5f}")

SUP = pd.DataFrame(supp)
SUP.to_csv('out/heading_support.csv', index=False)

fig, ax = plt.subplots(figsize=(3.33, 2.2))
ax.bar(edges[:-1], dens, width=5, color='#227722', alpha=0.75)
ax.set_xlabel('reported heading (deg)', fontsize=8)
ax.set_ylabel('benign density', fontsize=8)
ax.set_xticks(np.arange(0, 361, 90)); ax.tick_params(labelsize=7)
plt.tight_layout(pad=0.3)
plt.savefig('out/fig5_heading_marginal.pdf', bbox_inches='tight'); plt.show()

GOOD = SUP[SUP.frac_rare > 0.5].theta.tolist()
print(f"""
READ B1:
  angles where >50% of rotated headings land in rare/unoccupied bins:
    {GOOD}

  IF THIS LIST IS EMPTY OR SHORT
    The benign heading marginal is broad enough that rotation rarely
    produces an anomalous heading. The two-part criterion's second
    condition is never triggered, so rotation is invisible because the
    marginal is NOT violated -- not because it was 'never learned'.
    That is a cleaner explanation than the one in the current Section 5.2
    and it needs no further experiment. Rewrite 5.2 and stop here.

  IF THE LIST IS LONG
    Rotation does produce anomalous headings and the detector still
    misses them. Run B2 on exactly these angles.
""")


# =====================================================================
# B2 — size-matched injection.  Run ONLY if GOOD is long.
# =====================================================================
RUN_B2 = len(GOOD) >= 6
print("=" * 66)
print(f"B2  SIZE-MATCHED INJECTION   (RUN_B2 = {RUN_B2})")
print("=" * 66)

if RUN_B2:
    TARGET_N  = 4000                       # match a real attack's size
    TRAIN_ANG = GOOD[0::2]                 # alternate angles
    TEST_ANG  = GOOD[1::2]                 # held out, disjoint
    print(f"train angles {TRAIN_ANG}\ntest angles  {TEST_ANG}\n")

    btr_pool = tr[(tr.attacker == 0) & (tr.attack_type == 'constantPositionOffset')]
    tr_trajs = [t for t in (contiguous(g, 12) for _, g in btr_pool.groupby('sender_id'))
                if t is not None]

    inj = []
    for th in TRAIN_ANG:
        G = featurise([rotate(t0, float(th)) for t0 in tr_trajs], f'inj{th:03d}')
        G = G[G.groupby('sender_id').cumcount() > 0].copy()
        G['attacker'] = 1
        inj.append(G)
    inj = pd.concat(inj, ignore_index=True)
    if len(inj) > TARGET_N:
        inj = inj.sample(TARGET_N, random_state=SEED)
    print(f"injected messages: {len(inj):,} "
          f"(real attacks are ~3,000-4,000 each)\n")

    ok     = tr[RESID].notna().all(axis=1)
    tr_aug = pd.concat([tr[ok], inj], ignore_index=True)
    rf_aug = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)
    rf_aug.fit(X(tr_aug), tr_aug.attacker)

    pa   = rf_aug.predict(X(te))
    prob = rf_aug.predict_proba(X(te))[:, 1]
    fp   = pa[ben_m].mean()
    pr   = precision_score(te.attacker, pa)
    print(f"augmented model: precision {pr:.3f} (was 0.828)   "
          f"benign flag {fp:.4f} (was 0.046)")

    VALID = (fp < 0.10) and (pr > 0.70)
    if not VALID:
        print("\n*** RUN VOID: the model degraded globally. The injected class")
        print("*** is still not separable from benign traffic. Do not report")
        print("*** these numbers; report B1's finding instead.\n")
    else:
        print("\nbenchmark attacks (must not degrade):")
        for a in KERNEL + ['randomPositionOffset', 'dataReplay']:
            m = ((te.attack_type == a) & (te.attacker == 1)).values
            y = np.r_[np.zeros(ben_m.sum()), np.ones(m.sum())]
            print(f"  {a:26s} AUC {roc_auc_score(y, np.r_[prob[ben_m], prob[m]]):.3f}")

        G0 = featurise(te_trajs, 'ctl2')
        c0 = (G0.groupby('sender_id').cumcount() > 0).values
        ctl_b, ctl_a = rf.predict(X(G0))[c0].mean(), rf_aug.predict(X(G0))[c0].mean()
        print(f"\ncontrol (theta=0): baseline {ctl_b:.4f}   augmented {ctl_a:.4f}")
        print(f"\n{'theta':>6s} {'baseline':>10s} {'augmented':>11s} {'lift over ctl':>14s}")
        for th in TEST_ANG:
            G = featurise([rotate(t0, float(th)) for t0 in te_trajs], f'ho2_{th:03d}')
            c = (G.groupby('sender_id').cumcount() > 0).values
            r0, r1 = rf.predict(X(G))[c].mean(), rf_aug.predict(X(G))[c].mean()
            print(f"{th:6d} {r0:10.4f} {r1:11.4f} {r1 - ctl_a:14.4f}")
        print("""
READ B2:
  The 'lift over ctl' column is what matters. A large positive lift at
  HELD-OUT angles, with benchmark attacks unchanged and the benign flag
  rate near 0.046, demonstrates that the heading marginal was learnable
  and merely unpopulated. Anything else and the honest report is that
  the two-part criterion was falsified and the reason is B1's.
""")
else:
    print("Skipped: B1 shows rotation rarely produces an anomalous heading.")
    print("B1 IS the answer. Rewrite Section 5.2 around it.")

In [ ]:
# =====================================================================
#  ROUND 3 — answering the Weak Reject.  Priority order.
#
#   R1  Receiver-position plausibility check
#   R2  Fine-grained small-angle sweep, 0-20 deg
#   R3  Bootstrap CIs + leave-one-vehicle-out
#
#  Needs from the main notebook: tr, te, b, bt, rf, X, ATTACKS, FEATS,
#  RESID, SEED, SCENARIO, load, load_split, build_features, rotate,
#  featurise, ben_pool, nn, BEN_MED
# =====================================================================
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score

KERNEL = ['constantPositionOffset', 'timeDelayAttack']
WATCH  = KERNEL + ['dataReplay', 'positionMirroring', 'randomPositionOffset']


# =====================================================================
# R1 — RECEIVER-POSITION CHECK
#
#  The receiver's own GPS fix is not an external resource. Distance from
#  the CLAIMED sender position to the receiver's TRUE position, against
#  plausible communication range, is a standard cheap check.
#
#  Your dedup keeps one arbitrary receiver per transmission, which
#  discards exactly this. So compute it BEFORE dedup, over all
#  (sender, receiver) pairs, then attach summary statistics.
# =====================================================================
print("=" * 68)
print("R1  RECEIVER-POSITION PLAUSIBILITY")
print("=" * 68)

def receiver_stats(split):
    """Per-transmission receiver-distance statistics, computed over all
    log rows (i.e. all receivers) before deduplication."""
    out = []
    for a in ATTACKS:
        raw = load(f"data/{SCENARIO}_{a}.zip", split, a)
        raw = raw.dropna(subset=['s_x', 's_y', 'r_x', 'r_y'])
        raw['sr_dist'] = np.hypot(raw.s_x - raw.r_x, raw.s_y - raw.r_y)
        g = raw.groupby(['attack_type', 'sender_id', 'sendTime_s'])
        out.append(pd.DataFrame({
            'sr_min':  g.sr_dist.min(),
            'sr_max':  g.sr_dist.max(),
            'sr_mean': g.sr_dist.mean(),
            'n_recv':  g.sr_dist.size(),
        }).reset_index())
        del raw
    return pd.concat(out, ignore_index=True)

print("computing receiver statistics (a few minutes)...")
rs_tr = receiver_stats("Train")
rs_te = receiver_stats("Test")

trR = tr.merge(rs_tr, on=['attack_type', 'sender_id', 'sendTime_s'], how='left')
teR = te.merge(rs_te, on=['attack_type', 'sender_id', 'sendTime_s'], how='left')
RECV = ['sr_min', 'sr_max', 'sr_mean', 'n_recv']
print(f"merged: train {len(trR):,}  test {len(teR):,}   "
      f"missing {trR[RECV].isna().any(axis=1).mean():.3%}\n")

# ---- the selection-effect argument, measured ----
bR = teR[teR.attacker == 0]
print("Is the attack pushed outside the range legitimate traffic shows?")
print(f"{'':28s} {'sr_max med':>11s} {'sr_max p99':>11s} {'>benign p99':>12s}")
p99 = bR.sr_max.quantile(0.99)
print(f"{'BENIGN':28s} {bR.sr_max.median():11.1f} {p99:11.1f} {'--':>12s}")
for a in WATCH:
    d = teR[(teR.attack_type == a) & (teR.attacker == 1)]
    if not len(d): continue
    print(f"{a:28s} {d.sr_max.median():11.1f} "
          f"{d.sr_max.quantile(0.99):11.1f} {(d.sr_max > p99).mean():12.3f}")

print(f"""
>>> A message is only logged by vehicles that RECEIVED it, so every
>>> logged transmission is already inside communication range. If the
>>> '>benign p99' column is near zero for the kernel attacks, a bounded
>>> offset cannot push the claimed position outside the range of
>>> distances legitimate traffic exhibits. That is a one-sided
>>> constraint and it is the honest answer to the objection.
""")

# ---- a pure plausibility rule, no learning ----
print("Threshold rule: flag if sr_max exceeds a benign quantile.")
print(f"{'quantile':>9s} {'threshold':>10s} {'benign FP':>10s} " +
      "".join(f"{a[:14]:>15s}" for a in KERNEL))
for q in (0.95, 0.99, 0.999):
    thr = bR.sr_max.quantile(q)
    row = f"{q:9.3f} {thr:10.1f} {(bR.sr_max > thr).mean():10.4f}"
    for a in KERNEL:
        d = teR[(teR.attack_type == a) & (teR.attacker == 1)]
        row += f"{(d.sr_max > thr).mean():15.4f}"
    print(row)

# ---- learned detector with receiver features ----
FEATS_R = FEATS + RECV
XR = lambda d: d[FEATS_R].replace([np.inf, -np.inf], np.nan).fillna(0)

ok  = trR[RESID].notna().all(axis=1) & trR[RECV].notna().all(axis=1)
rfR = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)
rfR.fit(XR(trR[ok]), trR[ok].attacker)

pR   = rfR.predict_proba(XR(teR))[:, 1]
pdR  = rfR.predict(XR(teR))
benm = (teR.attacker == 0).values
print(f"\n+receiver model: precision {precision_score(teR.attacker, pdR):.3f} "
      f"(sender-only 0.828)   benign flag {pdR[benm].mean():.4f} (0.046)")

p0    = rf.predict_proba(X(te))[:, 1]
ben0  = p0[(te.attacker == 0).values]
benR  = pR[benm]
print(f"\n{'attack':28s} {'AUC sender':>11s} {'AUC +recv':>10s} {'delta':>8s}")
for a in ATTACKS:
    m0 = ((te.attack_type == a)  & (te.attacker == 1)).values
    mR = ((teR.attack_type == a) & (teR.attacker == 1)).values
    if not m0.sum(): continue
    y0 = np.r_[np.zeros(len(ben0)), np.ones(m0.sum())]
    yR = np.r_[np.zeros(len(benR)), np.ones(mR.sum())]
    a0 = roc_auc_score(y0, np.r_[ben0, p0[m0]])
    aR = roc_auc_score(yR, np.r_[benR, pR[mR]])
    star = "  *" if a in WATCH else ""
    print(f"{a:28s} {a0:11.3f} {aR:10.3f} {aR - a0:+8.3f}{star}")

print("\nreceiver-feature importances:")
print(pd.Series(rfR.feature_importances_, index=FEATS_R)
      .sort_values(ascending=False).head(6).round(4).to_string())

pd.DataFrame({'sender': [roc_auc_score(
                  np.r_[np.zeros(len(ben0)), np.ones(((te.attack_type==a)&(te.attacker==1)).sum())],
                  np.r_[ben0, p0[((te.attack_type==a)&(te.attacker==1)).values]])
                  for a in ATTACKS if ((te.attack_type==a)&(te.attacker==1)).sum()]},
             index=[a for a in ATTACKS if ((te.attack_type==a)&(te.attacker==1)).sum()]
            ).to_csv('out/receiver_comparison.csv')

print("""
READ R1:
  If the kernel attacks stay near AUC 0.50 with receiver features added,
  you can say so directly and the objection is answered with data.
  If they rise substantially, SAY THAT TOO -- it narrows the claim to
  detectors that do not use receiver context, which is still a real
  claim about the ML-MDS literature you survey in Table 2, and it is

""")


# =====================================================================
# R2 — FINE-GRAINED SMALL-ANGLE SWEEP
#   The 15-degree grid cannot resolve where road-plausibility breaks.
# =====================================================================
print("=" * 68)
print("R2  SMALL-ANGLE SWEEP, 0-20 DEGREES IN 1-DEGREE STEPS")
print("=" * 68)

def contiguous(g, n):
    g = g.sort_values('sendTime_s')
    if len(g) < n: return None
    d = g.sendTime_s.diff().iloc[1:n]
    return g.head(n) if d.between(0.8, 1.2).all() else None

trajs = [t for t in (contiguous(g, 12) for _, g in ben_pool.groupby('sender_id'))
         if t is not None]
print(f"trajectories: {len(trajs)}   benign NN baseline: {BEN_MED:.2f} m\n")

rows = []
for th in range(0, 21):
    rot = [rotate(t0, float(th)) for t0 in trajs]
    G   = featurise(rot, f'fine{th:03d}')
    core = (G.groupby('sender_id').cumcount() > 0).values
    d   = nn.kneighbors(G[['s_x', 's_y']].values)[0].ravel()
    disp = max(np.hypot(r.s_x.values - t.s_x.values,
                        r.s_y.values - t.s_y.values).max()
               for r, t in zip(rot, trajs))
    rows.append({'theta': th,
                 'recall':   float(rf.predict(X(G))[core].mean()),
                 'nn_med':   float(np.median(d)),
                 'nn_p90':   float(np.percentile(d, 90)),
                 'max_disp': float(disp)})
F = pd.DataFrame(rows)
print(F.round(3).to_string(index=False))
F.to_csv('out/fine_sweep.csv', index=False)

crossed = F[F.nn_med > 2.57]          # benign 90th percentile
print(f"\nbenign 90th percentile = 2.57 m")
if len(crossed):
    k = crossed.iloc[0]
    print(f"road-plausibility breaks at theta = {int(k.theta)} deg "
          f"(NN median {k.nn_med:.2f} m, displacement {k.max_disp:.0f} m)")
    print(f"largest displacement while still road-plausible: "
          f"{F[F.nn_med <= 2.57].max_disp.max():.0f} m")
else:
    print("still road-plausible at 20 deg; extend the sweep")

print("""
READ R2:
  'largest displacement while still road-plausible' is the number that
  belongs in your ABSTRACT. It is an attack that is invisible to
  content-based detection AND survives a map check, which is the only
  operationally live element of the group. Lead with it.
""")


# =====================================================================
# R3 — BOOTSTRAP CIs AND LEAVE-ONE-VEHICLE-OUT
# =====================================================================
print("=" * 68)
print("R3  UNCERTAINTY ON THE HEADLINE AUCs")
print("=" * 68)

rng = np.random.default_rng(SEED)
print(f"{'attack':28s} {'AUC':>7s} {'95% CI':>18s} {'LOVO min':>9s} {'LOVO max':>9s}")
for a in WATCH:
    m = ((te.attack_type == a) & (te.attacker == 1)).values
    if not m.sum(): continue
    pa, pb = p0[m], ben0
    y = np.r_[np.zeros(len(pb)), np.ones(len(pa))]
    auc = roc_auc_score(y, np.r_[pb, pa])

    bs = []
    for _ in range(1000):
        ia = rng.integers(0, len(pa), len(pa))
        ib = rng.integers(0, len(pb), min(len(pb), 20000))
        yy = np.r_[np.zeros(len(ib)), np.ones(len(ia))]
        bs.append(roc_auc_score(yy, np.r_[pb[ib], pa[ia]]))
    lo, hi = np.percentile(bs, [2.5, 97.5])

    sub = te[m]
    lov = []
    for v in sub.sender_id.unique():
        k = (sub.sender_id != v).values
        if k.sum() < 50: continue
        yy = np.r_[np.zeros(len(pb)), np.ones(k.sum())]
        lov.append(roc_auc_score(yy, np.r_[pb, pa[k]]))
    print(f"{a:28s} {auc:7.3f} [{lo:7.3f},{hi:7.3f}] "
          f"{min(lov):9.3f} {max(lov):9.3f}")

print("""
READ R3:
  If the CI for constantPositionOffset spans 0.500, say so: the value is
  chance, full stop. If it excludes 0.500 on the low side, the tilt is
  real and you must either explain it or flag it prominently -- after
  four preprocessing defects, 'unexplained' is a weak place to leave it.
  Leave-one-vehicle-out spread shows whether one vehicle drives the tilt.
""")

In [ ]:
# =====================================================================
#  CELL 0 — SESSION RESTORE.  Run this first after any runtime reset.
#  Rebuilds constants, loaders, features and the Random Forest.
#  ~12 minutes (download ~5, features ~4, fit ~3).
# =====================================================================
import os, io, json, zipfile, gc, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score
warnings.filterwarnings('ignore')

SEED, DT, GAP_MAX = 42, 1.0, 2.0
SCENARIO = "InTAS_highway_2"
BASE     = "https://zenodo.org/records/19665762/files"

ATTACKS = '''accelerationMultiplication constantPositionOffset constantSpeedOffset
dataReplay dosAttack feignedBraking positionMirroring randomPositionOffset
randomSpeedOffset reversedHeading suddenConstantSpeed suddenStop
timeDelayAttack trafficCongestionSybil zeroSpeedReport'''.split()

FEATS = ["s_spd","s_acl","s_hed","disp","dt","implied_spd",
         "spd_resid","hed_resid","acl_resid","latency"]
RESID = ["spd_resid","hed_resid","acl_resid"]

os.makedirs("data", exist_ok=True); os.makedirs("out", exist_ok=True)
for a in ATTACKS:
    p = f"data/{SCENARIO}_{a}.zip"
    if not (os.path.exists(p) and zipfile.is_zipfile(p)):
        if os.path.exists(p): os.remove(p)
        !wget -q --tries=3 "{BASE}/{SCENARIO}_{a}.zip?download=1" -O "{p}"
    assert zipfile.is_zipfile(p), f"download failed: {a}"
print(f"{len(ATTACKS)} subsets verified")


def num(v, idx=None):
    if v is None: return None
    if idx is not None:
        try: return float(str(v).split(",")[idx])
        except (ValueError, IndexError): return None
    try: return float(v)
    except (ValueError, TypeError): return None

def load(zip_path, split, label):
    outer = zipfile.ZipFile(zip_path)
    inner = zipfile.ZipFile(io.BytesIO(outer.read(
        [n for n in outer.namelist() if f"/{split}/" in n and n.endswith(".zip")][0])))
    rows = []
    for name in inner.namelist():
        if not name.endswith(".json"): continue
        for m in json.loads(inner.read(name)):
            s, r = m.get("sender", {}), m.get("receiver", {})
            rows.append({
                "rcvTime_s":  (num(m.get("rcvTime"))  or 0) / 1e9,
                "sendTime_s": (num(m.get("sendTime")) or 0) / 1e9,
                "sender_id":  m.get("sender_id"),
                "attacker":   int(m.get("attacker", 0)),
                "attack_type": label,
                "s_x": num(s.get("pos"), 0), "s_y": num(s.get("pos"), 1),
                "s_spd": num(s.get("spd")),  "s_acl": num(s.get("acl")),
                "s_hed": num(s.get("hed")),
                "r_x": num(r.get("pos"), 0), "r_y": num(r.get("pos"), 1),
            })
    return pd.DataFrame(rows)

def load_split(split):
    return pd.concat([load(f"data/{SCENARIO}_{a}.zip", split, a) for a in ATTACKS],
                     ignore_index=True)

def build_features(df, group_cols=("attack_type", "sender_id")):
    g_cols = list(group_cols)
    d = (df.sort_values(g_cols + ["sendTime_s"])
           .drop_duplicates(g_cols + ["sendTime_s"]).copy())
    g  = d.groupby(g_cols)
    dt = g.sendTime_s.diff()
    dx, dy = g.s_x.diff(), g.s_y.diff()
    d["dt"], d["disp"] = dt, np.hypot(dx, dy)
    d["implied_spd"]   = d.disp / dt.replace(0, np.nan)
    d["spd_resid"]     = d.implied_spd - d.s_spd
    move_hed = (90 - np.degrees(np.arctan2(dy, dx))) % 360
    hd = (move_hed - d.s_hed).abs() % 360
    d["hed_resid"] = np.minimum(hd, 360 - hd)
    d["acl_resid"] = g.s_spd.diff() / dt.replace(0, np.nan) - d.s_acl
    d["latency"]   = d.rcvTime_s - d.sendTime_s
    d.loc[d.dt > GAP_MAX, RESID] = np.nan      # KEEP AS NaN
    return d

X = lambda d, fill=0.0: d[FEATS].replace([np.inf, -np.inf], np.nan).fillna(fill)

tr = build_features(load_split("Train"))
te = build_features(load_split("Test"))
gc.collect()

b, bt = tr[tr.attacker == 0], te[te.attacker == 0]
assert abs(b.dt.median() - 1.0) < 0.02,                      "dedup failed"
assert abs(b.spd_resid.abs().median() - 0.098) < 0.02,       "residuals wrong"
assert abs(b.hed_resid.abs().median() - 2.9) < 0.4,          "heading convention wrong"
assert tr.spd_resid.isna().sum() > 0,                        "NaNs lost"

ok = tr[RESID].notna().all(axis=1)
rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)
rf.fit(X(tr[ok]), tr[ok].attacker)
pred = rf.predict(X(te))
print(f"\nrestored. precision {precision_score(te.attacker, pred):.3f}   "
      f"benign flag {pred[(te.attacker==0).values].mean():.4f}")
print("expect 0.828 and 0.046")

tr.to_parquet("out/trF.parquet"); te.to_parquet("out/teF.parquet")
print("saved out/trF.parquet, out/teF.parquet")


# =====================================================================
#  R1 — RECEIVER-POSITION PLAUSIBILITY
#
#  The receiver's own GPS fix is not an external resource. Distance from
#  the CLAIMED sender position to the receiver's TRUE position, against
#  plausible communication range, is a standard cheap check.
#
#  Dedup keeps one arbitrary receiver per transmission, discarding
#  exactly this, so compute it over ALL log rows before dedup.
# =====================================================================
print("\n" + "=" * 68)
print("R1  RECEIVER-POSITION PLAUSIBILITY")
print("=" * 68)

KERNEL = ['constantPositionOffset', 'timeDelayAttack']
WATCH  = KERNEL + ['dataReplay', 'positionMirroring', 'randomPositionOffset']

def receiver_stats(split):
    out = []
    for a in ATTACKS:
        raw = load(f"data/{SCENARIO}_{a}.zip", split, a)
        raw = raw.dropna(subset=['s_x', 's_y', 'r_x', 'r_y'])
        raw['sr_dist'] = np.hypot(raw.s_x - raw.r_x, raw.s_y - raw.r_y)
        g = raw.groupby(['attack_type', 'sender_id', 'sendTime_s'])
        out.append(pd.DataFrame({'sr_min':  g.sr_dist.min(),
                                 'sr_max':  g.sr_dist.max(),
                                 'sr_mean': g.sr_dist.mean(),
                                 'n_recv':  g.sr_dist.size()}).reset_index())
        del raw; gc.collect()
    return pd.concat(out, ignore_index=True)

print("computing receiver statistics (~4 min)...")
rs_tr, rs_te = receiver_stats("Train"), receiver_stats("Test")

KEY  = ['attack_type', 'sender_id', 'sendTime_s']
RECV = ['sr_min', 'sr_max', 'sr_mean', 'n_recv']
trR = tr.merge(rs_tr, on=KEY, how='left')
teR = te.merge(rs_te, on=KEY, how='left')
print(f"merged: train {len(trR):,}  test {len(teR):,}   "
      f"missing {trR[RECV].isna().any(axis=1).mean():.3%}\n")

# ---- the selection-effect argument, measured ----
bR  = teR[teR.attacker == 0]
p99 = bR.sr_max.quantile(0.99)
print("Does the attack push the claimed position outside the range")
print("legitimate traffic exhibits?")
print(f"{'':28s} {'sr_max med':>11s} {'sr_max p99':>11s} {'>benign p99':>12s}")
print(f"{'BENIGN':28s} {bR.sr_max.median():11.1f} {p99:11.1f} {'--':>12s}")
for a in WATCH:
    d = teR[(teR.attack_type == a) & (teR.attacker == 1)]
    if not len(d): continue
    print(f"{a:28s} {d.sr_max.median():11.1f} "
          f"{d.sr_max.quantile(0.99):11.1f} {(d.sr_max > p99).mean():12.3f}")

# ---- a pure plausibility rule, no learning ----
print("\nThreshold rule: flag if sr_max exceeds a benign quantile.")
print(f"{'quantile':>9s} {'thresh':>9s} {'benign FP':>10s}" +
      "".join(f"{a[:13]:>14s}" for a in KERNEL))
for q in (0.95, 0.99, 0.999):
    thr = bR.sr_max.quantile(q)
    row = f"{q:9.3f} {thr:9.1f} {(bR.sr_max > thr).mean():10.4f}"
    for a in KERNEL:
        d = teR[(teR.attack_type == a) & (teR.attacker == 1)]
        row += f"{(d.sr_max > thr).mean():14.4f}"
    print(row)

# ---- learned detector with receiver features ----
FEATS_R = FEATS + RECV
XR = lambda d: d[FEATS_R].replace([np.inf, -np.inf], np.nan).fillna(0)
okR = trR[RESID].notna().all(axis=1) & trR[RECV].notna().all(axis=1)
rfR = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)
rfR.fit(XR(trR[okR]), trR[okR].attacker)

pR, pdR = rfR.predict_proba(XR(teR))[:, 1], rfR.predict(XR(teR))
benm = (teR.attacker == 0).values
print(f"\n+receiver: precision {precision_score(teR.attacker, pdR):.3f} "
      f"(sender-only 0.828)   benign flag {pdR[benm].mean():.4f} (0.046)")

p0   = rf.predict_proba(X(te))[:, 1]
ben0 = p0[(te.attacker == 0).values]
benR = pR[benm]
print(f"\n{'attack':28s} {'AUC sender':>11s} {'AUC +recv':>10s} {'delta':>8s}")
res_cmp = {}
for a in ATTACKS:
    m0 = ((te.attack_type == a)  & (te.attacker == 1)).values
    mR = ((teR.attack_type == a) & (teR.attacker == 1)).values
    if not m0.sum(): continue
    a0 = roc_auc_score(np.r_[np.zeros(len(ben0)), np.ones(m0.sum())],
                       np.r_[ben0, p0[m0]])
    aR = roc_auc_score(np.r_[np.zeros(len(benR)), np.ones(mR.sum())],
                       np.r_[benR, pR[mR]])
    res_cmp[a] = (a0, aR)
    print(f"{a:28s} {a0:11.3f} {aR:10.3f} {aR-a0:+8.3f}"
          + ("  *" if a in WATCH else ""))

print("\ntop feature importances, +receiver model:")
print(pd.Series(rfR.feature_importances_, index=FEATS_R)
      .sort_values(ascending=False).head(6).round(4).to_string())

pd.DataFrame(res_cmp, index=['sender', 'plus_receiver']).T \
  .to_csv('out/receiver_comparison.csv')

print("""
READ:
  A message is only logged by vehicles that RECEIVED it, so every logged
  transmission is already inside communication range. If '>benign p99'
  is near zero for the kernel attacks, a bounded offset cannot push the
  claimed position outside the range legitimate traffic shows. That is a
  one-sided constraint and it answers the objection with data.

  If the kernel AUCs RISE substantially with receiver features, say so.
  The claim narrows to detectors that do not use receiver context, which
  is still a real claim about the literature -- and far better found by

""")

In [ ]:
# =====================================================================
#  ROUND 4 — three items, in priority order.
#
#   N1  Vehicle-level aggregation on the RECEIVER-CONTEXT model.
#       Highest risk. Your abstract says receiver context "does not
#       recover the kernel attacks either"; §4.4 only tested majority
#       vote on the BASE model. At 12.5% per-message against a 1%
#       benign FPR, a k-of-n rule with small k may separate attackers
#       cleanly over a 40-message trajectory. Find out before a
#       reviewer does.
#
#   N2  Residual ratios for all fifteen attacks -> restore the column.
#
#   S   Five RF seeds -> one sentence in Limitations.
#
#  Requires in memory (from restore_and_R1.py):
#     tr, te, trR, teR, rf, rfR, X, XR, FEATS, FEATS_R, RESID,
#     ATTACKS, bt, SEED, RECV
# =====================================================================
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score

KERNEL = ['constantPositionOffset', 'timeDelayAttack']
WATCH  = KERNEL + ['dataReplay', 'positionMirroring',
                   'randomPositionOffset', 'zeroSpeedReport']


# =====================================================================
# N1 — VEHICLE-LEVEL AGGREGATION, RECEIVER-CONTEXT MODEL
# =====================================================================
print("=" * 70)
print("N1  VEHICLE-LEVEL AGGREGATION UNDER RECEIVER CONTEXT")
print("=" * 70)

teR = teR.copy()
teR['pred_base'] = rf.predict(X(teR))
teR['pred_recv'] = rfR.predict(XR(teR))

def vehicle_rates(df, col):
    """Fraction of each vehicle's messages flagged."""
    out = {}
    ben = df[df.attacker == 0].groupby('sender_id')[col].agg(['mean', 'size'])
    out['BENIGN'] = ben
    for a in ATTACKS:
        d = df[(df.attack_type == a) & (df.attacker == 1)]
        if len(d):
            out[a] = d.groupby('sender_id')[col].agg(['mean', 'size'])
    return out

for tag, col in [('BASE (sender-only)', 'pred_base'),
                 ('RECEIVER CONTEXT',   'pred_recv')]:
    V = vehicle_rates(teR, col)
    ben = V['BENIGN']
    print(f"\n--- {tag} ---")
    print(f"benign vehicles: n={len(ben)}  mean flag rate {ben['mean'].mean():.4f}  "
          f"median msgs/vehicle {ben['size'].median():.0f}")
    print(f"{'attack':26s} {'n_veh':>6s} {'mean flag':>10s} "
          f"{'>50%':>7s} {'>20%':>7s} {'>10%':>7s} {'>5%':>7s}")
    print(f"{'BENIGN':26s} {len(ben):6d} {ben['mean'].mean():10.4f} "
          f"{(ben['mean']>0.50).mean():7.3f} {(ben['mean']>0.20).mean():7.3f} "
          f"{(ben['mean']>0.10).mean():7.3f} {(ben['mean']>0.05).mean():7.3f}")
    for a in WATCH:
        if a not in V: continue
        s = V[a]['mean']
        print(f"{a:26s} {len(s):6d} {s.mean():10.4f} "
              f"{(s>0.50).mean():7.3f} {(s>0.20).mean():7.3f} "
              f"{(s>0.10).mean():7.3f} {(s>0.05).mean():7.3f}")

# ---- the honest test: pick k to hold benign vehicle FPR at 5%, then
#      report attacker detection at that operating point ----
print("\n" + "-" * 70)
print("OPERATING POINT: threshold chosen so 5% of BENIGN vehicles are flagged")
print("-" * 70)
for tag, col in [('BASE', 'pred_base'), ('+RECV', 'pred_recv')]:
    V   = vehicle_rates(teR, col)
    ben = V['BENIGN']['mean']
    thr = ben.quantile(0.95)
    print(f"\n{tag}: threshold = {thr:.4f} of a vehicle's messages flagged")
    print(f"{'attack':26s} {'vehicles caught':>16s}")
    for a in WATCH:
        if a not in V: continue
        s = V[a]['mean']
        print(f"{a:26s} {(s > thr).mean():15.3f}  ({int((s>thr).sum())}/{len(s)})")

# ---- pure sr_max threshold rule at vehicle level (no learning) ----
if 'sr_max' in teR.columns:
    print("\n" + "-" * 70)
    print("PURE RANGE RULE AT VEHICLE LEVEL (no classifier)")
    print("-" * 70)
    bR   = teR[teR.attacker == 0]
    for q in (0.99, 0.999):
        thr_msg = bR.sr_max.quantile(q)
        teR['flag_range'] = (teR.sr_max > thr_msg).astype(int)
        V   = vehicle_rates(teR, 'flag_range')
        ben = V['BENIGN']['mean']
        vthr = ben.quantile(0.95)
        print(f"\nmessage threshold at benign q={q}: {thr_msg:.1f} m")
        print(f"vehicle threshold holding 5% benign FPR: "
              f"{vthr:.4f} of messages")
        for a in KERNEL:
            if a not in V: continue
            s = V[a]['mean']
            print(f"  {a:26s} caught {(s > vthr).mean():.3f}  "
                  f"(mean flag rate {s.mean():.4f})")

print("""
READ N1:
  Compare the '+RECV' block against 'BASE'. If attacker vehicles are
  still indistinguishable from benign vehicles at a 5% benign FPR, your
  abstract sentence stands and you should say so explicitly at vehicle
  granularity. If they ARE separated, narrow the claim: receiver context
  recovers constantPositionOffset at vehicle level but not at message
  level, and the theta=15 case survives regardless because those
  positions sit inside the lane-width spread where no range check helps.
  Report whichever it is. Silence here is the single largest risk left.
""")


# =====================================================================
# N2 — RESIDUAL RATIOS, ALL FIFTEEN ATTACKS
# =====================================================================
print("=" * 70)
print("N2  RESIDUAL RATIOS (restore the column)")
print("=" * 70)

base = {c: bt[c].abs().median() for c in RESID}
print("benign medians:", {k: round(v, 6) for k, v in base.items()}, "\n")

proba = rf.predict_proba(X(te))[:, 1]
ben_p = proba[(te.attacker == 0).values]

rows = []
for a in ATTACKS:
    m = ((te.attack_type == a) & (te.attacker == 1)).values
    if not m.sum(): continue
    d = te[m]
    per = {c: (d[c].abs().median() / base[c]) if base[c] else np.nan
           for c in RESID}
    # the acceleration residual has a near-zero benign median, so ratios
    # derived from it are unstable. Track which residual dominates.
    worst = max(per, key=lambda c: (per[c] if not np.isnan(per[c]) else -1))
    y = np.r_[np.zeros(len(ben_p)), np.ones(m.sum())]
    rows.append({'attack': a,
                 'auc':    roc_auc_score(y, np.r_[ben_p, proba[m]]),
                 'recall': float(rf.predict(X(te))[m].mean()),
                 'ratio':  per[worst],
                 'from':   worst,
                 'spd':    per['spd_resid'],
                 'hed':    per['hed_resid']})
R = pd.DataFrame(rows).sort_values('auc')
pd.set_option('display.width', 200)
print(R.round(3).to_string(index=False))
R.to_csv('out/residual_ratios.csv', index=False)

print("""
READ N2:
  Use the 'ratio' column in Table 3. Where 'from' is acl_resid the value
  is unstable (benign median ~5e-5) -- for those rows report the larger
  of spd and hed instead, or mark them with a dagger and say so in the
  caption. Do not print a five-digit ratio.
""")


# =====================================================================
# S — FIVE RF SEEDS
# =====================================================================
print("=" * 70)
print("S  FIVE-SEED STABILITY (Random Forest)")
print("=" * 70)

ok = tr[RESID].notna().all(axis=1)
aucs, precs, fps = [], [], []
for s in [42, 7, 123, 2026, 31337]:
    m = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=s)
    m.fit(X(tr[ok]), tr[ok].attacker)
    pr = m.predict_proba(X(te))[:, 1]
    pd_ = m.predict(X(te))
    bp = pr[(te.attacker == 0).values]
    row = {}
    for a in ATTACKS:
        mk = ((te.attack_type == a) & (te.attacker == 1)).values
        if not mk.sum(): continue
        row[a] = roc_auc_score(np.r_[np.zeros(len(bp)), np.ones(mk.sum())],
                               np.r_[bp, pr[mk]])
    aucs.append(row)
    precs.append(precision_score(te.attacker, pd_))
    fps.append(float(pd_[(te.attacker == 0).values].mean()))
    print(f"  seed {s:6d}  precision {precs[-1]:.4f}  "
          f"cPO {row['constantPositionOffset']:.4f}  "
          f"tDA {row['timeDelayAttack']:.4f}")

A = pd.DataFrame(aucs)
print(f"\nprecision      {np.mean(precs):.4f} +/- {np.std(precs, ddof=1):.4f}")
print(f"benign flag    {np.mean(fps):.4f} +/- {np.std(fps, ddof=1):.4f}")
print(f"\n{'attack':28s} {'mean AUC':>9s} {'sd':>8s}")
for a in WATCH:
    if a in A: print(f"{a:28s} {A[a].mean():9.4f} {A[a].std(ddof=1):8.4f}")
print(f"\nlargest AUC sd over all fifteen: {A.std(ddof=1).max():.4f} "
      f"({A.std(ddof=1).idxmax()})")
A.to_csv('out/seed_auc.csv', index=False)

print("""
PASTE-READY (Section 7):
  "Across five seeds the standard deviation of per-attack AUC never
   exceeds X; for the two kernel attacks it is Y and Z."
""")

In [ ]:
# =====================================================================
#  CELL A — BOOTSTRAP WITH DRIVE CACHE.
#
#  First run: ~16 min. Every run after a reset: ~40 seconds, because
#  everything is cached to Drive. Run this once and stop losing time.
#
#  Set USE_DRIVE = False if you would rather not mount.
# =====================================================================
import os, io, json, zipfile, gc, warnings
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score
warnings.filterwarnings('ignore')

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    CACHE = '/content/drive/MyDrive/aintec_cache'
else:
    CACHE = 'cache'
os.makedirs(CACHE, exist_ok=True)
os.makedirs('data', exist_ok=True)
os.makedirs('out', exist_ok=True)

SEED, DT, GAP_MAX = 42, 1.0, 2.0
SCENARIO = "InTAS_highway_2"
BASE     = "https://zenodo.org/records/19665762/files"

ATTACKS = '''accelerationMultiplication constantPositionOffset constantSpeedOffset
dataReplay dosAttack feignedBraking positionMirroring randomPositionOffset
randomSpeedOffset reversedHeading suddenConstantSpeed suddenStop
timeDelayAttack trafficCongestionSybil zeroSpeedReport'''.split()

FEATS = ["s_spd","s_acl","s_hed","disp","dt","implied_spd",
         "spd_resid","hed_resid","acl_resid","latency"]
RESID = ["spd_resid","hed_resid","acl_resid"]
RECV  = ['sr_min','sr_max','sr_mean','n_recv']
FEATS_R = FEATS + RECV
KEY   = ['attack_type','sender_id','sendTime_s']

X  = lambda d: d[FEATS  ].replace([np.inf,-np.inf], np.nan).fillna(0)
XR = lambda d: d[FEATS_R].replace([np.inf,-np.inf], np.nan).fillna(0)


def num(v, idx=None):
    if v is None: return None
    if idx is not None:
        try: return float(str(v).split(",")[idx])
        except (ValueError, IndexError): return None
    try: return float(v)
    except (ValueError, TypeError): return None

def load(zip_path, split, label):
    outer = zipfile.ZipFile(zip_path)
    inner = zipfile.ZipFile(io.BytesIO(outer.read(
        [n for n in outer.namelist() if f"/{split}/" in n and n.endswith(".zip")][0])))
    rows = []
    for name in inner.namelist():
        if not name.endswith(".json"): continue
        for m in json.loads(inner.read(name)):
            s, r = m.get("sender", {}), m.get("receiver", {})
            rows.append({
                "rcvTime_s":  (num(m.get("rcvTime"))  or 0)/1e9,
                "sendTime_s": (num(m.get("sendTime")) or 0)/1e9,
                "sender_id":  m.get("sender_id"),
                "attacker":   int(m.get("attacker", 0)),
                "attack_type": label,
                "s_x": num(s.get("pos"),0), "s_y": num(s.get("pos"),1),
                "s_spd": num(s.get("spd")), "s_acl": num(s.get("acl")),
                "s_hed": num(s.get("hed")),
                "r_x": num(r.get("pos"),0), "r_y": num(r.get("pos"),1)})
    return pd.DataFrame(rows)

def load_split(split):
    return pd.concat([load(f"data/{SCENARIO}_{a}.zip", split, a)
                      for a in ATTACKS], ignore_index=True)

def build_features(df, group_cols=("attack_type","sender_id")):
    g_cols = list(group_cols)
    d = (df.sort_values(g_cols+["sendTime_s"])
           .drop_duplicates(g_cols+["sendTime_s"]).copy())
    g  = d.groupby(g_cols)
    dt = g.sendTime_s.diff(); dx, dy = g.s_x.diff(), g.s_y.diff()
    d["dt"], d["disp"] = dt, np.hypot(dx, dy)
    d["implied_spd"] = d.disp / dt.replace(0, np.nan)
    d["spd_resid"]   = d.implied_spd - d.s_spd
    mh = (90 - np.degrees(np.arctan2(dy, dx))) % 360
    hd = (mh - d.s_hed).abs() % 360
    d["hed_resid"] = np.minimum(hd, 360-hd)
    d["acl_resid"] = g.s_spd.diff()/dt.replace(0, np.nan) - d.s_acl
    d["latency"]   = d.rcvTime_s - d.sendTime_s
    d.loc[d.dt > GAP_MAX, RESID] = np.nan
    return d

def receiver_stats(split):
    out = []
    for a in ATTACKS:
        raw = load(f"data/{SCENARIO}_{a}.zip", split, a)
        raw = raw.dropna(subset=['s_x','s_y','r_x','r_y'])
        raw['sr_dist'] = np.hypot(raw.s_x-raw.r_x, raw.s_y-raw.r_y)
        g = raw.groupby(KEY)
        out.append(pd.DataFrame({'sr_min':g.sr_dist.min(),
                                 'sr_max':g.sr_dist.max(),
                                 'sr_mean':g.sr_dist.mean(),
                                 'n_recv':g.sr_dist.size()}).reset_index())
        del raw; gc.collect()
    return pd.concat(out, ignore_index=True)


F_TR, F_TE = f'{CACHE}/trR.parquet', f'{CACHE}/teR.parquet'

if os.path.exists(F_TR) and os.path.exists(F_TE):
    trR, teR = pd.read_parquet(F_TR), pd.read_parquet(F_TE)
    print(f"loaded from cache: {len(trR):,} / {len(teR):,}")
else:
    print("building from scratch (~16 min)...")
    for a in ATTACKS:
        p = f"data/{SCENARIO}_{a}.zip"
        if not (os.path.exists(p) and zipfile.is_zipfile(p)):
            if os.path.exists(p): os.remove(p)
            !wget -q --tries=3 "{BASE}/{SCENARIO}_{a}.zip?download=1" -O "{p}"
        assert zipfile.is_zipfile(p), f"download failed: {a}"
    print("  downloads ok")
    tr = build_features(load_split("Train"))
    te = build_features(load_split("Test"))
    print("  features ok")
    trR = tr.merge(receiver_stats("Train"), on=KEY, how='left')
    teR = te.merge(receiver_stats("Test"),  on=KEY, how='left')
    print("  receiver stats ok")
    trR.to_parquet(F_TR); teR.to_parquet(F_TE)
    print(f"  cached to {CACHE}")
    del tr, te; gc.collect()

tr, te = trR, teR                       # base features are a subset
b, bt  = trR[trR.attacker==0], teR[teR.attacker==0]

assert abs(b.dt.median()-1.0) < 0.02,                "dedup failed"
assert abs(b.spd_resid.abs().median()-0.098) < 0.02, "residuals wrong"
assert abs(b.hed_resid.abs().median()-2.9) < 0.4,    "heading convention wrong"
assert trR.spd_resid.isna().sum() > 0,               "NaNs lost"

ok  = trR[RESID].notna().all(axis=1)
okR = ok & trR[RECV].notna().all(axis=1)

rf  = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)
rf.fit(X(trR[ok]), trR[ok].attacker)
rfR = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)
rfR.fit(XR(trR[okR]), trR[okR].attacker)

pb, pr_ = rf.predict(X(teR)), rfR.predict(XR(teR))
bm = (teR.attacker == 0).values
print(f"\nbase  precision {precision_score(teR.attacker, pb):.3f}  "
      f"benign flag {pb[bm].mean():.4f}   (expect 0.828 / 0.046)")
print(f"+recv precision {precision_score(teR.attacker, pr_):.3f}  "
      f"benign flag {pr_[bm].mean():.4f}   (expect 0.761 / 0.072)")
print("\nREADY. Run Cell B.")


# =====================================================================
#  CELL B — N1: VEHICLE-LEVEL AGGREGATION
# =====================================================================
print("\n" + "=" * 70)
print("N1  VEHICLE-LEVEL AGGREGATION UNDER RECEIVER CONTEXT")
print("=" * 70)

KERNEL = ['constantPositionOffset', 'timeDelayAttack']
WATCH  = KERNEL + ['dataReplay', 'positionMirroring',
                   'randomPositionOffset', 'zeroSpeedReport']

teR = teR.copy()
teR['pred_base'] = rf.predict(X(teR))
teR['pred_recv'] = rfR.predict(XR(teR))

def vehicle_rates(df, col):
    out = {'BENIGN': df[df.attacker==0].groupby('sender_id')[col]
                        .agg(['mean','size'])}
    for a in ATTACKS:
        d = df[(df.attack_type==a) & (df.attacker==1)]
        if len(d): out[a] = d.groupby('sender_id')[col].agg(['mean','size'])
    return out

for tag, col in [('BASE (sender-only)','pred_base'),
                 ('RECEIVER CONTEXT','pred_recv')]:
    V = vehicle_rates(teR, col); ben = V['BENIGN']
    print(f"\n--- {tag} ---")
    print(f"benign vehicles n={len(ben)}  mean flag {ben['mean'].mean():.4f}  "
          f"median msgs/vehicle {ben['size'].median():.0f}")
    print(f"{'attack':26s} {'n_veh':>6s} {'mean':>8s} "
          f"{'>50%':>7s} {'>20%':>7s} {'>10%':>7s} {'>5%':>7s}")
    print(f"{'BENIGN':26s} {len(ben):6d} {ben['mean'].mean():8.4f} "
          f"{(ben['mean']>.50).mean():7.3f} {(ben['mean']>.20).mean():7.3f} "
          f"{(ben['mean']>.10).mean():7.3f} {(ben['mean']>.05).mean():7.3f}")
    for a in WATCH:
        if a not in V: continue
        s = V[a]['mean']
        print(f"{a:26s} {len(s):6d} {s.mean():8.4f} "
              f"{(s>.50).mean():7.3f} {(s>.20).mean():7.3f} "
              f"{(s>.10).mean():7.3f} {(s>.05).mean():7.3f}")

print("\n" + "-"*70)
print("OPERATING POINT: threshold set so 5% of BENIGN vehicles are flagged")
print("-"*70)
for tag, col in [('BASE','pred_base'), ('+RECV','pred_recv')]:
    V = vehicle_rates(teR, col); thr = V['BENIGN']['mean'].quantile(0.95)
    print(f"\n{tag}: vehicle threshold = {thr:.4f} of messages flagged")
    for a in WATCH:
        if a not in V: continue
        s = V[a]['mean']
        print(f"  {a:26s} caught {(s>thr).mean():.3f}  "
              f"({int((s>thr).sum())}/{len(s)})")

# pure range rule, no classifier
print("\n" + "-"*70)
print("PURE RANGE RULE AT VEHICLE LEVEL (no classifier)")
print("-"*70)
bR = teR[teR.attacker==0]
for q in (0.99, 0.999):
    t_msg = bR.sr_max.quantile(q)
    teR['flag_range'] = (teR.sr_max > t_msg).astype(int)
    V = vehicle_rates(teR, 'flag_range'); vt = V['BENIGN']['mean'].quantile(0.95)
    print(f"\nmessage threshold at benign q={q}: {t_msg:.1f} m   "
          f"vehicle threshold {vt:.4f}")
    for a in KERNEL:
        if a not in V: continue
        s = V[a]['mean']
        print(f"  {a:26s} caught {(s>vt).mean():.3f}  mean flag {s.mean():.4f}")

print("""
READ:
  Compare '+RECV' against 'BASE' at the 5% benign operating point. If
  attacker vehicles remain indistinguishable, state that explicitly at
  vehicle granularity and the abstract stands. If they separate, narrow
  the claim -- and note the theta=15 case survives either way, because
  those positions sit 2.15 m from ordinary lane positions and no range
  check reaches them.
""")

In [ ]:
# =====================================================================
#  SEED STABILITY — standalone. Paste into one cell and run.
#
#  Loads from the Drive cache you built earlier, so no re-download.
#  Runtime ~25 minutes (five Random Forest fits at 200 trees each).
#  Start it and do something else.
#
#  Prints a paste-ready sentence at the end.
# =====================================================================
import os
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
CACHE = '/content/drive/MyDrive/aintec_cache'

assert os.path.exists(f'{CACHE}/trR.parquet'), \
    "cache missing -- run the bootstrap cell (Cell A) first"

tr = pd.read_parquet(f'{CACHE}/trR.parquet')
te = pd.read_parquet(f'{CACHE}/teR.parquet')
print(f"loaded {len(tr):,} train / {len(te):,} test")

FEATS = ["s_spd","s_acl","s_hed","disp","dt","implied_spd",
         "spd_resid","hed_resid","acl_resid","latency"]
RESID = ["spd_resid","hed_resid","acl_resid"]
ATTACKS = sorted(te.attack_type.unique())
X = lambda d: d[FEATS].replace([np.inf,-np.inf], np.nan).fillna(0)

KERNEL = ['constantPositionOffset', 'timeDelayAttack']
WATCH  = KERNEL + ['dataReplay', 'positionMirroring',
                   'randomPositionOffset', 'zeroSpeedReport']

ok    = tr[RESID].notna().all(axis=1)
Xtr, ytr = X(tr[ok]), tr[ok].attacker
Xte      = X(te)
benmask  = (te.attacker == 0).values

SEEDS = [42, 7, 123, 2026, 31337]
aucs, precs, fps = [], [], []

for i, s in enumerate(SEEDS, 1):
    m = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=s)
    m.fit(Xtr, ytr)
    proba = m.predict_proba(Xte)[:, 1]
    pred  = m.predict(Xte)
    bp    = proba[benmask]

    row = {}
    for a in ATTACKS:
        mk = ((te.attack_type == a) & (te.attacker == 1)).values
        if not mk.sum():
            continue
        y = np.r_[np.zeros(len(bp)), np.ones(mk.sum())]
        row[a] = roc_auc_score(y, np.r_[bp, proba[mk]])
    aucs.append(row)
    precs.append(precision_score(te.attacker, pred))
    fps.append(float(pred[benmask].mean()))
    print(f"  [{i}/5] seed {s:6d}  precision {precs[-1]:.4f}  "
          f"cPO {row['constantPositionOffset']:.4f}  "
          f"tDA {row['timeDelayAttack']:.4f}")

A = pd.DataFrame(aucs)
A.to_csv('out/seed_auc.csv', index=False) if os.path.isdir('out') else None

print("\n" + "=" * 62)
print(f"precision    {np.mean(precs):.4f} +/- {np.std(precs, ddof=1):.4f}")
print(f"benign flag  {np.mean(fps):.4f} +/- {np.std(fps, ddof=1):.4f}")
print(f"\n{'attack':28s} {'mean AUC':>9s} {'sd':>8s}")
for a in WATCH:
    if a in A:
        print(f"{a:28s} {A[a].mean():9.4f} {A[a].std(ddof=1):8.4f}")

sd_all = A.std(ddof=1)
mx, mx_a = sd_all.max(), sd_all.idxmax()
c_sd = A['constantPositionOffset'].std(ddof=1)
t_sd = A['timeDelayAttack'].std(ddof=1)
print(f"\nlargest sd over all fifteen: {mx:.4f}  ({mx_a})")

print("\n" + "=" * 62)
print("PASTE INTO SECTION 7, replacing the 'one random seed' sentence:")
print("=" * 62)
print(f"""
Across five random seeds the standard deviation of per-attack AUC never
exceeds {mx:.3f}; for the two kernel attacks it is {c_sd:.3f} and {t_sd:.3f}.
Because the splits are predefined by the benchmark, this bounds
model-initialisation variance rather than data variance, and establishing
the latter would require a benchmark offering more than one split.
Hyperparameters are left at their defaults throughout: the aim is a
representative baseline, not an optimised detector.
""")

In [ ]:
# =====================================================================
#  ROUND 5 — the objections that decide the paper.
#
#   E1  LOCAL heading check (road-bearing proxy).   -> kills or confirms
#   E2  Random-split protocol with absolute position -> breaks the
#       circularity objection either way.
#   E3  Rotation displacement vs arc length + midpoint anchor.
#   E4  Are NextGen attacker vehicles distributionally neutral?
#
#  Loads from the Drive cache. ~25 min total.
# =====================================================================
import os, numpy as np, pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score
from scipy import stats

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
CACHE = '/content/drive/MyDrive/aintec_cache'
tr = pd.read_parquet(f'{CACHE}/trR.parquet')
te = pd.read_parquet(f'{CACHE}/teR.parquet')

FEATS = ["s_spd","s_acl","s_hed","disp","dt","implied_spd",
         "spd_resid","hed_resid","acl_resid","latency"]
RESID = ["spd_resid","hed_resid","acl_resid"]
ATTACKS = sorted(te.attack_type.unique())
SEED, GAP_MAX = 42, 2.0
X = lambda d, f=FEATS: d[f].replace([np.inf,-np.inf], np.nan).fillna(0)
b, bt = tr[tr.attacker==0], te[te.attacker==0]
print(f"loaded {len(tr):,} / {len(te):,}")

def contiguous(g, n=12):
    g = g.sort_values('sendTime_s')
    if len(g) < n: return None
    d = g.sendTime_s.diff().iloc[1:n]
    return g.head(n) if d.between(0.8,1.2).all() else None

def rotate(traj, th):
    d = traj.copy().reset_index(drop=True); t = np.radians(th)
    x0, y0 = d.s_x.iloc[0], d.s_y.iloc[0]
    dx, dy = d.s_x-x0, d.s_y-y0
    d['s_x'] = x0 + dx*np.cos(t) - dy*np.sin(t)
    d['s_y'] = y0 + dx*np.sin(t) + dy*np.cos(t)
    d['s_hed'] = (d.s_hed - th) % 360
    return d

def rotate_mid(traj, th):
    d = traj.copy().reset_index(drop=True); t = np.radians(th)
    m = len(d)//2
    x0, y0 = d.s_x.iloc[m], d.s_y.iloc[m]
    dx, dy = d.s_x-x0, d.s_y-y0
    d['s_x'] = x0 + dx*np.cos(t) - dy*np.sin(t)
    d['s_y'] = y0 + dx*np.sin(t) + dy*np.cos(t)
    d['s_hed'] = (d.s_hed - th) % 360
    return d

ben_pool = te[(te.attacker==0) & (te.attack_type=='constantPositionOffset')]
trajs = [t for t in (contiguous(g) for _, g in ben_pool.groupby('sender_id'))
         if t is not None]
print(f"{len(trajs)} contiguous benign trajectories")


# =====================================================================
# E1 — LOCAL HEADING CHECK.
#
#  A map of road SEGMENT BEARINGS is far cheaper than lane-level HD
#  geometry. We approximate it from data: for each claimed position,
#  look up the headings benign vehicles actually used near that point,
#  and measure the circular distance from the reported heading to the
#  closest of them. This needs only position-indexed bearings.
#
#  If a 15-degree rotation is caught here, Section 6.2's resolution
#  ladder is wrong and must be rebuilt. Better you find that than a PC.
# =====================================================================
print("\n" + "="*68); print("E1  LOCAL HEADING CHECK"); print("="*68)

pos_b  = bt[['s_x','s_y']].dropna()
head_b = bt.loc[pos_b.index, 's_hed'].values
nnh = NearestNeighbors(n_neighbors=25).fit(pos_b.values)

def local_head_dev(df, radius=15.0, k=25):
    """Circular distance from reported heading to the nearest heading any
    benign vehicle used within `radius` metres of the claimed position."""
    d, idx = nnh.kneighbors(df[['s_x','s_y']].values, n_neighbors=k)
    out = np.full(len(df), np.nan)
    hv  = df.s_hed.values
    for i in range(len(df)):
        near = idx[i][d[i] <= radius]
        if len(near) < 3: continue
        diff = np.abs((head_b[near] - hv[i]) % 360)
        out[i] = np.minimum(diff, 360-diff).min()
    return out

# calibrate on unmodified benign traffic
probe = bt.dropna(subset=['s_x','s_y','s_hed']).sample(20000, random_state=SEED)
dev_b = local_head_dev(probe)
dev_b = dev_b[~np.isnan(dev_b)]
THR = np.percentile(dev_b, 99)
print(f"benign local heading deviation: median {np.median(dev_b):.2f} deg  "
      f"p95 {np.percentile(dev_b,95):.2f}  p99 {THR:.2f}")
print(f"coverage (points with >=3 benign neighbours within 15 m): "
      f"{len(dev_b)/len(probe):.1%}\n")

print(f"{'theta':>6s} {'median dev':>11s} {'frac > p99':>11s} {'veh caught':>11s}")
rows = []
for th in [0,5,10,15,20,30,45,90]:
    rot = [rotate(t0, float(th)) for t0 in trajs]
    G   = pd.concat([r.assign(_v=i) for i, r in enumerate(rot)],
                    ignore_index=True).dropna(subset=['s_x','s_y','s_hed'])
    dv  = local_head_dev(G)
    ok  = ~np.isnan(dv)
    G   = G[ok]; dv = dv[ok]
    G['flag'] = (dv > THR).astype(int)
    veh = G.groupby('_v').flag.mean()
    rows.append({'theta': th, 'median_dev': float(np.median(dv)),
                 'frac_flag': float((dv > THR).mean()),
                 'veh_caught': float((veh > 0.05).mean())})
    print(f"{th:6d} {rows[-1]['median_dev']:11.2f} "
          f"{rows[-1]['frac_flag']:11.3f} {rows[-1]['veh_caught']:11.3f}")
pd.DataFrame(rows).to_csv('out/local_heading.csv', index=False)

print("""
READ E1:
  theta=0 is the control; 'frac > p99' there should be about 0.01.
  If frac_flag at theta=15 is LARGE (say >0.3), a segment-bearing map
  catches the live case and Section 6.2's ladder is wrong: the required
  resolution is bearing-level, not lane-level. Rewrite 6.2 around that.
  It is a MORE interesting claim -- the live case needs no HD map, only
  segment bearings, and nobody deploys even that.
  If frac_flag stays near 0.01, your ladder survives and you can say the
  obvious cheap check was tested and fails.
""")


# =====================================================================
# E2 — RANDOM SPLIT WITH ABSOLUTE POSITION.  Breaks the circularity.
# =====================================================================
print("="*68); print("E2  RANDOM SPLIT, ABSOLUTE POSITION AVAILABLE"); print("="*68)

pool = pd.concat([tr, te], ignore_index=True)
pool['vkey'] = pool.attack_type + '|' + pool.sender_id.astype(str)
keys = pool.vkey.unique()
rs = np.random.default_rng(SEED)
half = set(rs.permutation(keys)[:len(keys)//2])
A, B = pool[pool.vkey.isin(half)], pool[~pool.vkey.isin(half)]
print(f"random split by vehicle: {len(A):,} / {len(B):,}")

bA, bB = A[A.attacker==0], B[B.attacker==0]
print(f"geographic overlap  x {min(bA.s_x.max(),bB.s_x.max())-max(bA.s_x.min(),bB.s_x.min()):.0f} m"
      f"   y {min(bA.s_y.max(),bB.s_y.max())-max(bA.s_y.min(),bB.s_y.min()):.0f} m"
      "   (predefined split was 36 m in y)")

FEATS_P = FEATS + ['s_x','s_y']
for tag, feats in [('no position', FEATS), ('WITH position', FEATS_P)]:
    okA = A[RESID].notna().all(axis=1)
    m = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)
    m.fit(X(A[okA], feats), A[okA].attacker)
    p  = m.predict_proba(X(B, feats))[:,1]
    pd_= m.predict(X(B, feats))
    bm = (B.attacker==0).values
    print(f"\n[{tag}] precision {precision_score(B.attacker, pd_):.3f}  "
          f"benign flag {pd_[bm].mean():.4f}")
    for a in ['constantPositionOffset','timeDelayAttack',
              'randomPositionOffset','dataReplay']:
        mk = ((B.attack_type==a) & (B.attacker==1)).values
        if not mk.sum(): continue
        y = np.r_[np.zeros(bm.sum()), np.ones(mk.sum())]
        print(f"    {a:26s} AUC {roc_auc_score(y, np.r_[p[bm], p[mk]]):.3f}")

print("""
READ E2:
  Both outcomes help. If constantPositionOffset rises sharply WITH
  position under a random split, you have MEASURED what spatial
  grounding buys and the resolution ladder becomes empirical instead of
  rhetorical -- and the geographic-split result in 4.6 becomes a
  statement about deployment transfer, not about detectability.
  If it does not rise, the circularity objection is dead: position is
  available, transferable, and still does not help.
  timeDelayAttack should stay at chance either way -- it moves nothing
  spatial. That contrast is the cleanest thing you can report here.
""")


# =====================================================================
# E3 — DISPLACEMENT VS ARC LENGTH, AND MIDPOINT ANCHORING
# =====================================================================
print("="*68); print("E3  ROTATION GEOMETRY BY MESSAGE INDEX"); print("="*68)

cloud = bt[['s_x','s_y']].dropna()
rsx = np.random.default_rng(SEED); idx = rsx.permutation(len(cloud))
nn2 = NearestNeighbors(n_neighbors=1).fit(cloud.values[idx[:60000]])
probe2 = cloud.values[idx[60000:65000]]
BEN = float(np.median(nn2.kneighbors(probe2)[0]))
print(f"benign-to-benign NN baseline: {BEN:.2f} m\n")

for anchor, fn in [('first point', rotate), ('midpoint', rotate_mid)]:
    print(f"--- anchor: {anchor}, theta = 15 deg ---")
    print(f"{'msg':>4s} {'displacement':>13s} {'NN dist':>9s} {'>2.57 m':>9s}")
    rot = [fn(t0, 15.0) for t0 in trajs]
    for i in range(12):
        disp = np.array([np.hypot(r.s_x.iloc[i]-t.s_x.iloc[i],
                                  r.s_y.iloc[i]-t.s_y.iloc[i])
                         for r, t in zip(rot, trajs)])
        pts  = np.array([[r.s_x.iloc[i], r.s_y.iloc[i]] for r in rot])
        d    = nn2.kneighbors(pts)[0].ravel()
        print(f"{i:4d} {np.median(disp):13.1f} {np.median(d):9.2f} "
              f"{(d > 2.57).mean():9.3f}")
    print()

print("""
READ E3:
   corridor-wide MEDIAN of 2.15 m is
  dominated by low-displacement early messages. This table answers it
  per message. Report the honest version: state the message range over
  which displacement exceeds X metres AND NN distance stays under
  2.57 m, and give the length of that stretch. If no such stretch
  exists, the 'operationally live' claim needs rewriting -- say so.
  The midpoint anchor halves the maximum displacement but makes it
  symmetric; report whichever you use and why.
""")


# =====================================================================
# E4 — ARE ATTACKER VEHICLES DISTRIBUTIONALLY NEUTRAL?
#      Proposition 1 predicts the transformed stream matches the SAME
#      vehicle untransformed. AUC 0.5 against BENIGN traffic needs the
#      extra assumption that attacker trajectories are drawn like benign
#      ones. Test it.
# =====================================================================
print("="*68); print("E4  ATTACKER VEHICLE NEUTRALITY"); print("="*68)

# speed and acceleration are UNCHANGED by a constant position offset and
# by a time shift, so they carry the underlying trajectory's character.
for a in ['constantPositionOffset','timeDelayAttack']:
    d = te[(te.attack_type==a) & (te.attacker==1)]
    print(f"\n--- {a} ---")
    for c in ['s_spd','s_acl','dt','n_recv']:
        if c not in te.columns: continue
        av, bv = d[c].dropna(), bt[c].dropna()
        if len(av) < 50: continue
        u = stats.mannwhitneyu(av, bv, alternative='two-sided')
        pr = stats.mannwhitneyu(av, bv, alternative='less').statistic/(len(av)*len(bv))
        print(f"  {c:9s} attack med {av.median():9.3f}  benign {bv.median():9.3f}"
              f"   P(a<b) {pr:.3f}   p={u.pvalue:.1e}")
    print(f"  attacking vehicles {d.sender_id.nunique()}   "
          f"msgs/vehicle {len(d)/max(d.sender_id.nunique(),1):.0f}  "
          f"(benign {len(bt)/bt.sender_id.nunique():.0f})")

print("""
READ E4:
  If P(a<b) sits near 0.500 on speed and acceleration, attacker vehicles
  are drawn like benign ones and AUC = 0.500 IS the right null; the
  sub-chance tilt stays a small unexplained residual and you say so in
  one sentence.
  If they differ, state the corrected null explicitly: the proposition
  predicts identity between transformed and untransformed streams of the
  SAME vehicle, and AUC against benign traffic inherits whatever
  selection bias the benchmark's attacker assignment carries. That is a
  better sentence than 'unexplained' and it removes the objection.
""")

In [ ]:
# =====================================================================
#  E1 (v2) — LOCAL HEADING CHECK, corrected.
#
#  v1 bug: probe points were drawn from the same set used to build the
#  neighbour index, so every point matched ITSELF at distance 0 with an
#  identical heading. Benign deviation came out 0.00 at every quantile
#  and the threshold collapsed to zero, flagging everything.
#
#  v2: cloud and probe come from DISJOINT vehicle sets, and each
#  evaluated trajectory's own vehicle is excluded from the cloud.
# =====================================================================
import os
import numpy as np, pandas as pd
from sklearn.neighbors import NearestNeighbors

os.makedirs('out', exist_ok=True)          # v1 crashed here

SEED = 42
rs   = np.random.default_rng(SEED)

# ---- split benign vehicles into cloud / probe halves -----------------
bt_c = bt.dropna(subset=['s_x', 's_y', 's_hed']).copy()
vids = bt_c.sender_id.unique()
half = set(rs.permutation(vids)[:len(vids)//2])

cloud_df = bt_c[bt_c.sender_id.isin(half)]
probe_df = bt_c[~bt_c.sender_id.isin(half)]
print(f"cloud {len(cloud_df):,} points from {len(half)} vehicles")
print(f"probe {len(probe_df):,} points from {bt_c.sender_id.nunique()-len(half)} vehicles")

CLOUD_XY = cloud_df[['s_x', 's_y']].values
CLOUD_H  = cloud_df.s_hed.values
CLOUD_V  = cloud_df.sender_id.values
nnh = NearestNeighbors(n_neighbors=30).fit(CLOUD_XY)

def local_head_dev(df, radius=15.0, k=30, exclude_vehicles=None):
    """Circular distance from the reported heading to the nearest heading
    any OTHER benign vehicle used within `radius` m of the claimed point."""
    d, idx = nnh.kneighbors(df[['s_x', 's_y']].values, n_neighbors=k)
    out = np.full(len(df), np.nan)
    hv  = df.s_hed.values
    excl = set(exclude_vehicles) if exclude_vehicles is not None else set()
    for i in range(len(df)):
        near = idx[i][d[i] <= radius]
        if excl:
            near = near[~np.isin(CLOUD_V[near], list(excl))]
        if len(near) < 3:
            continue
        diff = np.abs((CLOUD_H[near] - hv[i]) % 360)
        out[i] = np.minimum(diff, 360 - diff).min()
    return out

# ---- calibrate on held-out benign vehicles ---------------------------
probe = probe_df.sample(min(20000, len(probe_df)), random_state=SEED)
dev_b = local_head_dev(probe)
cov   = np.mean(~np.isnan(dev_b))
dev_b = dev_b[~np.isnan(dev_b)]
THR   = np.percentile(dev_b, 99)
print(f"\nbenign local heading deviation (held-out vehicles):")
print(f"  median {np.median(dev_b):6.2f}   p90 {np.percentile(dev_b,90):6.2f}"
      f"   p95 {np.percentile(dev_b,95):6.2f}   p99 {THR:6.2f} deg")
print(f"  coverage: {cov:.1%} of points had >=3 benign neighbours within 15 m")
assert THR > 0.5, "threshold still degenerate -- self-matching not eliminated"

# ---- sweep -----------------------------------------------------------
print(f"\n{'theta':>6s} {'median dev':>11s} {'frac > p99':>11s} "
      f"{'veh caught':>11s} {'coverage':>9s}")
rows = []
for th in [0, 2, 5, 10, 15, 20, 30, 45, 90]:
    rot = [rotate(t0, float(th)) for t0 in trajs]
    G = pd.concat([r.assign(_v=i) for i, r in enumerate(rot)],
                  ignore_index=True).dropna(subset=['s_x','s_y','s_hed'])
    own = set(t.sender_id.iloc[0] for t in trajs)
    dv  = local_head_dev(G, exclude_vehicles=own)
    keep = ~np.isnan(dv)
    Gk, dvk = G[keep], dv[keep]
    Gk = Gk.copy(); Gk['flag'] = (dvk > THR).astype(int)
    veh = Gk.groupby('_v').flag.mean()
    rows.append({'theta': th,
                 'median_dev': float(np.median(dvk)),
                 'frac_flag':  float((dvk > THR).mean()),
                 'veh_caught': float((veh > 0.05).mean()),
                 'coverage':   float(keep.mean())})
    r = rows[-1]
    print(f"{th:6d} {r['median_dev']:11.2f} {r['frac_flag']:11.3f} "
          f"{r['veh_caught']:11.3f} {r['coverage']:9.3f}")

pd.DataFrame(rows).to_csv('out/local_heading.csv', index=False)

r0 = rows[0]
print(f"\ntheta=0 control: frac_flag {r0['frac_flag']:.4f} "
      f"(should be about 0.01 by construction of the threshold)")

print("""
READ:
  theta=0 must come back near 0.01. If it does not, the control is still
  broken and nothing below it means anything.

  Then read theta=15. If frac_flag is large, a map associating position
  with expected bearing catches the live case, and Section 6.2's ladder
  gains a rung BELOW lane-level HD geometry. Note what this does and
  does not touch: it is an EXTERNAL reference, so Proposition 1 and every
  content-based result stand unchanged. What changes is only the claimed
  COST of defeating the live case -- and the honest version is more
  interesting than the current one: the operationally live element needs
  no HD map, only position-indexed road bearings, and that is not
  deployed either.
""")

In [ ]:
# =====================================================================
#  E1 (v3) — LOCAL BEARING CHECK, properly separated.
#
#  What went wrong before:
#   v1  probe drawn from the index -> every point matched itself,
#       threshold collapsed to 0.
#   v2  cloud built from only HALF the benign vehicles -> unmodified
#       benign trajectories lost neighbours to the excluded half, so
#       "coverage" at theta=0 was 0.495 and the control read 0.053.
#
#  v3  Cloud = ALL benign points; the evaluated vehicle is excluded per
#      query (leave-one-vehicle-out). Calibration uses the same rule, so
#      the control is comparable by construction.
#
#  And the two checks are now reported separately, because they are
#  different mechanisms with different costs:
#     OCCUPANCY  is this position anywhere benign vehicles drive?
#     BEARING    given it is, is the heading consistent with local traffic?
# =====================================================================
import os
import numpy as np, pandas as pd
from sklearn.neighbors import NearestNeighbors

os.makedirs('out', exist_ok=True)
SEED = 42
RADIUS, KQ, MINN = 15.0, 60, 3

bt_c = bt.dropna(subset=['s_x','s_y','s_hed']).copy()
XY   = bt_c[['s_x','s_y']].values
HD   = bt_c.s_hed.values
VID  = bt_c.sender_id.values
nnh  = NearestNeighbors(n_neighbors=KQ).fit(XY)
print(f"cloud: {len(bt_c):,} benign points from {bt_c.sender_id.nunique()} vehicles")


def probe_points(xy, hed, own_vid):
    """Per point: (n_neighbours_found, min circular heading distance).
    The point's own vehicle is always excluded from its own query."""
    d, idx = nnh.kneighbors(xy, n_neighbors=KQ)
    n_out  = np.zeros(len(xy), dtype=int)
    dev    = np.full(len(xy), np.nan)
    for i in range(len(xy)):
        near = idx[i][d[i] <= RADIUS]
        near = near[VID[near] != own_vid[i]]
        n_out[i] = len(near)
        if len(near) >= MINN:
            df_ = np.abs((HD[near] - hed[i]) % 360)
            dev[i] = np.minimum(df_, 360 - df_).min()
    return n_out, dev


# ---- calibrate: leave-one-vehicle-out over benign traffic ------------
cal = bt_c.sample(min(20000, len(bt_c)), random_state=SEED)
n_c, dev_c = probe_points(cal[['s_x','s_y']].values,
                          cal.s_hed.values, cal.sender_id.values)
occ_c = n_c >= MINN
d_ok  = dev_c[occ_c]
T95, T99 = np.percentile(d_ok, [95, 99])
print(f"\nbenign calibration (leave-one-vehicle-out):")
print(f"  occupancy: {occ_c.mean():.3f} of benign points have >={MINN} "
      f"other-vehicle neighbours within {RADIUS:.0f} m")
print(f"  bearing deviation | occupied: median {np.median(d_ok):.2f}"
      f"  p90 {np.percentile(d_ok,90):.2f}  p95 {T95:.2f}  p99 {T99:.2f} deg")
print(f"  (the p99 tail is the opposite carriageway: points whose only"
      f" neighbours travel the other way)")

OCC_FLOOR = 1 - occ_c.mean()          # benign 'off-road' false-positive rate
print(f"\nbenign false-positive rates at these thresholds:")
print(f"  occupancy alone            {OCC_FLOOR:.4f}")
print(f"  bearing alone (p95)        {0.05*occ_c.mean():.4f}")
comb95 = OCC_FLOOR + occ_c.mean()*0.05
print(f"  either (occupancy or p95)  {comb95:.4f}")


# ---- sweep -----------------------------------------------------------
own = np.array([t.sender_id.iloc[0] for t in trajs])
print(f"\n{'theta':>6s} {'occupied':>9s} {'dev|occ':>9s} "
      f"{'bear>p95':>9s} {'EITHER':>8s} {'veh':>6s}")
rows = []
for th in [0, 2, 5, 10, 15, 20, 30, 45, 90]:
    rot = [rotate(t0, float(th)) for t0 in trajs]
    G = pd.concat([r.assign(_v=i, _own=own[i]) for i, r in enumerate(rot)],
                  ignore_index=True).dropna(subset=['s_x','s_y','s_hed'])
    n_g, dev_g = probe_points(G[['s_x','s_y']].values,
                              G.s_hed.values, G._own.values)
    occ = n_g >= MINN
    bearing_bad = np.zeros(len(G), dtype=bool)
    bearing_bad[occ] = dev_g[occ] > T95
    either = (~occ) | bearing_bad
    G = G.copy(); G['flag'] = either.astype(int)
    veh = G.groupby('_v').flag.mean()
    rows.append({'theta': th,
                 'occupied':  float(occ.mean()),
                 'dev_occ':   float(np.nanmedian(dev_g[occ])) if occ.any() else np.nan,
                 'bear_p95':  float(bearing_bad[occ].mean()) if occ.any() else np.nan,
                 'either':    float(either.mean()),
                 'veh':       float((veh > 0.05).mean())})
    r = rows[-1]
    print(f"{th:6d} {r['occupied']:9.3f} {r['dev_occ']:9.2f} "
          f"{r['bear_p95']:9.3f} {r['either']:8.3f} {r['veh']:6.3f}")

pd.DataFrame(rows).to_csv('out/local_bearing.csv', index=False)

r0 = rows[0]
print(f"\nCONTROL theta=0:")
print(f"  occupied {r0['occupied']:.3f}   (benign calibration {occ_c.mean():.3f})")
print(f"  EITHER   {r0['either']:.4f}   (benign expectation {comb95:.4f})")
ok = abs(r0['either'] - comb95) < 0.05
print("  -> control", "OK" if ok else "STILL OFF; do not read the rest")

print("""
COLUMNS
  occupied  fraction of claimed positions with >=3 other-vehicle benign
            neighbours within 15 m. 1 - this is the OCCUPANCY check: a
            position nothing ever drove through.
  dev|occ   median bearing deviation among occupied points, degrees.
  bear>p95  fraction of OCCUPIED points whose heading is inconsistent
            with local benign traffic. This is the BEARING check alone.
  EITHER    caught by occupancy or bearing. This is what a position-plus-
            bearing map buys.

WHAT TO WRITE
  These are two mechanisms at two costs, and they should be reported as
  two rungs, not one. Occupancy needs a drivable-region mask. Bearing
  needs the same mask annotated with expected direction. Both are static
  and far cheaper than current lane-level HD geometry. Neither is
  content-based, so Proposition 1 and every result in Section 4 stand.
  What changes is only Section 6.2's claim about the cost of defeating
  the live case -- and if EITHER at theta=15 is high, the honest version
  is stronger: the live case needs no HD map, only a static bearing-
  annotated road mask, and no misbehavior detector we are aware of uses
  one.
""")

In [ ]:
# --- complete the ladder: benchmark attacks through the same checks ---
print(f"{'attack':26s} {'occupied':>9s} {'dev|occ':>9s} {'bear>p95':>9s} {'EITHER':>8s}")
for a in ['constantPositionOffset','timeDelayAttack',
          'randomPositionOffset','dataReplay']:
    d = te[(te.attack_type==a) & (te.attacker==1)].dropna(
            subset=['s_x','s_y','s_hed'])
    if len(d) > 40000:
        d = d.sample(40000, random_state=SEED)
    n_a, dev_a = probe_points(d[['s_x','s_y']].values,
                              d.s_hed.values, d.sender_id.values)
    occ = n_a >= MINN
    bad = np.zeros(len(d), bool); bad[occ] = dev_a[occ] > T95
    either = (~occ) | bad
    print(f"{a:26s} {occ.mean():9.3f} "
          f"{np.nanmedian(dev_a[occ]) if occ.any() else float('nan'):9.2f} "
          f"{bad[occ].mean() if occ.any() else float('nan'):9.3f} "
          f"{either.mean():8.3f}")

In [ ]:
# schema-agnostic sketch — adapt field names to your loader
df['latency'] = df.rcvTime - df.sendTime

benign = df[df.label == 0].latency
tda    = df[(df.label == 1) & (df.attack_type == 'timeDelayAttack')].latency

for name, s in [('benign', benign), ('timeDelay', tda)]:
    print(name, len(s), s.median(), s.quantile([.01,.25,.75,.99]).values, s.min(), s.max())

In [ ]:
# ============================================================
# D1 — LATENCY DIAGNOSTIC (fast path, from cache)
#   Question: does rcvTime - sendTime shift under timeDelayAttack?
# ============================================================
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
CACHE = '/content/drive/MyDrive/aintec_cache'
te = pd.read_parquet(f'{CACHE}/teR.parquet')

sub = te[te.attack_type == 'timeDelayAttack']
ben = sub.loc[sub.attacker == 0, 'latency'].dropna()
tda = sub.loc[sub.attacker == 1, 'latency'].dropna()
ben_all = te.loc[te.attacker == 0, 'latency'].dropna()

print(f"{'group':22s} {'n':>9s} {'median':>12s} {'p01':>12s} {'p99':>12s} {'min':>12s} {'max':>12s}")
for nm, s in [('benign (tDA subset)', ben), ('timeDelayAttack', tda),
              ('benign (all subsets)', ben_all)]:
    q = s.quantile([.01, .99])
    print(f"{nm:22s} {len(s):9,d} {s.median():12.6f} {q.iloc[0]:12.6f} "
          f"{q.iloc[1]:12.6f} {s.min():12.6f} {s.max():12.6f}")

print(f"\nmedian difference (attack - benign): {tda.median() - ben.median():.9f} s")

# loader sanity: `or 0` in num() turns a missing timestamp into 0.0
for c in ['rcvTime_s', 'sendTime_s']:
    z = (te[c] == 0).sum()
    print(f"{c}: {z:,} rows exactly zero" + ("   <-- CHECK THIS" if z else ""))

In [ ]:
# ============================================================
# D3 — RAW SCHEMA: what fields exist that load() ignores?
# ============================================================
import io, json, zipfile
outer = zipfile.ZipFile(f"data/{SCENARIO}_timeDelayAttack.zip")
inner_name = [n for n in outer.namelist() if "/Test/" in n and n.endswith(".zip")][0]
inner = zipfile.ZipFile(io.BytesIO(outer.read(inner_name)))
names = [n for n in inner.namelist() if n.endswith(".json")]
msgs = json.loads(inner.read(names[0]))

print(f"{len(msgs)} messages in {names[0]}\n")
print("TOP-LEVEL KEYS:", sorted(msgs[0].keys()))
for k in ('sender', 'receiver'):
    if isinstance(msgs[0].get(k), dict):
        print(f"{k.upper()} KEYS:", sorted(msgs[0][k].keys()))
print("\nFULL FIRST MESSAGE:")
print(json.dumps(msgs[0], indent=2)[:1500])

# an attacker message, for comparison
atk = next((m for m in msgs if int(m.get('attacker', 0)) == 1), None)
if atk:
    print("\nFULL ATTACKER MESSAGE:")
    print(json.dumps(atk, indent=2)[:1500])
else:
    print("\nno attacker message in this file — try another from `names`")

In [ ]:
# after printing the key lists in D3:
TIMEKEYS = [k for k in msgs[0] if any(s in k.lower()
            for s in ('time', 'delta', 'gen', 'stamp'))]
print("TIME-LIKE TOP-LEVEL KEYS:", TIMEKEYS)
for k in ('sender', 'receiver'):
    if isinstance(msgs[0].get(k), dict):
        print(f"  in {k}:", [j for j in msgs[0][k]
               if any(s in j.lower() for s in ('time','delta','gen','stamp'))])

# and: does anything look like ground truth?
GTKEYS = [k for k in msgs[0] if any(s in k.lower()
          for s in ('real', 'true', 'gt', 'noise', 'actual', 'ground'))]
print("GROUND-TRUTH-LIKE KEYS:", GTKEYS)

In [ ]:
# ============================================================
# D3 (standalone) — raw schema for timeDelayAttack
# ============================================================
import os, io, json, zipfile

SCENARIO = "InTAS_highway_2"
BASE     = "https://zenodo.org/records/19665762/files"
os.makedirs("data", exist_ok=True)

p = f"data/{SCENARIO}_timeDelayAttack.zip"
if not (os.path.exists(p) and zipfile.is_zipfile(p)):
    if os.path.exists(p): os.remove(p)
    !wget -q --tries=3 "{BASE}/{SCENARIO}_timeDelayAttack.zip?download=1" -O "{p}"
assert zipfile.is_zipfile(p), "download failed"

outer = zipfile.ZipFile(p)
inner = zipfile.ZipFile(io.BytesIO(outer.read(
    [n for n in outer.namelist() if "/Test/" in n and n.endswith(".zip")][0])))
names = [n for n in inner.namelist() if n.endswith(".json")]

# find a file that actually contains an attacker message
msgs, atk = None, None
for nm in names[:40]:
    cand = json.loads(inner.read(nm))
    a = next((m for m in cand if int(m.get('attacker', 0)) == 1), None)
    if a is not None:
        msgs, atk, src = cand, a, nm
        break
print(f"using {src}: {len(msgs)} messages\n")

print("TOP-LEVEL KEYS:", sorted(msgs[0].keys()))
for k in ('sender', 'receiver'):
    if isinstance(msgs[0].get(k), dict):
        print(f"{k.upper()} KEYS:", sorted(msgs[0][k].keys()))

flat = list(msgs[0].keys()) + [f"{k}.{j}" for k in ('sender','receiver')
            if isinstance(msgs[0].get(k), dict) for j in msgs[0][k]]
print("\nTIME-LIKE :", [k for k in flat if any(s in k.lower()
      for s in ('time','delta','gen','stamp','age'))])
print("GT-LIKE   :", [k for k in flat if any(s in k.lower()
      for s in ('real','true','gt','noise','actual','ground','orig'))])

print("\nBENIGN MESSAGE:\n", json.dumps(msgs[0], indent=2)[:1200])
print("\nATTACKER MESSAGE:\n", json.dumps(atk, indent=2)[:1200])

In [ ]:
sub = te[te.attack_type == 'timeDelayAttack']
for nm, d in [('benign', sub[sub.attacker == 0]), ('tDA', sub[sub.attacker == 1])]:
    print(f"{nm:8s} sr_mean med {d.sr_mean.median():7.1f}  "
          f"sr_max med {d.sr_max.median():7.1f}  n={len(d):,}")
print(f"\nbenign median speed: {sub[sub.attacker==0].s_spd.median():.1f} m/s")

In [ ]:
# E1 — is pos_noise the sensor error?  Benign de-noised residuals should collapse.
d = te[te.attacker == 0].copy()   # needs s_x_true; see E2 loader below

In [ ]:
# ============================================================
# E2 — recover true trajectories from receiver blocks
# ============================================================
import io, json, zipfile, numpy as np, pandas as pd

def px(v, i):
    try: return float(str(v).split(",")[i])
    except Exception: return np.nan

def read_subset(attack):
    p = f"data/{SCENARIO}_{attack}.zip"
    outer = zipfile.ZipFile(p)
    inner = zipfile.ZipFile(io.BytesIO(outer.read(
        [n for n in outer.namelist() if "/Test/" in n and n.endswith(".zip")][0])))
    sent, true = [], []
    for name in inner.namelist():
        if not name.endswith(".json"): continue
        rid = name.split("/")[-1].replace(".json", "")   # receiver vehicle id
        for m in json.loads(inner.read(name)):
            s, r = m.get("sender", {}), m.get("receiver", {})
            sent.append((m["sender_id"], m["sendTime"]/1e9,
                         px(s.get("pos"),0), px(s.get("pos"),1),
                         px(s.get("pos_noise"),0), px(s.get("pos_noise"),1),
                         s.get("spd"), int(m.get("attacker",0))))
            true.append((rid, m["rcvTime"]/1e9,
                         px(r.get("pos"),0), px(r.get("pos"),1)))
    S = pd.DataFrame(sent, columns=['vid','t','x','y','nx','ny','spd','atk'])
    T = pd.DataFrame(true, columns=['vid','t','tx','ty']).drop_duplicates(['vid','t'])
    return S.drop_duplicates(['vid','t']), T

# ---- (a) pos_noise semantics, on benign senders ----
S, T = read_subset('constantPositionOffset')
ben = S[S.atk == 0].merge(T, on='vid', suffixes=('','_r'))
ben = ben[(ben.t - ben.t_r).abs() < 0.05]          # match same instant
print(f"matched benign sender/receiver rows: {len(ben):,}")
for tag, ex, ey in [('raw pos vs true', ben.x, ben.y),
                    ('pos - pos_noise', ben.x - ben.nx, ben.y - ben.ny),
                    ('pos + pos_noise', ben.x + ben.nx, ben.y + ben.ny)]:
    err = np.hypot(ex - ben.tx, ey - ben.ty)
    print(f"  {tag:18s} median {err.median():8.3f} m   p90 {err.quantile(.9):8.3f}")

# ---- (b) constantPositionOffset magnitude ----
atk = S[S.atk == 1].merge(T, on='vid', suffixes=('','_r'))
atk = atk[(atk.t - atk.t_r).abs() < 0.05]
off = np.hypot(atk.x - atk.tx, atk.y - atk.ty)
print(f"\nconstantPositionOffset |c|: n={len(off):,}  median {off.median():.1f} m"
      f"  p05 {off.quantile(.05):.1f}  p95 {off.quantile(.95):.1f}")
print("per-vehicle offset (first 10):")
print(atk.assign(mag=off).groupby('vid').mag.agg(['median','std','size']).head(10).round(2))

# ---- (c) timeDelayAttack: find tau by lag alignment ----
S2, T2 = read_subset('timeDelayAttack')
A = S2[S2.atk == 1]
taus = []
for vid, g in A.groupby('vid'):
    tt = T2[T2.vid == vid].sort_values('t')
    if len(tt) < 30 or len(g) < 10: continue
    best, bt_ = None, None
    for lag in np.arange(0.0, 15.1, 0.5):
        q = np.interp(g.t.values - lag, tt.t.values, tt.tx.values)
        w = np.interp(g.t.values - lag, tt.t.values, tt.ty.values)
        e = np.median(np.hypot(g.x.values - q, g.y.values - w))
        if best is None or e < best: best, bt_ = e, lag
    taus.append((vid, bt_, best))
D = pd.DataFrame(taus, columns=['vid','tau_s','resid_m'])
print(f"\ntimeDelayAttack tau over {len(D)} vehicles:")
print(D.tau_s.describe().round(2).to_string())
print(f"median alignment residual at best lag: {D.resid_m.median():.2f} m")

In [ ]:
def ensure(attack):
    p = f"data/{SCENARIO}_{attack}.zip"
    if not (os.path.exists(p) and zipfile.is_zipfile(p)):
        if os.path.exists(p): os.remove(p)
        !wget -q --tries=3 "{BASE}/{SCENARIO}_{attack}.zip?download=1" -O "{p}"
    assert zipfile.is_zipfile(p), f"download failed: {attack}"
    return p

In [ ]:
import os, zipfile
SCENARIO = "InTAS_highway_2"
BASE = "https://zenodo.org/records/19665762/files"
for a in ('constantPositionOffset', 'timeDelayAttack'):
    ensure(a)
print("ready")

In [ ]:
def read_subset(attack):
    p = ensure(attack)
    outer = zipfile.ZipFile(p)
    ...   # rest unchanged

In [ ]:
# ============================================================
# E2 — recover true trajectories from receiver blocks
# ============================================================
import io, json, zipfile, gc, numpy as np, pandas as pd

def px(v, i):
    try: return float(str(v).split(",")[i])
    except Exception: return np.nan

def read_subset(attack):
    p = ensure(attack)
    outer = zipfile.ZipFile(p)
    inner = zipfile.ZipFile(io.BytesIO(outer.read(
        [n for n in outer.namelist() if "/Test/" in n and n.endswith(".zip")][0])))
    sent, true = [], []
    for name in inner.namelist():
        if not name.endswith(".json"): continue
        rid = name.split("/")[-1].replace(".json", "")
        for m in json.loads(inner.read(name)):
            s, r = m.get("sender", {}), m.get("receiver", {})
            sent.append((m["sender_id"], m["sendTime"]/1e9,
                         px(s.get("pos"),0),       px(s.get("pos"),1),
                         px(s.get("pos_noise"),0), px(s.get("pos_noise"),1),
                         s.get("spd"), int(m.get("attacker", 0))))
            true.append((rid, m["rcvTime"]/1e9,
                         px(r.get("pos"),0),       px(r.get("pos"),1),
                         px(r.get("pos_noise"),0), px(r.get("pos_noise"),1)))
    S = (pd.DataFrame(sent, columns=['vid','t','x','y','nx','ny','spd','atk'])
           .drop_duplicates(['vid','t']).sort_values('t').reset_index(drop=True))
    T = (pd.DataFrame(true, columns=['vid','t','tx','ty','tnx','tny'])
           .drop_duplicates(['vid','t']).sort_values('t').reset_index(drop=True))
    del outer, inner; gc.collect()
    return S, T

def match(S, T, tol=0.05):
    """Nearest-in-time join per vehicle. Avoids the cartesian blowup."""
    return pd.merge_asof(S, T, on='t', by='vid',
                         direction='nearest', tolerance=tol).dropna(subset=['tx'])

# ---------- (a) pos_noise semantics, benign senders ----------
S, T = read_subset('constantPositionOffset')
print(f"sender rows {len(S):,}   true-position rows {len(T):,}")
ben = match(S[S.atk == 0], T)
print(f"matched benign rows: {len(ben):,}\n")

# de-noise the reference too, using whichever convention wins
for tag, ex, ey in [('sender raw        ', ben.x,          ben.y),
                    ('sender - noise    ', ben.x - ben.nx, ben.y - ben.ny),
                    ('sender + noise    ', ben.x + ben.nx, ben.y + ben.ny)]:
    for rtag, rx, ry in [('vs ref raw', ben.tx, ben.ty),
                         ('vs ref-noise', ben.tx - ben.tnx, ben.ty - ben.tny)]:
        e = np.hypot(ex - rx, ey - ry)
        print(f"  {tag} {rtag:14s} median {e.median():8.3f} m   p90 {e.quantile(.9):8.3f}")
print(f"\n|pos_noise| median: {np.hypot(ben.nx, ben.ny).median():.2f} m")

In [ ]:
SGN = -1   # <-- set from part (a): -1 if "sender - noise" won, +1 if "+ noise"

def denoise(d):
    return d.x + SGN*d.nx, d.y + SGN*d.ny
def denoise_ref(d):
    return d.tx + SGN*d.tnx, d.ty + SGN*d.tny

# ---------- (b) constantPositionOffset magnitude ----------
atk = match(S[S.atk == 1], T)
ax, ay = denoise(atk); rx, ry = denoise_ref(atk)
off = np.hypot(ax - rx, ay - ry)
print(f"|c|  n={len(off):,}  median {off.median():.2f} m  "
      f"p05 {off.quantile(.05):.2f}  p95 {off.quantile(.95):.2f}")
per = atk.assign(mag=off).groupby('vid').mag.agg(['median','std','size'])
print(f"\nper-vehicle |c|: median of medians {per['median'].median():.2f} m, "
      f"spread {per['median'].min():.1f}–{per['median'].max():.1f} m, "
      f"within-vehicle sd median {per['std'].median():.3f} m")
print(per.head(8).round(2).to_string())
del S, T; gc.collect()

# ---------- (c) timeDelayAttack: tau by lag alignment ----------
S2, T2 = read_subset('timeDelayAttack')
A = S2[S2.atk == 1]
rows = []
for vid, g in A.groupby('vid'):
    tt = T2[T2.vid == vid].sort_values('t')
    if len(tt) < 30 or len(g) < 10: continue
    gx, gy = denoise(g); ttx, tty = denoise_ref(tt.rename(columns={}))
    best = None
    for lag in np.arange(-2.0, 15.01, 0.1):
        q = np.interp(g.t.values - lag, tt.t.values, ttx.values)
        w = np.interp(g.t.values - lag, tt.t.values, tty.values)
        e = np.median(np.hypot(gx.values - q, gy.values - w))
        if best is None or e < best[1]: best = (lag, e)
    rows.append((vid, best[0], best[1]))
D = pd.DataFrame(rows, columns=['vid','tau_s','resid_m'])
print(f"\ntau over {len(D)} attacker vehicles:")
print(D.tau_s.describe().round(3).to_string())
print(f"median alignment residual at best lag: {D.resid_m.median():.2f} m")
print(f"  (compare: residual at lag 0 tells you whether tau is real)")

In [ ]:
S, T = read_subset('constantPositionOffset')
print("S.vid samples:", S.vid.unique()[:3])
print("T.vid samples:", T.vid.unique()[:3])
print("overlap:", len(set(S.vid) & set(T.vid)))

In [ ]:
rid = name.replace('\\', '/').split('/')[-1].replace('.json', '')

In [ ]:
def read_subset(attack):
    p = ensure(attack)
    outer = zipfile.ZipFile(p)
    inner = zipfile.ZipFile(io.BytesIO(outer.read(
        [n for n in outer.namelist() if "/Test/" in n and n.endswith(".zip")][0])))
    sent, true = [], []
    for name in inner.namelist():
        if not name.endswith(".json"): continue
        rid = name.replace('\\', '/').split('/')[-1].replace('.json', '')   # <-- FIX
        for m in json.loads(inner.read(name)):
            s, r = m.get("sender", {}), m.get("receiver", {})
            sent.append((m["sender_id"], m["sendTime"]/1e9,
                         px(s.get("pos"),0),       px(s.get("pos"),1),
                         px(s.get("pos_noise"),0), px(s.get("pos_noise"),1),
                         s.get("spd"), int(m.get("attacker", 0))))
            true.append((rid, m["rcvTime"]/1e9,
                         px(r.get("pos"),0),       px(r.get("pos"),1),
                         px(r.get("pos_noise"),0), px(r.get("pos_noise"),1)))
    S = (pd.DataFrame(sent, columns=['vid','t','x','y','nx','ny','spd','atk'])
           .drop_duplicates(['vid','t']).sort_values('t').reset_index(drop=True))
    T = (pd.DataFrame(true, columns=['vid','t','tx','ty','tnx','tny'])
           .drop_duplicates(['vid','t']).sort_values('t').reset_index(drop=True))
    del outer, inner; gc.collect()
    return S, T

In [ ]:
S, T = read_subset('constantPositionOffset')
print(f"S {len(S):,} rows / {S.vid.nunique()} vehicles   "
      f"T {len(T):,} rows / {T.vid.nunique()} vehicles")
print(f"vid overlap: {len(set(S.vid) & set(T.vid))}")

atk_vids = set(S.loc[S.atk == 1, 'vid'])
print(f"attacker vehicles {len(atk_vids)}, also logged as receivers: "
      f"{len(atk_vids & set(T.vid))}")

ben = match(S[S.atk == 0], T)
print(f"matched benign rows: {len(ben):,} "
      f"({len(ben)/max(len(S[S.atk==0]),1):.1%} of benign sender rows)")

In [ ]:
# ---------- (a) pos_noise semantics, via interpolation ----------
def true_at(T, vid, times):
    tt = T[T.vid == vid].sort_values('t')
    if len(tt) < 2: return None, None
    return (np.interp(times, tt.t.values, tt.tx.values),
            np.interp(times, tt.t.values, tt.ty.values),
            np.interp(times, tt.t.values, (tt.tx - tt.tnx).values),
            np.interp(times, tt.t.values, (tt.ty - tt.tny).values))

rows = []
for vid, g in S[S.atk == 0].groupby('vid'):
    r = true_at(T, vid, g.t.values)
    if r[0] is None: continue
    rx, ry, rdx, rdy = r
    rows.append(pd.DataFrame({'x': g.x.values, 'y': g.y.values,
                              'nx': g.nx.values, 'ny': g.ny.values,
                              'rx': rx, 'ry': ry, 'rdx': rdx, 'rdy': rdy}))
B = pd.concat(rows, ignore_index=True)
print(f"benign rows with interpolated reference: {len(B):,}\n")

for stag, sx, sy in [('raw      ', B.x,        B.y),
                     ('- noise  ', B.x - B.nx, B.y - B.ny),
                     ('+ noise  ', B.x + B.nx, B.y + B.ny)]:
    for rtag, rx, ry in [('ref raw  ', B.rx, B.ry), ('ref-noise', B.rdx, B.rdy)]:
        e = np.hypot(sx - rx, sy - ry)
        print(f"  sender {stag} vs {rtag}  median {e.median():7.3f} m  p90 {e.quantile(.9):7.3f}")
print(f"\n|pos_noise| median: {np.hypot(B.nx, B.ny).median():.2f} m")

In [ ]:
SGN = -1        # from (a): -1 if "- noise" won on the sender side
REF_DENOISE = True   # from (a): True if "ref-noise" won on the reference side

def ref_at(T, vid, times):
    tt = T[T.vid == vid].sort_values('t')
    if len(tt) < 2: return None
    X = (tt.tx + SGN*tt.tnx) if REF_DENOISE else tt.tx
    Y = (tt.ty + SGN*tt.tny) if REF_DENOISE else tt.ty
    return (np.interp(times, tt.t.values, X.values),
            np.interp(times, tt.t.values, Y.values))

# (b) |c|
mags = []
for vid, g in S[S.atk == 1].groupby('vid'):
    r = ref_at(T, vid, g.t.values)
    if r is None: continue
    mags.append(pd.DataFrame({'vid': vid,
        'mag': np.hypot(g.x.values + SGN*g.nx.values - r[0],
                        g.y.values + SGN*g.ny.values - r[1])}))
M = pd.concat(mags, ignore_index=True)
print(f"|c|  n={len(M):,}  median {M.mag.median():.2f} m  "
      f"p05 {M.mag.quantile(.05):.2f}  p95 {M.mag.quantile(.95):.2f}")
per = M.groupby('vid').mag.agg(['median','std','size'])
print(f"per-vehicle: median of medians {per['median'].median():.2f} m, "
      f"range {per['median'].min():.1f}-{per['median'].max():.1f} m, "
      f"within-vehicle sd median {per['std'].median():.3f} m")

# (c) tau
S2, T2 = read_subset('timeDelayAttack')
out = []
for vid, g in S2[S2.atk == 1].groupby('vid'):
    tt = T2[T2.vid == vid].sort_values('t')
    if len(tt) < 30 or len(g) < 10: continue
    X = (tt.tx + SGN*tt.tnx) if REF_DENOISE else tt.tx
    Y = (tt.ty + SGN*tt.tny) if REF_DENOISE else tt.ty
    gx, gy = g.x.values + SGN*g.nx.values, g.y.values + SGN*g.ny.values
    errs = [(lag, np.median(np.hypot(
                gx - np.interp(g.t.values - lag, tt.t.values, X.values),
                gy - np.interp(g.t.values - lag, tt.t.values, Y.values))))
            for lag in np.arange(-2.0, 15.01, 0.1)]
    lag0 = dict(np.round(errs, 6)).get(0.0, np.nan)
    best = min(errs, key=lambda e: e[1])
    out.append((vid, best[0], best[1], lag0))
D = pd.DataFrame(out, columns=['vid','tau_s','resid_m','resid_at_0'])
print(f"\ntau over {len(D)} vehicles: median {D.tau_s.median():.2f} s, "
      f"IQR [{D.tau_s.quantile(.25):.2f}, {D.tau_s.quantile(.75):.2f}]")
print(f"residual at best lag {D.resid_m.median():.2f} m  vs at lag 0 "
      f"{D.resid_at_0.median():.2f} m")

In [ ]:
def read_pairs(attack):
    p = ensure(attack); outer = zipfile.ZipFile(p)
    inner = zipfile.ZipFile(io.BytesIO(outer.read(
        [n for n in outer.namelist() if "/Test/" in n and n.endswith(".zip")][0])))
    rows = []
    for name in inner.namelist():
        if not name.endswith(".json"): continue
        rid = name.replace('\\','/').split('/')[-1].replace('.json','')
        for m in json.loads(inner.read(name)):
            r = m.get("receiver", {})
            rows.append((rid, m["rcvTime"]/1e9, int(m.get("attacker",0)),
                         px(r.get("pos"),0) - px(r.get("pos_noise"),0),
                         px(r.get("pos"),1) - px(r.get("pos_noise"),1)))
    del outer, inner; gc.collect()
    return pd.DataFrame(rows, columns=['rid','r_t','from_atk','tx','ty'])

P = read_pairs('timeDelayAttack')
errs = []
for rid, g in P.groupby('rid'):
    ben = g[g.from_atk == 0].sort_values('r_t').drop_duplicates('r_t')
    atk = g[g.from_atk == 1].sort_values('r_t')
    if len(ben) < 30 or len(atk) < 5: continue
    a = atk[(atk.r_t >= ben.r_t.min()) & (atk.r_t <= ben.r_t.max())]
    if not len(a): continue
    errs.append(np.hypot(a.tx.values - np.interp(a.r_t.values, ben.r_t.values, ben.tx.values),
                         a.ty.values - np.interp(a.r_t.values, ben.r_t.values, ben.ty.values)))
E = np.concatenate(errs)
print(f"receiver self-position discrepancy on attacker-sourced rows: "
      f"n={len(E):,}  median {np.median(E):.3f} m  p90 {np.percentile(E,90):.3f} m")

In [ ]:
for atk_name in ('constantPositionOffset', 'timeDelayAttack'):
    P = read_pairs(atk_name)
    errs = []
    for rid, g in P.groupby('rid'):
        ben = g[g.from_atk == 0].sort_values('r_t').drop_duplicates('r_t')
        a   = g[g.from_atk == 1].sort_values('r_t')
        if len(ben) < 30 or len(a) < 5: continue
        a = a[(a.r_t >= ben.r_t.min()) & (a.r_t <= ben.r_t.max())]
        if not len(a): continue
        errs.append(np.hypot(
            a.tx.values - np.interp(a.r_t.values, ben.r_t.values, ben.tx.values),
            a.ty.values - np.interp(a.r_t.values, ben.r_t.values, ben.ty.values)))
    E = np.concatenate(errs)
    print(f"{atk_name:24s} n={len(E):,}  median {np.median(E):7.3f} m  "
          f"p90 {np.percentile(E,90):7.3f} m")


In [ ]:
P = read_pairs('timeDelayAttack')
for shift in np.arange(-4.0, 4.01, 0.2):
    errs = []
    for rid, g in P.groupby('rid'):
        ben = g[g.from_atk == 0].sort_values('r_t').drop_duplicates('r_t')
        a   = g[g.from_atk == 1].sort_values('r_t')
        if len(ben) < 30 or len(a) < 5: continue
        t = a.r_t.values + shift
        k = (t >= ben.r_t.min()) & (t <= ben.r_t.max())
        if not k.any(): continue
        errs.append(np.hypot(
            a.tx.values[k] - np.interp(t[k], ben.r_t.values, ben.tx.values),
            a.ty.values[k] - np.interp(t[k], ben.r_t.values, ben.ty.values)))
    if errs:
        print(f"shift {shift:+5.1f} s   median {np.median(np.concatenate(errs)):7.3f} m")

In [ ]:
def read_pairs2(attack):
    p = ensure(attack); outer = zipfile.ZipFile(p)
    inner = zipfile.ZipFile(io.BytesIO(outer.read(
        [n for n in outer.namelist() if "/Test/" in n and n.endswith(".zip")][0])))
    rows = []
    for name in inner.namelist():
        if not name.endswith(".json"): continue
        rid = name.replace('\\','/').split('/')[-1].replace('.json','')
        for m in json.loads(inner.read(name)):
            r = m.get("receiver", {})
            rows.append((rid, m["sender_id"], m["rcvTime"]/1e9,
                         int(m.get("attacker",0)),
                         px(r.get("pos"),0) - px(r.get("pos_noise"),0),
                         px(r.get("pos"),1) - px(r.get("pos_noise"),1)))
    del outer, inner; gc.collect()
    return pd.DataFrame(rows, columns=['rid','sid','r_t','from_atk','tx','ty'])

P = read_pairs2('timeDelayAttack')
res = []
for rid, g in P.groupby('rid'):
    ben = g[g.from_atk == 0].sort_values('r_t').drop_duplicates('r_t')
    if len(ben) < 30: continue
    for sid, a in g[g.from_atk == 1].groupby('sid'):
        a = a.sort_values('r_t'); best = None
        for sh in np.arange(-6, 6.01, 0.1):
            t = a.r_t.values + sh
            k = (t >= ben.r_t.min()) & (t <= ben.r_t.max())
            if k.sum() < 5: continue
            e = np.median(np.hypot(
                a.tx.values[k] - np.interp(t[k], ben.r_t.values, ben.tx.values),
                a.ty.values[k] - np.interp(t[k], ben.r_t.values, ben.ty.values)))
            if best is None or e < best[1]: best = (sh, e)
        if best: res.append((rid, sid, best[0], best[1]))
R = pd.DataFrame(res, columns=['rid','sid','shift_s','resid_m'])
print(f"pairs {len(R)}   median residual at best shift {R.resid_m.median():.3f} m")
per = R.groupby('sid').shift_s.agg(['median','std','size'])
print(f"per-attacker shift: median {per['median'].median():.2f} s, "
      f"range {per['median'].min():.2f} to {per['median'].max():.2f}, "
      f"within-attacker sd median {per['std'].median():.3f} s")

# does the rcvTime shift match tau from part (c), vehicle by vehicle?
if 'D' in dir():
    cmp = per[['median']].join(D.set_index('vid').tau_s, how='inner')
    cmp['sum'] = cmp['median'] + cmp.tau_s
    print(f"\nmatched vehicles {len(cmp)}   |shift| vs tau: "
          f"corr {cmp['median'].abs().corr(cmp.tau_s):.3f}   "
          f"median(shift + tau) {cmp['sum'].median():.3f} s")

In [ ]:
# ============================================================
# MASK — three limbs, calibration, rotation sweep, attack rows
# ============================================================
import os, numpy as np, pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

CACHE = '/content/drive/MyDrive/aintec_cache'
tr = pd.read_parquet(f'{CACHE}/trR.parquet')
te = pd.read_parquet(f'{CACHE}/teR.parquet')
FEATS = ["s_spd","s_acl","s_hed","disp","dt","implied_spd",
         "spd_resid","hed_resid","acl_resid","latency"]
RESID = ["spd_resid","hed_resid","acl_resid"]
ATTACKS = sorted(te.attack_type.unique())
SEED, GAP_MAX = 42, 2.0
X = lambda d, f=FEATS: d[f].replace([np.inf,-np.inf], np.nan).fillna(0)
b, bt = tr[tr.attacker==0], te[te.attacker==0]

ok = tr[RESID].notna().all(axis=1)
rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)
rf.fit(X(tr[ok]), tr[ok].attacker)
FP = float(rf.predict(X(te))[(te.attacker==0).values].mean())
print(f"rf restored: precision {precision_score(te.attacker, rf.predict(X(te))):.3f}"
      f"  benign flag {FP:.4f}   (expect 0.828 / 0.046)")

def contiguous(g, n=12):
    g = g.sort_values('sendTime_s')
    if len(g) < n: return None
    d = g.sendTime_s.diff().iloc[1:n]
    return g.head(n) if d.between(0.8,1.2).all() else None

def rotate(traj, th):
    d = traj.copy().reset_index(drop=True); t = np.radians(th)
    x0, y0 = d.s_x.iloc[0], d.s_y.iloc[0]
    dx, dy = d.s_x-x0, d.s_y-y0
    d['s_x'] = x0 + dx*np.cos(t) - dy*np.sin(t)
    d['s_y'] = y0 + dx*np.sin(t) + dy*np.cos(t)
    d['s_hed'] = (d.s_hed - th) % 360
    return d

ben_pool = te[(te.attacker==0) & (te.attack_type=='constantPositionOffset')]
trajs = [t for t in (contiguous(g) for _, g in ben_pool.groupby('sender_id'))
         if t is not None]
own = np.array([t.sender_id.iloc[0] for t in trajs])
print(f"{len(trajs)} contiguous benign trajectories\n")

# ---- mask: all benign test points, leave-one-vehicle-out ----
RADIUS, KQ, MINN = 15.0, 60, 3
bt_c = bt.dropna(subset=['s_x','s_y','s_hed']).copy()
XY, HD, VID = bt_c[['s_x','s_y']].values, bt_c.s_hed.values, bt_c.sender_id.values
nnh = NearestNeighbors(n_neighbors=KQ).fit(XY)

def probe(xy, hed, own_vid):
    d, idx = nnh.kneighbors(xy, n_neighbors=KQ)
    n_out = np.zeros(len(xy), int); dev = np.full(len(xy), np.nan)
    for i in range(len(xy)):
        near = idx[i][d[i] <= RADIUS]
        near = near[VID[near] != own_vid[i]]
        n_out[i] = len(near)
        if len(near) >= MINN:
            df_ = np.abs((HD[near] - hed[i]) % 360)
            dev[i] = np.minimum(df_, 360-df_).min()
    return n_out, dev

cal = bt_c.sample(min(20000, len(bt_c)), random_state=SEED)
n_c, dev_c = probe(cal[['s_x','s_y']].values, cal.s_hed.values, cal.sender_id.values)
occ_c = n_c >= MINN
T95 = np.percentile(dev_c[occ_c], 95)
OCC_FLOOR = 1 - occ_c.mean()
comb95 = OCC_FLOOR + occ_c.mean()*0.05

print("CALIBRATION")
print(f"  occupancy       {occ_c.mean():.4f}  -> position-limb floor {OCC_FLOOR:.4f}")
print(f"  bearing T95     {T95:.2f} deg   (paper says 6.13)")
print(f"  predicted EITHER benign rate  {comb95:.4f}   <-- is this the 11%?")

def three_limbs(df, own_vid):
    n, dv = probe(df[['s_x','s_y']].values, df.s_hed.values, own_vid)
    occ = n >= MINN
    bear = np.zeros(len(df), bool); bear[occ] = dv[occ] > T95
    return (~occ).mean(), (bear[occ].mean() if occ.any() else np.nan), \
           ((~occ) | bear).mean(), (np.nanmedian(dv[occ]) if occ.any() else np.nan)

print(f"\n{'row':28s} {'pos limb':>9s} {'bear limb':>10s} {'EITHER':>8s} {'dev|occ':>8s}")
for th in (0, 5, 10, 15, 30, 90):
    G = pd.concat([r.assign(_o=own[i]) for i, r in enumerate(rotate(t, float(th))
                   for t in trajs)], ignore_index=True).dropna(
                   subset=['s_x','s_y','s_hed'])
    p_, br, ei, dv = three_limbs(G, G._o.values)
    tag = f"rotation {th}deg" + ("  <-CONTROL" if th==0 else "")
    print(f"{tag:28s} {p_:9.3f} {br:10.3f} {ei:8.3f} {dv:8.2f}")

for a in ['constantPositionOffset','timeDelayAttack',
          'randomPositionOffset','dataReplay']:
    d = te[(te.attack_type==a) & (te.attacker==1)].dropna(
            subset=['s_x','s_y','s_hed'])
    if len(d) > 40000: d = d.sample(40000, random_state=SEED)
    p_, br, ei, dv = three_limbs(d, d.sender_id.values)
    print(f"{a:28s} {p_:9.3f} {br:10.3f} {ei:8.3f} {dv:8.2f}")

In [ ]:
# ============================================================
# T1  Proposition 1 as an exact prediction: transformed stream
#     scored against the SAME vehicle's untransformed stream.
# T2  Selection: do attacker vehicles differ from benign?
# ============================================================
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from scipy import stats

CACHE='/content/drive/MyDrive/aintec_cache'
tr=pd.read_parquet(f'{CACHE}/trR.parquet'); te=pd.read_parquet(f'{CACHE}/teR.parquet')
FEATS=["s_spd","s_acl","s_hed","disp","dt","implied_spd",
       "spd_resid","hed_resid","acl_resid","latency"]
RESID=["spd_resid","hed_resid","acl_resid"]
SEED=42; GAP_MAX=2.0
X=lambda d: d[FEATS].replace([np.inf,-np.inf],np.nan).fillna(0)
ok=tr[RESID].notna().all(axis=1)
rf=RandomForestClassifier(n_estimators=200,n_jobs=-1,random_state=SEED
   ).fit(X(tr[ok]),tr[ok].attacker)
bt=te[te.attacker==0]

def build_features(df, group_cols=("attack_type","sender_id")):
    g=list(group_cols)
    d=(df.sort_values(g+["sendTime_s"]).drop_duplicates(g+["sendTime_s"]).copy())
    gr=d.groupby(g); dt=gr.sendTime_s.diff(); dx,dy=gr.s_x.diff(),gr.s_y.diff()
    d["dt"],d["disp"]=dt,np.hypot(dx,dy)
    d["implied_spd"]=d.disp/dt.replace(0,np.nan)
    d["spd_resid"]=d.implied_spd-d.s_spd
    mh=(90-np.degrees(np.arctan2(dy,dx)))%360; hd=(mh-d.s_hed).abs()%360
    d["hed_resid"]=np.minimum(hd,360-hd)
    d["acl_resid"]=gr.s_spd.diff()/dt.replace(0,np.nan)-d.s_acl
    d["latency"]=d.rcvTime_s-d.sendTime_s
    d.loc[d.dt>GAP_MAX,RESID]=np.nan
    return d

def contiguous(g,n=12):
    g=g.sort_values('sendTime_s')
    if len(g)<n: return None
    return g.head(n) if g.sendTime_s.diff().iloc[1:n].between(0.8,1.2).all() else None

pool=te[(te.attacker==0)&(te.attack_type=='constantPositionOffset')]
trajs=[t for t in (contiguous(g) for _,g in pool.groupby('sender_id')) if t is not None]
print(f"{len(trajs)} trajectories\n")

# ---- T1: score T_{c,tau}(stream) against the same stream ----
def transform(t, c=(65.9,0.0), tau=0.0):
    d=t.copy().reset_index(drop=True)
    d['s_x']=d.s_x+c[0]; d['s_y']=d.s_y+c[1]
    d['sendTime_s']=d.sendTime_s+tau; d['rcvTime_s']=d.rcvTime_s+tau
    return d

for name,kw in [('translation |c|=65.9 m',dict(c=(65.9,0.0))),
                ('time shift tau=2.80 s',dict(c=(0.0,0.0),tau=2.80))]:
    A=build_features(pd.concat([t.assign(attack_type='u',sender_id=f'v{i}')
        for i,t in enumerate(trajs)],ignore_index=True))
    B=build_features(pd.concat([transform(t,**kw).assign(attack_type='t',sender_id=f'v{i}')
        for i,t in enumerate(trajs)],ignore_index=True))
    pa,pb=rf.predict_proba(X(A))[:,1],rf.predict_proba(X(B))[:,1]
    y=np.r_[np.zeros(len(pa)),np.ones(len(pb))]
    print(f"{name:24s} AUC vs own untransformed {roc_auc_score(y,np.r_[pa,pb]):.6f}"
          f"   max |score diff| {np.abs(pa-pb).max():.2e}")

# ---- T2: do attacker vehicles differ from benign on covariates? ----
print()
for a in ['constantPositionOffset','timeDelayAttack']:
    atk=te[(te.attack_type==a)&(te.attacker==1)]
    print(f"--- {a} ({atk.sender_id.nunique()} vehicles) ---")
    for c in ['s_spd','s_acl','dt','n_recv','sr_mean']:
        if c not in te.columns: continue
        av,bv=atk[c].dropna(),bt[c].dropna()
        if len(av)<50: continue
        u=stats.mannwhitneyu(av,bv,alternative='two-sided')
        pr=stats.mannwhitneyu(av,bv,alternative='less').statistic/(len(av)*len(bv))
        flag=" <-- differs" if abs(pr-0.5)>0.05 else ""
        print(f"  {c:9s} attack med {av.median():9.3f}  benign {bv.median():9.3f}"
              f"   P(a<b) {pr:.3f}   p={u.pvalue:.1e}{flag}")
    n=atk.groupby('sender_id').size()
    print(f"  msgs/vehicle {n.median():.0f} (benign {bt.groupby('sender_id').size().median():.0f})")

In [ ]:
print(f"{'attack':24s} {'covariate':9s} {'atk med':>9s} {'ben med':>9s} {'P(a<b)':>8s} {'p':>9s}")
for a in ['constantPositionOffset','timeDelayAttack']:
    sub = te[te.attack_type == a]                 # same subset for both sides
    atk, ben = sub[sub.attacker == 1], sub[sub.attacker == 0]
    for c in ['s_spd','s_acl','dt','n_recv','sr_mean','s_x','s_y']:
        if c not in te.columns: continue
        av, bv = atk[c].dropna(), ben[c].dropna()
        if len(av) < 50: continue
        pr = stats.mannwhitneyu(av, bv, alternative='less').statistic/(len(av)*len(bv))
        p  = stats.mannwhitneyu(av, bv, alternative='two-sided').pvalue
        print(f"{a:24s} {c:9s} {av.median():9.3f} {bv.median():9.3f} "
              f"{pr:8.3f} {p:9.1e}{'  <-- differs' if abs(pr-0.5)>0.03 else ''}")
    print(f"{'':24s} msgs/veh {len(atk)/atk.sender_id.nunique():.0f} vs "
          f"{len(ben)/ben.sender_id.nunique():.0f}\n")

In [ ]:
# ============================================================
# B1  Mask flag rate by offset BEARING -> defeat direction
# B2  Benign-neighbour density at attacker vs benign positions
#     -> explains the three below-control cells in Table 3
# Needs from the mask cell: trajs, own, three_limbs, probe, MINN,
#   nnh, XY, VID, build_features, X, rf, te, bt
# ============================================================
import numpy as np, pandas as pd

# ---- B1: hold |c| fixed, sweep bearing ----
def translate(t, mag, bearing):
    d=t.copy().reset_index(drop=True); r=np.radians(bearing)
    d['s_x']=d.s_x+mag*np.sin(r); d['s_y']=d.s_y+mag*np.cos(r)
    return d

# corridor axis: principal direction of benign travel
hd = bt.s_hed.dropna().values
axis = np.degrees(np.arctan2(np.sin(np.radians(hd)).mean(),
                             np.cos(np.radians(hd)).mean())) % 360
print(f"benign mean heading (corridor axis): {axis:.1f} deg\n")
print(f"{'bearing':>8s} {'rel.axis':>9s} {'pos limb':>9s} {'bearing':>8s} {'EITHER':>8s}")
rows=[]
for bd in range(0, 360, 15):
    G = pd.concat([translate(t, 65.9, bd).assign(
            attack_type=f'b{bd}', sender_id=f'v{i:03d}', _o=own[i])
          for i,t in enumerate(trajs)], ignore_index=True)
    Gc = G.dropna(subset=['s_x','s_y','s_hed'])
    p_, br, ei, _ = three_limbs(Gc, Gc._o.values)
    rel = min(abs(bd-axis), 360-abs(bd-axis))
    rows.append((bd, rel, p_, br, ei))
    print(f"{bd:8d} {rel:9.1f} {p_:9.3f} {br:8.3f} {ei:8.3f}")
D=pd.DataFrame(rows, columns=['bearing','rel_axis','pos','bear','either'])
D.to_csv('out/bearing_sweep.csv', index=False)
along = D[D.rel_axis < 30]; across = D[D.rel_axis > 60]
print(f"\nalong-corridor (<30 deg off axis): EITHER {along.either.mean():.3f}")
print(f"across-corridor (>60 deg off axis): EITHER {across.either.mean():.3f}")
print(f"aggregate over all bearings:        EITHER {D.either.mean():.3f}  (paper: 0.887)")

# ---- B2: neighbour density, attacker vs benign positions ----
def density(df):
    d,_ = nnh.kneighbors(df[['s_x','s_y']].values, n_neighbors=60)
    return (d <= 15.0).sum(axis=1)

print()
for a in ['constantPositionOffset','timeDelayAttack']:
    sub = te[te.attack_type==a]
    A = sub[(sub.attacker==1)].dropna(subset=['s_x','s_y'])
    B = sub[(sub.attacker==0)].dropna(subset=['s_x','s_y'])
    if len(A)>20000: A=A.sample(20000, random_state=42)
    if len(B)>20000: B=B.sample(20000, random_state=42)
    da, db = density(A), density(B)
    print(f"{a:24s} attacker nbrs med {np.median(da):5.1f}  "
          f"benign {np.median(db):5.1f}  frac<{MINN} {np.mean(da<MINN):.3f} vs "
          f"{np.mean(db<MINN):.3f}")

In [ ]:
# ============================================================
# Rebuild the mask machinery, then B1 (bearing) + B2 (density)
# Assumes te, bt, trajs already exist. Rebuilds everything else.
# ============================================================
import numpy as np, pandas as pd
from sklearn.neighbors import NearestNeighbors

SEED, RADIUS, KQ, MINN = 42, 15.0, 60, 3
own = np.array([t.sender_id.iloc[0] for t in trajs])
print(f"{len(trajs)} trajectories, {len(set(own))} distinct vehicles")

bt_c = bt.dropna(subset=['s_x','s_y','s_hed']).copy()
XY, HD, VID = bt_c[['s_x','s_y']].values, bt_c.s_hed.values, bt_c.sender_id.values
nnh = NearestNeighbors(n_neighbors=KQ).fit(XY)

def probe(xy, hed, own_vid):
    d, idx = nnh.kneighbors(xy, n_neighbors=KQ)
    n_out = np.zeros(len(xy), int); dev = np.full(len(xy), np.nan)
    for i in range(len(xy)):
        near = idx[i][d[i] <= RADIUS]
        near = near[VID[near] != own_vid[i]]
        n_out[i] = len(near)
        if len(near) >= MINN:
            df_ = np.abs((HD[near] - hed[i]) % 360)
            dev[i] = np.minimum(df_, 360 - df_).min()
    return n_out, dev

cal = bt_c.sample(min(20000, len(bt_c)), random_state=SEED)
n_c, dev_c = probe(cal[['s_x','s_y']].values, cal.s_hed.values, cal.sender_id.values)
occ_c = n_c >= MINN
T95 = np.percentile(dev_c[occ_c], 95)
print(f"occupancy {occ_c.mean():.4f}  T95 {T95:.2f} deg   (expect 0.9379 / 6.13)")

def three_limbs(df, own_vid):
    n, dv = probe(df[['s_x','s_y']].values, df.s_hed.values, own_vid)
    occ = n >= MINN
    bear = np.zeros(len(df), bool); bear[occ] = dv[occ] > T95
    return (~occ).mean(), (bear[occ].mean() if occ.any() else np.nan), ((~occ)|bear).mean()

def translate(t, mag, bearing):
    d = t.copy().reset_index(drop=True); r = np.radians(bearing)
    d['s_x'] = d.s_x + mag*np.sin(r); d['s_y'] = d.s_y + mag*np.cos(r)
    return d

# ---- B1: |c| = 65.9 m held fixed, bearing swept ----
hd = bt.s_hed.dropna().values
axis = np.degrees(np.arctan2(np.sin(np.radians(hd)).mean(),
                             np.cos(np.radians(hd)).mean())) % 360
print(f"\ncorridor axis {axis:.1f} deg (also {(axis+180)%360:.1f} for the other carriageway)\n")
print(f"{'bearing':>8s} {'rel.axis':>9s} {'pos':>7s} {'bear':>7s} {'EITHER':>8s}")
rows = []
for bd in range(0, 360, 15):
    G = pd.concat([translate(t, 65.9, bd).assign(_o=own[i])
                   for i, t in enumerate(trajs)], ignore_index=True)
    Gc = G.dropna(subset=['s_x','s_y','s_hed'])
    p_, br, ei = three_limbs(Gc, Gc._o.values)
    # distance to the nearer of the two carriageway directions
    rel = min(min(abs(bd-axis), 360-abs(bd-axis)),
              min(abs(bd-(axis+180)%360), 360-abs(bd-(axis+180)%360)))
    rows.append((bd, rel, p_, br, ei))
    print(f"{bd:8d} {rel:9.1f} {p_:7.3f} {br:7.3f} {ei:8.3f}")
D = pd.DataFrame(rows, columns=['bearing','rel_axis','pos','bear','either'])
D.to_csv('out/bearing_sweep.csv', index=False)
print(f"\nalong-corridor (<30 deg off either axis direction): EITHER {D[D.rel_axis<30].either.mean():.3f}")
print(f"across-corridor (>60 deg):                          EITHER {D[D.rel_axis>60].either.mean():.3f}")
print(f"aggregate over all bearings:                        EITHER {D.either.mean():.3f}   (paper 0.887)")

# ---- B2: neighbour density at attacker vs benign positions ----
def density(df):
    d, _ = nnh.kneighbors(df[['s_x','s_y']].values, n_neighbors=KQ)
    return (d <= RADIUS).sum(axis=1)

print()
for a in ['constantPositionOffset','timeDelayAttack']:
    sub = te[te.attack_type == a]
    A = sub[sub.attacker == 1].dropna(subset=['s_x','s_y'])
    B = sub[sub.attacker == 0].dropna(subset=['s_x','s_y'])
    if len(A) > 20000: A = A.sample(20000, random_state=SEED)
    if len(B) > 20000: B = B.sample(20000, random_state=SEED)
    da, db = density(A), density(B)
    print(f"{a:24s} nbrs med {np.median(da):5.1f} vs {np.median(db):5.1f}   "
          f"frac<{MINN}: {np.mean(da<MINN):.3f} vs {np.mean(db<MINN):.3f}")

In [ ]:
import os; os.makedirs('out', exist_ok=True)

def density(df):
    d, _ = nnh.kneighbors(df[['s_x','s_y']].values, n_neighbors=KQ)
    return (d <= RADIUS).sum(axis=1)

for a in ['constantPositionOffset','timeDelayAttack']:
    sub = te[te.attack_type == a]
    A = sub[sub.attacker == 1].dropna(subset=['s_x','s_y'])
    B = sub[sub.attacker == 0].dropna(subset=['s_x','s_y'])
    if len(A) > 20000: A = A.sample(20000, random_state=42)
    if len(B) > 20000: B = B.sample(20000, random_state=42)
    da, db = density(A), density(B)
    print(f"{a:24s} nbrs med {np.median(da):5.1f} vs {np.median(db):5.1f}   "
          f"frac<{MINN}: {np.mean(da<MINN):.3f} vs {np.mean(db<MINN):.3f}")


In [ ]:
def frac_unoccupied(df):
    n, _ = probe(df[['s_x','s_y']].values, df.s_hed.values, df.sender_id.values)
    return (n < MINN).mean(), np.median(n)

print(f"{'':26s} {'frac<3':>8s} {'nbrs med':>9s}")
for a in ['constantPositionOffset','timeDelayAttack']:
    sub = te[te.attack_type == a]
    for lbl, d in [('attacker', sub[sub.attacker==1]), ('benign, same subset', sub[sub.attacker==0])]:
        d = d.dropna(subset=['s_x','s_y','s_hed'])
        if len(d) > 20000: d = d.sample(20000, random_state=42)
        f, m = frac_unoccupied(d)
        print(f"{a[:14]:14s} {lbl:11s} {f:8.3f} {m:9.1f}")
print(f"\ncorridor calibration floor was {1-occ_c.mean():.3f}")

In [ ]:
hd = bt.s_hed.dropna().values
r = np.radians(hd)
R = np.hypot(np.sin(r).mean(), np.cos(r).mean())
print(f"circular concentration R = {R:.3f}  (1.0 = perfectly straight corridor)")
print(f"fraction within 15 deg of mean heading: "
      f"{np.mean(np.minimum(np.abs(hd-199.9), 360-np.abs(hd-199.9)) < 15):.3f}")

In [ ]:
# CIs for the mask columns. Reuses probe/three_limbs/T95/MINN from the mask cell.
import numpy as np, pandas as pd

def limbs_vec(df, own_vid):
    n, dv = probe(df[['s_x','s_y']].values, df.s_hed.values, own_vid)
    occ = n >= MINN
    bear = np.zeros(len(df), bool); bear[occ] = dv[occ] > T95
    return (~occ).astype(int), bear.astype(int), ((~occ)|bear).astype(int)

def ci(x, n=2000, seed=42):
    r = np.random.default_rng(seed)
    bs = [r.choice(x, len(x), replace=True).mean() for _ in range(n)]
    return np.percentile(bs, [2.5, 97.5])

rows = [('constantPositionOffset', te[(te.attack_type=='constantPositionOffset')&(te.attacker==1)]),
        ('timeDelayAttack',        te[(te.attack_type=='timeDelayAttack')&(te.attacker==1)]),
        ('dataReplay',             te[(te.attack_type=='dataReplay')&(te.attacker==1)]),
        ('control, corridor',      bt)]
print(f"{'row':24s} {'pos [95% CI]':>22s} {'bearing':>22s} {'either':>22s}")
for name, d in rows:
    d = d.dropna(subset=['s_x','s_y','s_hed'])
    if len(d) > 20000: d = d.sample(20000, random_state=42)
    p, b, e = limbs_vec(d, d.sender_id.values)
    out = f"{name:24s}"
    for v in (p, b, e):
        lo, hi = ci(v)
        out += f"  {v.mean():.3f} [{lo:.3f},{hi:.3f}]"
    print(out)

# rotation at 15 deg, over the 262 trajectories
G = pd.concat([rotate(t, 15.0).assign(_o=own[i]) for i, t in enumerate(trajs)],
              ignore_index=True).dropna(subset=['s_x','s_y','s_hed'])
p, b, e = limbs_vec(G, G._o.values)
out = f"{'rotation, 15 deg':24s}"
for v in (p, b, e):
    lo, hi = ci(v); out += f"  {v.mean():.3f} [{lo:.3f},{hi:.3f}]"
print(out)

In [ ]:
# ============================================================
# Full rebuild from cache, then bootstrap CIs for Table 3
# ============================================================
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors

CACHE = '/content/drive/MyDrive/aintec_cache'
tr = pd.read_parquet(f'{CACHE}/trR.parquet')
te = pd.read_parquet(f'{CACHE}/teR.parquet')

SEED, GAP_MAX, RADIUS, KQ, MINN = 42, 2.0, 15.0, 60, 3
FEATS = ["s_spd","s_acl","s_hed","disp","dt","implied_spd",
         "spd_resid","hed_resid","acl_resid","latency"]
RESID = ["spd_resid","hed_resid","acl_resid"]
X = lambda d: d[FEATS].replace([np.inf,-np.inf], np.nan).fillna(0)
bt = te[te.attacker == 0]

def build_features(df, group_cols=("attack_type","sender_id")):
    g = list(group_cols)
    d = (df.sort_values(g+["sendTime_s"]).drop_duplicates(g+["sendTime_s"]).copy())
    gr = d.groupby(g); dt = gr.sendTime_s.diff(); dx, dy = gr.s_x.diff(), gr.s_y.diff()
    d["dt"], d["disp"] = dt, np.hypot(dx, dy)
    d["implied_spd"] = d.disp / dt.replace(0, np.nan)
    d["spd_resid"] = d.implied_spd - d.s_spd
    mh = (90 - np.degrees(np.arctan2(dy, dx))) % 360
    hd = (mh - d.s_hed).abs() % 360
    d["hed_resid"] = np.minimum(hd, 360-hd)
    d["acl_resid"] = gr.s_spd.diff()/dt.replace(0, np.nan) - d.s_acl
    d["latency"] = d.rcvTime_s - d.sendTime_s
    d.loc[d.dt > GAP_MAX, RESID] = np.nan
    return d

def rotate(traj, th):
    d = traj.copy().reset_index(drop=True); t = np.radians(th)
    x0, y0 = d.s_x.iloc[0], d.s_y.iloc[0]
    dx, dy = d.s_x-x0, d.s_y-y0
    d['s_x'] = x0 + dx*np.cos(t) - dy*np.sin(t)
    d['s_y'] = y0 + dx*np.sin(t) + dy*np.cos(t)
    d['s_hed'] = (d.s_hed - th) % 360
    return d

def contiguous(g, n=12):
    g = g.sort_values('sendTime_s')
    if len(g) < n: return None
    return g.head(n) if g.sendTime_s.diff().iloc[1:n].between(0.8,1.2).all() else None

ok = tr[RESID].notna().all(axis=1)
rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED).fit(X(tr[ok]), tr[ok].attacker)
print(f"rf sanity: precision {(rf.predict(X(te))==te.attacker).mean():.3f}  "
      f"benign flag {rf.predict(X(te))[(te.attacker==0).values].mean():.4f}  (expect ~0.828 / 0.046)")

ben_pool = te[(te.attacker==0) & (te.attack_type=='constantPositionOffset')]
trajs = [t for t in (contiguous(g) for _, g in ben_pool.groupby('sender_id')) if t is not None]
own = np.array([t.sender_id.iloc[0] for t in trajs])
print(f"{len(trajs)} trajectories")

bt_c = bt.dropna(subset=['s_x','s_y','s_hed']).copy()
XY, HD, VID = bt_c[['s_x','s_y']].values, bt_c.s_hed.values, bt_c.sender_id.values
nnh = NearestNeighbors(n_neighbors=KQ).fit(XY)

def probe(xy, hed, own_vid):
    d, idx = nnh.kneighbors(xy, n_neighbors=KQ)
    n_out = np.zeros(len(xy), int); dev = np.full(len(xy), np.nan)
    for i in range(len(xy)):
        near = idx[i][d[i] <= RADIUS]
        near = near[VID[near] != own_vid[i]]
        n_out[i] = len(near)
        if len(near) >= MINN:
            df_ = np.abs((HD[near] - hed[i]) % 360)
            dev[i] = np.minimum(df_, 360-df_).min()
    return n_out, dev

cal = bt_c.sample(min(20000, len(bt_c)), random_state=SEED)
n_c, dev_c = probe(cal[['s_x','s_y']].values, cal.s_hed.values, cal.sender_id.values)
occ_c = n_c >= MINN
T95 = np.percentile(dev_c[occ_c], 95)
print(f"occupancy {occ_c.mean():.4f}  T95 {T95:.2f} deg  (expect 0.9379 / 6.13)")

# ---- now the CI code ----
def limbs_vec(df, own_vid):
    n, dv = probe(df[['s_x','s_y']].values, df.s_hed.values, own_vid)
    occ = n >= MINN
    bear = np.zeros(len(df), bool); bear[occ] = dv[occ] > T95
    return (~occ).astype(int), bear.astype(int), ((~occ)|bear).astype(int)

def ci(x, n=2000, seed=42):
    r = np.random.default_rng(seed)
    bs = [r.choice(x, len(x), replace=True).mean() for _ in range(n)]
    return np.percentile(bs, [2.5, 97.5])

rows = [('constantPositionOffset', te[(te.attack_type=='constantPositionOffset')&(te.attacker==1)]),
        ('timeDelayAttack',        te[(te.attack_type=='timeDelayAttack')&(te.attacker==1)]),
        ('dataReplay',             te[(te.attack_type=='dataReplay')&(te.attacker==1)]),
        ('control, corridor',      bt)]
print(f"\n{'row':24s} {'pos [95% CI]':>26s} {'bearing':>26s} {'either':>26s}")
for name, d in rows:
    d = d.dropna(subset=['s_x','s_y','s_hed'])
    if len(d) > 20000: d = d.sample(20000, random_state=42)
    p, b, e = limbs_vec(d, d.sender_id.values)
    out = f"{name:24s}"
    for v in (p, b, e):
        lo, hi = ci(v)
        out += f"  {v.mean():.3f} [{lo:.3f},{hi:.3f}]"
    print(out)

G = pd.concat([rotate(t, 15.0).assign(_o=own[i]) for i, t in enumerate(trajs)],
              ignore_index=True).dropna(subset=['s_x','s_y','s_hed'])
p, b, e = limbs_vec(G, G._o.values)
out = f"{'rotation, 15 deg':24s}"
for v in (p, b, e):
    lo, hi = ci(v); out += f"  {v.mean():.3f} [{lo:.3f},{hi:.3f}]"
print(out)

In [ ]:
def limbs_vec_fixed(df, own_vid):
    n, dv = probe(df[['s_x','s_y']].values, df.s_hed.values, own_vid)
    occ = n >= MINN
    bear_occ = (dv[occ] > T95).astype(int)   # only occupied points
    pos = (~occ).astype(int)
    either = ((~occ) | ((n>=MINN)&(dv>T95))).astype(int)
    return pos, bear_occ, either  # note: bear_occ has different length than pos/either

In [ ]:
def ci(x, n=2000, seed=42):
    r = np.random.default_rng(seed)
    bs = [r.choice(x, len(x), replace=True).mean() for _ in range(n)]
    return np.percentile(bs, [2.5, 97.5])

rows = [('constantPositionOffset', te[(te.attack_type=='constantPositionOffset')&(te.attacker==1)]),
        ('timeDelayAttack',        te[(te.attack_type=='timeDelayAttack')&(te.attacker==1)]),
        ('dataReplay',             te[(te.attack_type=='dataReplay')&(te.attacker==1)]),
        ('control, corridor',      bt)]

print(f"{'row':24s} {'pos [CI]':>22s} {'bearing|occ [CI]':>26s} {'either [CI]':>22s}")
for name, d in rows:
    d = d.dropna(subset=['s_x','s_y','s_hed'])
    if len(d) > 20000: d = d.sample(20000, random_state=42)
    pos, bear_occ, either = limbs_vec_fixed(d, d.sender_id.values)
    p_lo, p_hi = ci(pos)
    b_lo, b_hi = ci(bear_occ)     # bootstrapped over occupied points only — different n
    e_lo, e_hi = ci(either)
    print(f"{name:24s}  {pos.mean():.3f} [{p_lo:.3f},{p_hi:.3f}]"
          f"  {bear_occ.mean():.3f} [{b_lo:.3f},{b_hi:.3f}]"
          f"  {either.mean():.3f} [{e_lo:.3f},{e_hi:.3f}]")

G = pd.concat([rotate(t, 15.0).assign(_o=own[i]) for i, t in enumerate(trajs)],
              ignore_index=True).dropna(subset=['s_x','s_y','s_hed'])
pos, bear_occ, either = limbs_vec_fixed(G, G._o.values)
p_lo, p_hi = ci(pos); b_lo, b_hi = ci(bear_occ); e_lo, e_hi = ci(either)
print(f"{'rotation, 15 deg':24s}  {pos.mean():.3f} [{p_lo:.3f},{p_hi:.3f}]"
      f"  {bear_occ.mean():.3f} [{b_lo:.3f},{b_hi:.3f}]"
      f"  {either.mean():.3f} [{e_lo:.3f},{e_hi:.3f}]")

In [ ]:
# ============================================================
# Overlapping-split test: does absolute position help the
# kernel attacks once the geographic-split confound is removed?
# ============================================================
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

CACHE = '/content/drive/MyDrive/aintec_cache'
tr = pd.read_parquet(f'{CACHE}/trR.parquet')
te = pd.read_parquet(f'{CACHE}/teR.parquet')
FEATS = ["s_spd","s_acl","s_hed","disp","dt","implied_spd",
         "spd_resid","hed_resid","acl_resid","latency"]
FEATS_POS = FEATS + ["s_x","s_y"]
RESID = ["spd_resid","hed_resid","acl_resid"]
SEED = 42
X = lambda d, f=FEATS: d[f].replace([np.inf,-np.inf], np.nan).fillna(0)

pool = pd.concat([tr, te], ignore_index=True)
pool['vkey'] = pool.attack_type + '|' + pool.sender_id.astype(str)
keys = pool.vkey.unique()
rng = np.random.default_rng(SEED)
half = set(rng.permutation(keys)[:len(keys)//2])
A, B = pool[pool.vkey.isin(half)], pool[~pool.vkey.isin(half)]
print(f"random split by vehicle: {len(A):,} / {len(B):,}")

bA, bB = A[A.attacker==0], B[B.attacker==0]
print(f"geographic overlap after random split: x range "
      f"{min(bA.s_x.max(),bB.s_x.max())-max(bA.s_x.min(),bB.s_x.min()):.0f} m")

for tag, feats in [('no position', FEATS), ('WITH position', FEATS_POS)]:
    okA = A[RESID].notna().all(axis=1)
    m = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)
    m.fit(X(A[okA], feats), A[okA].attacker)
    p = m.predict_proba(X(B, feats))[:,1]
    bm = (B.attacker==0).values
    print(f"\n[{tag}] benign flag {p[bm].mean():.4f}  (was 0.046/0.135 under geo split)")
    for a in ['constantPositionOffset','timeDelayAttack','randomPositionOffset','dataReplay']:
        mk = ((B.attack_type==a) & (B.attacker==1)).values
        if not mk.sum(): continue
        y = np.r_[np.zeros(bm.sum()), np.ones(mk.sum())]
        print(f"    {a:26s} AUC {roc_auc_score(y, np.r_[p[bm], p[mk]]):.3f}")

In [ ]:
# Reuses A, B, FEATS_POS, X from the overlapping-split cell, and
# rotate/trajs/own from the mask cell. Rebuild session state first if needed.
okA = A[RESID].notna().all(axis=1)
m_pos = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
m_pos.fit(X(A[okA], FEATS_POS), A[okA].attacker)

def rotate_pos(traj, th):
    d = rotate(traj, th)  # existing rotate() from the mask cell
    return d

G = pd.concat([rotate_pos(t, 15.0).assign(attack_type='rot', sender_id=f'v{i}')
               for i, t in enumerate(trajs)], ignore_index=True)
Gf = build_features(G)  # needs s_x, s_y already present post-rotation
core = (Gf.groupby(['attack_type','sender_id']).cumcount() > 0).values
p = m_pos.predict_proba(X(Gf, FEATS_POS))[:, 1]
pred = m_pos.predict(X(Gf, FEATS_POS))
print(f"learned position prior, RF, rotation 15deg: "
      f"recall {pred[core].mean():.3f}")

In [ ]:
# ============================================================
# Full rebuild + overlapping-split position-aware detector vs rotation
# ============================================================
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

CACHE = '/content/drive/MyDrive/aintec_cache'
tr = pd.read_parquet(f'{CACHE}/trR.parquet')
te = pd.read_parquet(f'{CACHE}/teR.parquet')

SEED, GAP_MAX = 42, 2.0
FEATS = ["s_spd","s_acl","s_hed","disp","dt","implied_spd",
         "spd_resid","hed_resid","acl_resid","latency"]
FEATS_POS = FEATS + ["s_x","s_y"]
RESID = ["spd_resid","hed_resid","acl_resid"]
X = lambda d, f=FEATS: d[f].replace([np.inf,-np.inf], np.nan).fillna(0)
bt = te[te.attacker == 0]

def build_features(df, group_cols=("attack_type","sender_id")):
    g = list(group_cols)
    d = (df.sort_values(g+["sendTime_s"]).drop_duplicates(g+["sendTime_s"]).copy())
    gr = d.groupby(g); dt = gr.sendTime_s.diff(); dx, dy = gr.s_x.diff(), gr.s_y.diff()
    d["dt"], d["disp"] = dt, np.hypot(dx, dy)
    d["implied_spd"] = d.disp / dt.replace(0, np.nan)
    d["spd_resid"] = d.implied_spd - d.s_spd
    mh = (90 - np.degrees(np.arctan2(dy, dx))) % 360
    hd = (mh - d.s_hed).abs() % 360
    d["hed_resid"] = np.minimum(hd, 360-hd)
    d["acl_resid"] = gr.s_spd.diff()/dt.replace(0, np.nan) - d.s_acl
    d["latency"] = d.rcvTime_s - d.sendTime_s
    d.loc[d.dt > GAP_MAX, RESID] = np.nan
    return d

def rotate(traj, th):
    d = traj.copy().reset_index(drop=True); t = np.radians(th)
    x0, y0 = d.s_x.iloc[0], d.s_y.iloc[0]
    dx, dy = d.s_x-x0, d.s_y-y0
    d['s_x'] = x0 + dx*np.cos(t) - dy*np.sin(t)
    d['s_y'] = y0 + dx*np.sin(t) + dy*np.cos(t)
    d['s_hed'] = (d.s_hed - th) % 360
    return d

def contiguous(g, n=12):
    g = g.sort_values('sendTime_s')
    if len(g) < n: return None
    return g.head(n) if g.sendTime_s.diff().iloc[1:n].between(0.8,1.2).all() else None

# ---- rebuild the overlapping (random vehicle-level) split ----
pool = pd.concat([tr, te], ignore_index=True)
pool['vkey'] = pool.attack_type + '|' + pool.sender_id.astype(str)
keys = pool.vkey.unique()
rng = np.random.default_rng(SEED)
half = set(rng.permutation(keys)[:len(keys)//2])
A, B = pool[pool.vkey.isin(half)], pool[~pool.vkey.isin(half)]
print(f"random split by vehicle: {len(A):,} / {len(B):,}")

okA = A[RESID].notna().all(axis=1)
m_pos = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)
m_pos.fit(X(A[okA], FEATS_POS), A[okA].attacker)

# sanity: reproduce the known result before trusting the rebuild
bm = (B.attacker==0).values
p_sanity = m_pos.predict_proba(X(B, FEATS_POS))[:,1]
print("\n--- sanity check against known result ---")
for a in ['constantPositionOffset','timeDelayAttack']:
    mk = ((B.attack_type==a)&(B.attacker==1)).values
    y = np.r_[np.zeros(bm.sum()), np.ones(mk.sum())]
    auc = roc_auc_score(y, np.r_[p_sanity[bm], p_sanity[mk]])
    print(f"{a:24s} AUC {auc:.3f}  (expect 0.828 / 0.500)")

# ---- rotation trajectories ----
ben_pool = te[(te.attacker==0) & (te.attack_type=='constantPositionOffset')]
trajs = [t for t in (contiguous(g) for _, g in ben_pool.groupby('sender_id')) if t is not None]
own = np.array([t.sender_id.iloc[0] for t in trajs])
print(f"\n{len(trajs)} trajectories")

# control at theta=0
G0 = pd.concat([t.assign(attack_type='ctl', sender_id=f'v{i}') for i,t in enumerate(trajs)],
               ignore_index=True)
Gf0 = build_features(G0)
core0 = (Gf0.groupby(['attack_type','sender_id']).cumcount() > 0).values
pred0 = m_pos.predict(X(Gf0, FEATS_POS))
print(f"learned position prior, control (theta=0): recall {pred0[core0].mean():.4f}")

# the new test: rotation 15 deg
G = pd.concat([rotate(t, 15.0).assign(attack_type='rot', sender_id=f'v{i}')
               for i, t in enumerate(trajs)], ignore_index=True)
Gf = build_features(G)
core = (Gf.groupby(['attack_type','sender_id']).cumcount() > 0).values
pred = m_pos.predict(X(Gf, FEATS_POS))
print(f"learned position prior, RF, rotation 15deg: recall {pred[core].mean():.4f}")

In [ ]:
# Reuses A, B, FEATS_POS, X from the overlapping-split cell, and
# rotate/trajs/own from the mask cell. Rebuild session state first if needed.
okA = A[RESID].notna().all(axis=1)
m_pos = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
m_pos.fit(X(A[okA], FEATS_POS), A[okA].attacker)

def rotate_pos(traj, th):
    d = rotate(traj, th)  # existing rotate() from the mask cell
    return d

G = pd.concat([rotate_pos(t, 15.0).assign(attack_type='rot', sender_id=f'v{i}')
               for i, t in enumerate(trajs)], ignore_index=True)
Gf = build_features(G)  # needs s_x, s_y already present post-rotation
core = (Gf.groupby(['attack_type','sender_id']).cumcount() > 0).values
p = m_pos.predict_proba(X(Gf, FEATS_POS))[:, 1]
pred = m_pos.predict(X(Gf, FEATS_POS))
print(f"learned position prior, RF, rotation 15deg: "
      f"recall {pred[core].mean():.3f}")

In [ ]:
import requests
r = requests.get("https://zenodo.org/api/records/19665762")
data = r.json()
files = [f['key'] for f in data['files']]
densities = sorted(set(f.split('_constantPositionOffset')[0] for f in files
                        if 'constantPositionOffset' in f))
print("Available scenario/density prefixes:")
for d in densities:
    print(" ", d)
print(f"\ntotal files in record: {len(files)}")

In [ ]:
import os, zipfile

SCENARIO2 = "InTAS_urban_2"
BASE = "https://zenodo.org/records/19665762/files"
p = f"data/{SCENARIO2}_constantPositionOffset.zip"
os.makedirs("data", exist_ok=True)
if not (os.path.exists(p) and zipfile.is_zipfile(p)):
    if os.path.exists(p): os.remove(p)
    !wget -q --tries=3 "{BASE}/{SCENARIO2}_constantPositionOffset.zip?download=1" -O "{p}"
assert zipfile.is_zipfile(p), "download failed"

outer = zipfile.ZipFile(p)
print("top-level entries:", outer.namelist()[:5])
inner_candidates = [n for n in outer.namelist() if "/Test/" in n and n.endswith(".zip")]
print("inner Test zip found:", inner_candidates[:2])

In [ ]:
SCENARIO = "InTAS_urban_2"
# download all 15 attack subsets, same as your original setup
for a in ATTACKS:
    p = f"data/{SCENARIO}_{a}.zip"
    if not (os.path.exists(p) and zipfile.is_zipfile(p)):
        if os.path.exists(p): os.remove(p)
        !wget -q --tries=3 "{BASE}/{SCENARIO}_{a}.zip?download=1" -O "{p}"
    assert zipfile.is_zipfile(p), f"download failed: {a}"

tr2 = build_features(load_split("Train"))
te2 = build_features(load_split("Test"))
ok2 = tr2[RESID].notna().all(axis=1)

rf2 = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
rf2.fit(X(tr2[ok2]), tr2[ok2].attacker)
p2 = rf2.predict_proba(X(te2))[:,1]
bm2 = (te2.attacker == 0).values

for a in ['constantPositionOffset', 'timeDelayAttack']:
    m = ((te2.attack_type==a) & (te2.attacker==1)).values
    y = np.r_[np.zeros(bm2.sum()), np.ones(m.sum())]
    print(f"{a:24s} AUC {roc_auc_score(y, np.r_[p2[bm2], p2[m]]):.3f}")

print(f"\nbenign flag rate: {rf2.predict(X(te2))[bm2].mean():.4f}  (highway_2 was 0.046)")

In [ ]:
# ============================================================
# InTAS_urban_2 — self-contained: schema check + full pipeline
# ============================================================
import os, io, json, zipfile, gc
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

SCENARIO = "InTAS_urban_2"
BASE     = "https://zenodo.org/records/19665762/files"
GAP_MAX  = 2.0
ATTACKS = '''accelerationMultiplication constantPositionOffset constantSpeedOffset
dataReplay dosAttack feignedBraking positionMirroring randomPositionOffset
randomSpeedOffset reversedHeading suddenConstantSpeed suddenStop
timeDelayAttack trafficCongestionSybil zeroSpeedReport'''.split()
FEATS = ["s_spd","s_acl","s_hed","disp","dt","implied_spd",
         "spd_resid","hed_resid","acl_resid","latency"]
RESID = ["spd_resid","hed_resid","acl_resid"]
X = lambda d: d[FEATS].replace([np.inf,-np.inf], np.nan).fillna(0)

os.makedirs("data", exist_ok=True)

def num(v, idx=None):
    if v is None: return None
    if idx is not None:
        try: return float(str(v).split(",")[idx])
        except (ValueError, IndexError): return None
    try: return float(v)
    except (ValueError, TypeError): return None

def ensure(scenario, attack):
    p = f"data/{scenario}_{attack}.zip"
    if not (os.path.exists(p) and zipfile.is_zipfile(p)):
        if os.path.exists(p): os.remove(p)
        !wget -q --tries=3 "{BASE}/{scenario}_{attack}.zip?download=1" -O "{p}"
    assert zipfile.is_zipfile(p), f"download failed: {scenario}_{attack}"
    return p

def load(zip_path, split, label):
    outer = zipfile.ZipFile(zip_path)
    inner_candidates = [n for n in outer.namelist() if f"/{split}/" in n and n.endswith(".zip")]
    inner = zipfile.ZipFile(io.BytesIO(outer.read(inner_candidates[0])))
    rows = []
    for name in inner.namelist():
        if not name.endswith(".json"): continue
        for m in json.loads(inner.read(name)):
            s, r = m.get("sender", {}), m.get("receiver", {})
            rows.append({
                "rcvTime_s": (num(m.get("rcvTime")) or 0)/1e9,
                "sendTime_s": (num(m.get("sendTime")) or 0)/1e9,
                "sender_id":  m.get("sender_id"),
                "attacker":   int(m.get("attacker", 0)),
                "attack_type": label,
                "s_x": num(s.get("pos"),0), "s_y": num(s.get("pos"),1),
                "s_spd": num(s.get("spd")), "s_acl": num(s.get("acl")),
                "s_hed": num(s.get("hed")),
                "r_x": num(r.get("pos"),0), "r_y": num(r.get("pos"),1)})
    del outer, inner; gc.collect()
    return pd.DataFrame(rows)

def load_split(scenario, split):
    return pd.concat([load(ensure(scenario, a), split, a) for a in ATTACKS],
                      ignore_index=True)

def build_features(df, group_cols=("attack_type","sender_id")):
    g = list(group_cols)
    d = (df.sort_values(g+["sendTime_s"]).drop_duplicates(g+["sendTime_s"]).copy())
    gr = d.groupby(g); dt = gr.sendTime_s.diff(); dx, dy = gr.s_x.diff(), gr.s_y.diff()
    d["dt"], d["disp"] = dt, np.hypot(dx, dy)
    d["implied_spd"] = d.disp / dt.replace(0, np.nan)
    d["spd_resid"] = d.implied_spd - d.s_spd
    mh = (90 - np.degrees(np.arctan2(dy, dx))) % 360
    hd = (mh - d.s_hed).abs() % 360
    d["hed_resid"] = np.minimum(hd, 360-hd)
    d["acl_resid"] = gr.s_spd.diff()/dt.replace(0, np.nan) - d.s_acl
    d["latency"] = d.rcvTime_s - d.sendTime_s
    d.loc[d.dt > GAP_MAX, RESID] = np.nan
    return d

# ---- schema check first ----
p0 = ensure(SCENARIO, 'constantPositionOffset')
outer0 = zipfile.ZipFile(p0)
print("top-level entries:", outer0.namelist()[:5])
inner0 = [n for n in outer0.namelist() if "/Test/" in n and n.endswith(".zip")]
print("inner Test zip found:", inner0[:2])
assert inner0, "schema mismatch: no /Test/ inner zip found — stop and report this"
print("schema OK, proceeding to full build\n")

# ---- full pipeline ----
for a in ATTACKS:
    ensure(SCENARIO, a)
print(f"{len(ATTACKS)} subsets verified\n")

tr2 = build_features(load_split(SCENARIO, "Train"))
te2 = build_features(load_split(SCENARIO, "Test"))
gc.collect()
ok2 = tr2[RESID].notna().all(axis=1)

print(f"train {len(tr2):,}  test {len(te2):,}")
b2 = tr2[tr2.attacker==0]
print(f"benign medians: dt {b2.dt.median():.3f}s  spd_resid {b2.spd_resid.abs().median():.4f}  "
      f"hed_resid {b2.hed_resid.abs().median():.3f}deg\n")

rf2 = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
rf2.fit(X(tr2[ok2]), tr2[ok2].attacker)
p2 = rf2.predict_proba(X(te2))[:,1]
pred2 = rf2.predict(X(te2))
bm2 = (te2.attacker == 0).values

print(f"precision {precision_score(te2.attacker, pred2):.3f}  "
      f"benign flag {pred2[bm2].mean():.4f}   (highway_2 was 0.828 / 0.046)\n")

for a in ['constantPositionOffset', 'timeDelayAttack']:
    m = ((te2.attack_type==a) & (te2.attacker==1)).values
    y = np.r_[np.zeros(bm2.sum()), np.ones(m.sum())]
    print(f"{a:24s} AUC {roc_auc_score(y, np.r_[p2[bm2], p2[m]]):.3f}   "
          f"(highway_2: 0.498 / 0.490)")

In [ ]:
import os
bad = "data/InTAS_urban_2_accelerationMultiplication.zip"
if os.path.exists(bad):
    os.remove(bad)
    print("removed partial file")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score

In [ ]:
# ============================================================
# InTAS_urban_2 — self-contained: schema check + full pipeline
# ============================================================
import os, io, json, zipfile, gc
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

SCENARIO = "InTAS_urban_2"
BASE     = "https://zenodo.org/records/19665762/files"
GAP_MAX  = 2.0
ATTACKS = '''accelerationMultiplication constantPositionOffset constantSpeedOffset
dataReplay dosAttack feignedBraking positionMirroring randomPositionOffset
randomSpeedOffset reversedHeading suddenConstantSpeed suddenStop
timeDelayAttack trafficCongestionSybil zeroSpeedReport'''.split()
FEATS = ["s_spd","s_acl","s_hed","disp","dt","implied_spd",
         "spd_resid","hed_resid","acl_resid","latency"]
RESID = ["spd_resid","hed_resid","acl_resid"]
X = lambda d: d[FEATS].replace([np.inf,-np.inf], np.nan).fillna(0)

os.makedirs("data", exist_ok=True)

def num(v, idx=None):
    if v is None: return None
    if idx is not None:
        try: return float(str(v).split(",")[idx])
        except (ValueError, IndexError): return None
    try: return float(v)
    except (ValueError, TypeError): return None

def ensure(scenario, attack):
    p = f"data/{scenario}_{attack}.zip"
    if not (os.path.exists(p) and zipfile.is_zipfile(p)):
        if os.path.exists(p): os.remove(p)
        !wget -q --tries=3 "{BASE}/{scenario}_{attack}.zip?download=1" -O "{p}"
    assert zipfile.is_zipfile(p), f"download failed: {scenario}_{attack}"
    return p

def load(zip_path, split, label):
    outer = zipfile.ZipFile(zip_path)
    inner_candidates = [n for n in outer.namelist() if f"/{split}/" in n and n.endswith(".zip")]
    inner = zipfile.ZipFile(io.BytesIO(outer.read(inner_candidates[0])))
    rows = []
    for name in inner.namelist():
        if not name.endswith(".json"): continue
        for m in json.loads(inner.read(name)):
            s, r = m.get("sender", {}), m.get("receiver", {})
            rows.append({
                "rcvTime_s": (num(m.get("rcvTime")) or 0)/1e9,
                "sendTime_s": (num(m.get("sendTime")) or 0)/1e9,
                "sender_id":  m.get("sender_id"),
                "attacker":   int(m.get("attacker", 0)),
                "attack_type": label,
                "s_x": num(s.get("pos"),0), "s_y": num(s.get("pos"),1),
                "s_spd": num(s.get("spd")), "s_acl": num(s.get("acl")),
                "s_hed": num(s.get("hed")),
                "r_x": num(r.get("pos"),0), "r_y": num(r.get("pos"),1)})
    del outer, inner; gc.collect()
    return pd.DataFrame(rows)

def load_split(scenario, split):
    return pd.concat([load(ensure(scenario, a), split, a) for a in ATTACKS],
                      ignore_index=True)

def build_features(df, group_cols=("attack_type","sender_id")):
    g = list(group_cols)
    d = (df.sort_values(g+["sendTime_s"]).drop_duplicates(g+["sendTime_s"]).copy())
    gr = d.groupby(g); dt = gr.sendTime_s.diff(); dx, dy = gr.s_x.diff(), gr.s_y.diff()
    d["dt"], d["disp"] = dt, np.hypot(dx, dy)
    d["implied_spd"] = d.disp / dt.replace(0, np.nan)
    d["spd_resid"] = d.implied_spd - d.s_spd
    mh = (90 - np.degrees(np.arctan2(dy, dx))) % 360
    hd = (mh - d.s_hed).abs() % 360
    d["hed_resid"] = np.minimum(hd, 360-hd)
    d["acl_resid"] = gr.s_spd.diff()/dt.replace(0, np.nan) - d.s_acl
    d["latency"] = d.rcvTime_s - d.sendTime_s
    d.loc[d.dt > GAP_MAX, RESID] = np.nan
    return d

# ---- schema check first ----
p0 = ensure(SCENARIO, 'constantPositionOffset')
outer0 = zipfile.ZipFile(p0)
print("top-level entries:", outer0.namelist()[:5])
inner0 = [n for n in outer0.namelist() if "/Test/" in n and n.endswith(".zip")]
print("inner Test zip found:", inner0[:2])
assert inner0, "schema mismatch: no /Test/ inner zip found — stop and report this"
print("schema OK, proceeding to full build\n")

# ---- full pipeline ----
for a in ATTACKS:
    ensure(SCENARIO, a)
print(f"{len(ATTACKS)} subsets verified\n")

tr2 = build_features(load_split(SCENARIO, "Train"))
te2 = build_features(load_split(SCENARIO, "Test"))
gc.collect()
ok2 = tr2[RESID].notna().all(axis=1)

print(f"train {len(tr2):,}  test {len(te2):,}")
b2 = tr2[tr2.attacker==0]
print(f"benign medians: dt {b2.dt.median():.3f}s  spd_resid {b2.spd_resid.abs().median():.4f}  "
      f"hed_resid {b2.hed_resid.abs().median():.3f}deg\n")

rf2 = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
rf2.fit(X(tr2[ok2]), tr2[ok2].attacker)
p2 = rf2.predict_proba(X(te2))[:,1]
pred2 = rf2.predict(X(te2))
bm2 = (te2.attacker == 0).values

print(f"precision {precision_score(te2.attacker, pred2):.3f}  "
      f"benign flag {pred2[bm2].mean():.4f}   (highway_2 was 0.828 / 0.046)\n")

for a in ['constantPositionOffset', 'timeDelayAttack']:
    m = ((te2.attack_type==a) & (te2.attacker==1)).values
    y = np.r_[np.zeros(bm2.sum()), np.ones(m.sum())]
    print(f"{a:24s} AUC {roc_auc_score(y, np.r_[p2[bm2], p2[m]]):.3f}   "
          f"(highway_2: 0.498 / 0.490)")

In [ ]:
# Check whether the same physical sender_id's benign trajectory
# is duplicated across different attack_type subset folders
sample_vid = pool[pool.attacker==0].sender_id.value_counts().index[0]
subs = pool[(pool.sender_id==sample_vid) & (pool.attacker==0)]
print(f"'{sample_vid}' appears as benign in subsets: {subs.attack_type.unique()}")

if subs.attack_type.nunique() >= 2:
    types = subs.attack_type.unique()[:2]
        t0 = subs[subs.attack_type==types[0]].sort_values('sendTime_s')[['sendTime_s','s_x','s_y']].head(5)
            t1 = subs[subs.attack_type==types[1]].sort_values('sendTime_s')[['sendTime_s','s_x','s_y']].head(5)
                print(f"\n--- under '{types[0]}' ---\n{t0.to_string(index=False)}")
                    print(f"\n--- under '{types[1]}' ---\n{t1.to_string(index=False)}")
                        print("\nIdentical rows = same trajectory reused across subsets = leakage risk.")
                            print("Different rows = independently simulated per subset = split is clean.")

In [ ]:
sample_vid = pool[pool.attacker==0].sender_id.value_counts().index[0]
subs = pool[(pool.sender_id==sample_vid) & (pool.attacker==0)]
print(f"'{sample_vid}' appears as benign in subsets: {subs.attack_type.unique()}")

n_types = subs.attack_type.nunique()
print(f"number of distinct attack_type subsets this vehicle appears in: {n_types}")

In [ ]:
# ============================================================
# Rebuild pool from cache, then check for benign-trajectory
# duplication across attack_type subsets
# ============================================================
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

CACHE = '/content/drive/MyDrive/aintec_cache'
tr = pd.read_parquet(f'{CACHE}/trR.parquet')
te = pd.read_parquet(f'{CACHE}/teR.parquet')

pool = pd.concat([tr, te], ignore_index=True)
pool['vkey'] = pool.attack_type + '|' + pool.sender_id.astype(str)
print(f"pool rebuilt: {len(pool):,} rows")

# ---- sanity check: does this match the known overlapping-split sizes? ----
keys = pool.vkey.unique()
rng = np.random.default_rng(42)
half = set(rng.permutation(keys)[:len(keys)//2])
A = pool[pool.vkey.isin(half)]
B = pool[~pool.vkey.isin(half)]
print(f"random split by vkey: {len(A):,} / {len(B):,}  (expect ~308,031 / 314,320)")

# ---- the actual diagnostic ----
sample_vid = pool[pool.attacker==0].sender_id.value_counts().index[0]
subs = pool[(pool.sender_id==sample_vid) & (pool.attacker==0)]
print(f"\n'{sample_vid}' appears as benign in subsets: {subs.attack_type.unique()}")

n_types = subs.attack_type.nunique()
print(f"number of distinct attack_type subsets this vehicle appears in: {n_types}")

In [ ]:
types = subs.attack_type.unique()[:2]
t0 = subs[subs.attack_type == types[0]].sort_values('sendTime_s')
t1 = subs[subs.attack_type == types[1]].sort_values('sendTime_s')

print(f"--- under '{types[0]}' ---")
print(t0[['sendTime_s','s_x','s_y']].head(5).to_string(index=False))

print(f"\n--- under '{types[1]}' ---")
print(t1[['sendTime_s','s_x','s_y']].head(5).to_string(index=False))

n_common_times = len(set(t0.sendTime_s.round(3)) & set(t1.sendTime_s.round(3)))
print(f"\ntimestamps in common (rounded): {n_common_times} of {min(len(t0),len(t1))}")

In [ ]:
# ============================================================
# CORRECTED overlapping split: key on sender_id alone,
# so no physical vehicle's data can cross train/test
# ============================================================
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# rebuild pool if needed (skip if already in session)
CACHE = '/content/drive/MyDrive/aintec_cache'
tr = pd.read_parquet(f'{CACHE}/trR.parquet')
te = pd.read_parquet(f'{CACHE}/teR.parquet')
pool = pd.concat([tr, te], ignore_index=True)

SEED = 42
FEATS = ["s_spd","s_acl","s_hed","disp","dt","implied_spd",
         "spd_resid","hed_resid","acl_resid","latency"]
         FEATS_POS = FEATS + ["s_x","s_y"]
         RESID = ["spd_resid","hed_resid","acl_resid"]
         X = lambda d, f=FEATS: d[f].replace([np.inf,-np.inf], np.nan).fillna(0)

         # corrected split: sender_id ALONE, not attack_type|sender_id
         vids = pool.sender_id.unique()
         rng = np.random.default_rng(SEED)
         half = set(rng.permutation(vids)[:len(vids)//2])
         A, B = pool[pool.sender_id.isin(half)], pool[~pool.sender_id.isin(half)]
         print(f"corrected split by vehicle only: {len(A):,} / {len(B):,}")

         # verify: NO vehicle should appear on both sides now
         overlap = set(A.sender_id) & set(B.sender_id)
         print(f"vehicles appearing on both sides: {len(overlap)}  (must be 0)")
         assert len(overlap) == 0, "split still leaking — stop and report this"

         okA = A[RESID].notna().all(axis=1)
         bm = (B.attacker == 0).values

         for tag, feats in [('no position', FEATS), ('WITH position', FEATS_POS)]:
             m = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)
                 m.fit(X(A[okA], feats), A[okA].attacker)
                     p = m.predict_proba(X(B, feats))[:, 1]
                         print(f"\n[{tag}] benign flag {p[bm].mean() if False else (m.predict(X(B, feats))[bm]).mean():.4f}")
                             for a in ['constantPositionOffset', 'timeDelayAttack']:
                                     mk = ((B.attack_type == a) & (B.attacker == 1)).values
                                             if not mk.sum(): continue
                                                     y = np.r_[np.zeros(bm.sum()), np.ones(mk.sum())]
                                                             auc = roc_auc_score(y, np.r_[p[bm], p[mk]])
                                                                     print(f"    {a:24s} AUC {auc:.3f}   (leaky split gave 0.429/0.454 no-pos, 0.929/0.449 with-pos)")

In [ ]:
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

CACHE = '/content/drive/MyDrive/aintec_cache'
tr = pd.read_parquet(f'{CACHE}/trR.parquet')
te = pd.read_parquet(f'{CACHE}/teR.parquet')
pool = pd.concat([tr, te], ignore_index=True)

SEED = 42
FEATS = ["s_spd","s_acl","s_hed","disp","dt","implied_spd","spd_resid","hed_resid","acl_resid","latency"]
FEATS_POS = FEATS + ["s_x","s_y"]
RESID = ["spd_resid","hed_resid","acl_resid"]

vids = pool.sender_id.unique()
rng = np.random.default_rng(SEED)
half = set(rng.permutation(vids)[:len(vids)//2])
A = pool[pool.sender_id.isin(half)]
B = pool[~pool.sender_id.isin(half)]
print(f"corrected split by vehicle only: {len(A):,} / {len(B):,}")

overlap = set(A.sender_id) & set(B.sender_id)
print(f"vehicles appearing on both sides: {len(overlap)}  (must be 0)")

In [ ]:
X = lambda d, f=FEATS: d[f].replace([np.inf, -np.inf], np.nan).fillna(0)
okA = A[RESID].notna().all(axis=1)
bm = (B.attacker == 0).values

for tag, feats in [('no position', FEATS), ('WITH position', FEATS_POS)]:
    m = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)
        m.fit(X(A[okA], feats), A[okA].attacker)
            p = m.predict_proba(X(B, feats))[:, 1]
                pred = m.predict(X(B, feats))
                    print(f"\n[{tag}] benign flag {pred[bm].mean():.4f}")
                        for a in ['constantPositionOffset', 'timeDelayAttack']:
                                mk = ((B.attack_type == a) & (B.attacker == 1)).values
                                        if not mk.sum():
                                                    continue
                                                            y = np.r_[np.zeros(bm.sum()), np.ones(mk.sum())]
                                                                    auc = roc_auc_score(y, np.r_[p[bm], p[mk]])
                                                                            print(f"    {a:24s} AUC {auc:.3f}")

In [ ]:
X = lambda d, f=FEATS: d[f].replace([np.inf, -np.inf], np.nan).fillna(0)
okA = A[RESID].notna().all(axis=1)
bm = (B.attacker == 0).values
y_c = np.r_[np.zeros(bm.sum()), np.ones((( B.attack_type=='constantPositionOffset')&(B.attacker==1)).sum())]
y_t = np.r_[np.zeros(bm.sum()), np.ones((( B.attack_type=='timeDelayAttack')&(B.attacker==1)).sum())]

m1 = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)
m1.fit(X(A[okA], FEATS), A[okA].attacker)
p1 = m1.predict_proba(X(B, FEATS))[:, 1]
pred1 = m1.predict(X(B, FEATS))
mk_c = ((B.attack_type=='constantPositionOffset')&(B.attacker==1)).values
mk_t = ((B.attack_type=='timeDelayAttack')&(B.attacker==1)).values
print("no position:")
print("  benign flag", round(pred1[bm].mean(), 4))
print("  constantPositionOffset AUC", round(roc_auc_score(y_c, np.r_[p1[bm], p1[mk_c]]), 3))
print("  timeDelayAttack AUC", round(roc_auc_score(y_t, np.r_[p1[bm], p1[mk_t]]), 3))

m2 = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)
m2.fit(X(A[okA], FEATS_POS), A[okA].attacker)
p2 = m2.predict_proba(X(B, FEATS_POS))[:, 1]
pred2 = m2.predict(X(B, FEATS_POS))
print("with position:")
print("  benign flag", round(pred2[bm].mean(), 4))
print("  constantPositionOffset AUC", round(roc_auc_score(y_c, np.r_[p2[bm], p2[mk_c]]), 3))
print("  timeDelayAttack AUC", round(roc_auc_score(y_t, np.r_[p2[bm], p2[mk_t]]), 3))

In [ ]:
from sklearn.metrics import roc_auc_score
import pandas as pd

models      = {"RandomForest": rf, "DecisionTree": dt, "LogisticReg": lr}
scores_X    = X_test          # same feature matrix used for table_auc.csv
attack_col  = attack_test     # per-message attack name; benign rows are "" / "benign" / NaN

benign = attack_col.isna() | attack_col.isin(["", "benign", "Benign"])
rows = []
for name, m in models.items():
    s = m.predict_proba(scores_X)[:, 1]
    for atk in ["constantPositionOffset", "timeDelayAttack"]:
        sel = benign | (attack_col == atk)
        y   = (attack_col[sel] == atk).astype(int)
        rows.append({"model": name, "attack": atk,
                     "auc": roc_auc_score(y, s[sel])})

out = pd.DataFrame(rows).pivot(index="attack", columns="model", values="auc")
print(out.round(3))

In [ ]:
print([k for k,v in globals().items() if hasattr(v,'predict_proba')])